# Module 06 · Composition, burden and susceptibility

How much senescence is there, and does it change with age or disease?

Two quantities, kept separate throughout because they answer different
questions:

**Burden** — `n_snc(ct) / n_total_cells(donor)`. Abundance-weighted. What
share of the donor's brain is senescent cells of this type. A cell type can
gain burden purely by becoming more common.

**Susceptibility** — `n_snc(ct) / n_cells(ct)`. Abundance-independent. How
likely a cell *of this type* is to be senescent.

They are linked by `Burden = CellProp × SenFrac`, and the decomposition in
section 18 splits an observed change into the two contributions.

Unit of analysis is the **donor** throughout. The cell-level models carry
`(1 | Donor)`.

No depth covariate appears in the aging models: the SenePy input was already
depth-adjusted upstream (module 02), so regressing depth again here would
over-correct. See section 26 for the one place the source does include it.

| Sections | |
|---|---|
| 01-14 | config, load, frames, specs |
| 15-19 | burden build, composition, decomposition |
| 20-22 | burden trajectory — OLS · RLM · GLMM |
| 23-28 | group contrasts — OLS · RLR · GLMM · cross-model |
| 29-33 | per-decade GLMMs — susceptibility and burden |
| 34 | Garg differential proportion |

Runs in either mode. `STUDY_TYPE = "aging"` uses age bins with the youngest as
reference; `STUDY_TYPE = "disease"` uses the diagnosis column with the control
group as reference. Nothing else changes.

> **Note on the numbers inside the code cells.** The lifted code refers to
> *Module 01* (scoring) and *Module 03* (burden) in its comments and in the
> output path scheme — that is the source notebook's own older numbering, kept
> verbatim. In this repo those are modules **03** and **07**. The on-disk paths
> it writes (`module_03_burden_modeling/...`) are unchanged so that module 07
> can still find them.


---
## 01 · Config

**Why.** One cell fixes the run: which dataset, which mode, which reference
group, which cell types survive. Everything downstream reads from here.

Paths come from the environment — see `.env.example`. `_env()` below is the
only line that is not in the source notebook; it replaces the two hardcoded
roots.

**Set before running:** `STUDY_TYPE`, `DISEASE`, `DATASET`, `REFERENCE_GROUP`.

In [ ]:
# -----------------------------------------------------------------------------
# Path resolution - the only addition to the source config
# -----------------------------------------------------------------------------
#   SENESCENCE_DATA : analysis root (module outputs written under it)
#   SENESCENCE_REF  : reference root (published marker files, read-only)
# See .env.example. Both must be set before the kernel starts.
import os


def _env(name):
    v = os.environ.get(name)
    if not v:
        raise RuntimeError(
            f"{name} is not set. Copy .env.example to .env, edit the two "
            f"paths, and source it before starting the kernel.")
    return v.rstrip("/")


print(f"  SENESCENCE_DATA -> {_env('SENESCENCE_DATA')}")
print(f"  SENESCENCE_REF  -> {_env('SENESCENCE_REF')}")

In [ ]:
# =============================================================================
# MODULE 03 — Senescence Burden Modeling  ·  CONFIG (Cell 0)
# =============================================================================
# Cross-tissue parameterization. Edit this section to switch tissue/dataset.
# All downstream cells read from this config block.
# =============================================================================

import os

# ─────────────────────────────────────────────────────────────────────────────
# Run parameters — edit these
# ─────────────────────────────────────────────────────────────────────────────
TISSUE     = "brain"        # brain | pbmc | csf
STUDY_TYPE = "aging"      # aging | disease
DISEASE    = None         # AD | PD | FTD | DLB | ALS | MS  (used when STUDY_TYPE=='disease')
DATASET    = "psychad_aging"       # cohort key

# Reference group for disease contrasts (the level the others are compared TO)
REFERENCE_GROUP = "Age_20_29"

# ─────────────────────────────────────────────────────────────────────────────
# Section toggle — pathology biomarker analyses (§7a, §7b, §7c)
#
# These sections require donor-level continuous biomarkers (tau, amyloid)
# AND/OR a categorical staging variable (pathology_group). Most datasets
# don't have these, so the toggle defaults to "auto":
#
#   "auto"  — run if §1.4b assigns at least one of tau/amyloid/pathology_group
#   "on"    — force-run; error in §7a if data missing
#   "off"   — force-skip §7 regardless of data availability
#
# §3, §4, §5, §6, §8.x are CORE and always run. They validate the senescence
# labels themselves (independent of pathology).
# ─────────────────────────────────────────────────────────────────────────────
RUN_PATHOLOGY_SECTIONS = "auto"

# ─────────────────────────────────────────────────────────────────────────────
# Derived condition tags
# ─────────────────────────────────────────────────────────────────────────────
IS_AGING            = (STUDY_TYPE == "aging")
IS_DISEASE          = (STUDY_TYPE == "disease")
CONDITION           = STUDY_TYPE
CONDITION_TAG       = STUDY_TYPE if IS_AGING else f"{STUDY_TYPE}_{DISEASE}"
CONDITION_SUBPATH   = STUDY_TYPE if IS_AGING else f"{STUDY_TYPE}/{DISEASE}"

# ─────────────────────────────────────────────────────────────────────────────
# Paths — input from Module 01, output to Module 03 directory
# ─────────────────────────────────────────────────────────────────────────────
SCRATCH = _env("SENESCENCE_DATA")

BASE_M01    = f"{SCRATCH}/{TISSUE}/module_01_senepy_scoring/{CONDITION_SUBPATH}/{DATASET}"
BASE_OUTPUT = f"{SCRATCH}/{TISSUE}/module_03_burden_modeling/{CONDITION_SUBPATH}/{DATASET}"

PATHS = {
    "input_data":    f"{BASE_M01}/data",
    "input_results": f"{BASE_M01}/results",
    "data":          f"{BASE_OUTPUT}/data",
    "figures":       f"{BASE_OUTPUT}/figures",
    "results":       f"{BASE_OUTPUT}/results",
}

# ─────────────────────────────────────────────────────────────────────────────
# Core metadata column mapping — varies by dataset
#
# These are the CORE fields needed by every downstream section. They are
# stable enough across cohorts to keep in a static per-dataset registry.
#
# PATHOLOGY columns (tau, amyloid, cognition, etc.) are handled separately
# via the discovery + assignment workflow in §1.4a / §1.4b — those are NOT
# harmonized across cohorts.
# ─────────────────────────────────────────────────────────────────────────────
META_BY_DATASET = {
    # ── Brain ──────────────────────────────────────────────────────────────
    "australian": {
        "donor":       "Study_ID",
        "celltype":    "subclass_main",
        "study_group": "Condition",
        "sen_score":   "hippocampus_score",
        "sen_label":   "hippocampus_SnC",
        "age":         "Age_of_death",
        "sex":         "Sex",
        "cohort":      None,
        "n_counts":    "total_counts",
    },
    "psychad_aging": {
        "donor":       "Sample",
        "celltype":    "subclass",
        "study_group": "Study_Group",
        "sen_score":   "senescence_score",
        "sen_label":   "is_senescent",
        "age":         "Age",
        "sex":         "Sex",
        "cohort":      None,
        "n_counts":    "total_counts",
    },
    "psychad_ad": {
        "donor":       "Sample",
        "celltype":    "subclass",
        "study_group": "Disease_Group",
        "sen_score":   "senescence_score",
        "sen_label":   "is_senescent",
        "age":         "Age",
        "sex":         "Sex",
        "cohort":      None,
        "n_counts":    "total_counts",
    },
    "mathys": {
        "donor":       "projid",
        "celltype":    "cell_type",
        "study_group": "Diagnosis",
        "sen_score":   "senescence_score",
        "sen_label":   "is_senescent",
        "age":         "Age",
        "sex":         "Sex",
        "cohort":      None,
        "n_counts":    "total_counts",
    },
    # ── PBMC (placeholders — confirm at first run) ─────────────────────────
    "terekhova": {
        "donor":       "Sample",
        "celltype":    "cell_type",
        "study_group": "age_group",
        "sen_score":   "sen_score",
        "sen_label":   "putative_sen",
        "age":         "Age",
        "sex":         "Sex",
        "cohort":      "Batch",
        "n_counts":    "nCount_RNA",
    },
    "yazar": {
        "donor":       "Sample",
        "celltype":    "cell_type",
        "study_group": "age_group",
        "sen_score":   "sen_score",
        "sen_label":   "putative_sen",
        "age":         "Age",
        "sex":         "Sex",
        "cohort":      "dataset",
        "n_counts":    "nCount_RNA",
    },
    "gate": {
        "donor":       "Sample",
        "celltype":    "cell_type",
        "study_group": "Study_Group",
        "sen_score":   "sen_score",
        "sen_label":   "putative_sen",
        "age":         "Age",
        "sex":         "Sex",
        "cohort":      "study_short",
        "n_counts":    "nCount_RNA",
    },
    "ostkamp": {
        "donor":       "Sample",
        "celltype":    "cell_type",
        "study_group": "Study_Group",
        "sen_score":   "sen_score",
        "sen_label":   "putative_sen",
        "age":         "Age",
        "sex":         "Sex",
        "cohort":      "study_short",
        "n_counts":    "nCount_RNA",
    },
    # ── CSF ────────────────────────────────────────────────────────────────
    "gate_csf": {
        "donor":       "Sample",
        "celltype":    "cell_type",
        "study_group": "Study_Group",
        "sen_score":   "sen_score",
        "sen_label":   "putative_sen",
        "age":         "Age",
        "sex":         "Sex",
        "cohort":      None,
        "n_counts":    "nCount_RNA",
    },
}
META = META_BY_DATASET[DATASET]

# ─────────────────────────────────────────────────────────────────────────────
# Pathology concept registry — used by §1.4a (discovery)
#
# Each canonical concept gets:
#   - patterns: substring matches against obs column names (case-insensitive)
#   - description: shown in discovery output
#
# Adding a new concept here makes §1.4a search for it. To use a discovered
# concept downstream, edit §1.4b's PATHOLOGY_META dict.
# ─────────────────────────────────────────────────────────────────────────────
PATHOLOGY_CONCEPTS = {
    "pathology_group": {
        "description": "categorical staging (e.g. no/early/late, low/intermediate/high)",
        "patterns":    ["pathology_group", "path_group", "ad_status", "diagnosis_path",
                        "path_severity", "path_stage"],
        "expected":    "categorical",
    },
    "tau": {
        "description": "tau / NFT / tangle burden",
        "patterns":    ["tau", "nft", "tangle", "braak"],
        "expected":    "continuous_or_ordinal",
    },
    "amyloid": {
        "description": "amyloid / plaque / Aβ burden",
        "patterns":    ["amyloid", "abeta", "ab42", "ab40", "plaq", "cerad"],
        "expected":    "continuous_or_ordinal",
    },
    "cognition": {
        "description": "cognitive score (Cogdx, MMSE, MoCA, CDR, etc.)",
        "patterns":    ["cog", "mmse", "moca", "cdr", "dementia", "mci_score"],
        "expected":    "continuous_or_ordinal",
    },
    "apoe": {
        "description": "APOE genotype",
        "patterns":    ["apoe", "apo_e", "apoe_genotype", "apoe4"],
        "expected":    "categorical",
    },
    "pmi": {
        "description": "postmortem interval",
        "patterns":    ["pmi", "postmortem", "post_mortem", "post.mortem"],
        "expected":    "continuous",
    },
}

# ─────────────────────────────────────────────────────────────────────────────
# Lineage registry — per tissue
# ─────────────────────────────────────────────────────────────────────────────
LINEAGE_CONFIGS_BY_TISSUE = {
    "brain": {
        "Excitatory":      {"label": "Excitatory neuron"},
        "Inhibitory":      {"label": "Inhibitory neuron"},
        "Astrocyte":       {"label": "Astrocyte"},
        "Oligodendrocyte": {"label": "Oligodendrocyte"},
        "Microglia":       {"label": "Microglia"},
        "OPC":             {"label": "OPC"},
        "Endothelial":     {"label": "Endothelial"},
        "Pericyte":        {"label": "Pericyte"},
    },
    "pbmc": {
        "cd4t":     {"label": "CD4 T cell"},
        "cd8t":     {"label": "CD8 T cell"},
        "unconvT":  {"label": "Unconventional T"},
        "nkcell":   {"label": "NK cell"},
        "cd14mono": {"label": "CD14 Monocyte"},
        "cd16mono": {"label": "CD16 Monocyte"},
        "memB":     {"label": "Memory B cell"},
        "naiveB":   {"label": "Naive B cell"},
    },
    "csf": {
        "cd4t":     {"label": "CD4 T cell"},
        "cd8t":     {"label": "CD8 T cell"},
        "nkcell":   {"label": "NK cell"},
        "monocyte": {"label": "Monocyte"},
        "bcell":    {"label": "B cell"},
        "dc":       {"label": "Dendritic cell"},
    },
}
LINEAGE_CONFIGS = LINEAGE_CONFIGS_BY_TISSUE[TISSUE]

CELLTYPE_COL_BY_TISSUE = {"brain": "cell_type", "pbmc": "cell_type", "csf": "cell_type"}
CELLTYPE_COL = CELLTYPE_COL_BY_TISSUE[TISSUE]

# ─────────────────────────────────────────────────────────────────────────────
# Lineage colors — per tissue (Okabe–Ito for brain; Tableau for blood)
# ─────────────────────────────────────────────────────────────────────────────
LINEAGE_COLORS_BY_TISSUE = {
    "brain": {
        "Excitatory":      "#0072B2",
        "Inhibitory":      "#E69F00",
        "Astrocyte":       "#009E73",
        "Oligodendrocyte": "#56B4E9",
        "Microglia":       "#D55E00",
        "OPC":             "#CC79A7",
        "Endothelial":     "#7F7F7F",
        "Pericyte":        "#999999",
    },
    "pbmc": {
        "cd4t":     "#4E79A7",
        "cd8t":     "#A0CBE8",
        "unconvT":  "#BAB0AC",
        "nkcell":   "#59A14F",
        "cd14mono": "#F28E2B",
        "cd16mono": "#FFBE7D",
        "memB":     "#B07AA1",
        "naiveB":   "#76B7B2",
    },
    "csf": {
        "cd4t":     "#4E79A7",
        "cd8t":     "#A0CBE8",
        "nkcell":   "#59A14F",
        "monocyte": "#F28E2B",
        "bcell":    "#B07AA1",
        "dc":       "#76B7B2",
    },
}
LINEAGE_COLORS = LINEAGE_COLORS_BY_TISSUE[TISSUE]
CELL_TYPE_COLORS = LINEAGE_COLORS    # back-compat alias
CELLTYPE_COLORS  = LINEAGE_COLORS

# ─────────────────────────────────────────────────────────────────────────────
# Study-group colors (mixed: aging brackets + disease groups)
# ─────────────────────────────────────────────────────────────────────────────
STUDY_GROUP_COLORS = {
    # Aging brackets — cool → warm
    "20-29": "#2E86AB", "30-39": "#4A90E2", "40-49": "#50C878",
    "50-59": "#FFB347", "60-69": "#FF8C00", "70-79": "#E24A4A",
    "80-89": "#8B0000", "90+":   "#4A0000",
    # Aging brackets — underscored variants
    "Age_20_29": "#2E86AB", "Age_30_39": "#4A90E2", "Age_40_49": "#50C878",
    "Age_50_59": "#FFB347", "Age_60_69": "#FF8C00", "Age_70_79": "#E24A4A",
    "Age_80_100": "#8B0000",
    # Disease — generic
    "Control":            "#4E79A7",
    "control":            "#4E79A7",
    "MCI":                "#F28E2B",
    "AD":                 "#E15759",
    "Alzheimers Disease": "#E15759",
    "Alzheimer's Disease": "#E15759",
    "MCI/AD":             "#E15759",
    "Other":              "#BAB0AC",
    # PsychAD / ROSMAP composites
    "Young_Healthy_Control": "#4A90E2",
    "Old_Healthy_Control":   "#4E79A7",
    "Old_AD":                "#E15759",
    # Neurology cohorts
    "Healthy Control":  "#4E79A7",
    "Healthy":          "#4E79A7",
    "Non-rapid ALS":    "#F28E2B",
    "Rapid ALS":        "#E15759",
    "MS":               "#E15759",
    "PD":               "#E15759",
    "FTD":              "#E15759",
    "DLB":              "#E15759",
}

# Pathology group color palette — diverging green→yellow→red.
# §1.4b's PATHOLOGY_GROUP_LEVEL_ORDER drives which keys are looked up.
# For non-standard pathology levels (Braak 0-6, CERAD 1-4, etc.), §3/§4
# fall back to a generated gradient at runtime.
PATHOLOGY_GROUP_COLORS = {
    "no-pathology":    "#59A14F",
    "early-pathology": "#EDC948",
    "late-pathology":  "#E15759",
    "low":             "#59A14F",
    "intermediate":    "#EDC948",
    "high":            "#E15759",
}

# Senescence labels (handles binary 0/1 AND string encoding)
SNC_COLORS = {
    "Senescent":     "#E15759", "Non-senescent": "#4E79A7",
    "senescent":     "#E15759", "non-senescent": "#4E79A7",
    1:               "#E15759", 0:                "#4E79A7",
    True:            "#E15759", False:            "#4E79A7",
}

SEX_COLORS = {
    "Male":   "#5D6D7E", "Female": "#A569BD",
    "male":   "#5D6D7E", "female": "#A569BD",
    "M":      "#5D6D7E", "F":      "#A569BD",
    "m":      "#5D6D7E", "f":      "#A569BD",
    1:        "#5D6D7E", 0:        "#A569BD",
}

COHORT_COLORS = {
    "PsychAD":     "#4E79A7",
    "PsychENCODE": "#A0CBE8",
    "ROSMAP":      "#59A14F",
    "Mathys":      "#76B7B2",
    "MSSM":        "#B07AA1",
    "HBCC":        "#F28E2B",
    "Cohort 1":    "#E15759",
    "Cohort 2":    "#FFBE7D",
    "Cohort 3":    "#9C755F",
}

PHASE_COLORS = {"G1": "#4E79A7", "S": "#F28E2B", "G2M": "#E15759"}

# ─────────────────────────────────────────────────────────────────────────────
# Diagnosis ordering (used in plots when applicable)
# ─────────────────────────────────────────────────────────────────────────────
DIAGNOSIS_ORDER = ["Control", "MCI", "AD", "Other"]

# ─────────────────────────────────────────────────────────────────────────────
# Comparison / focus restrictions
# ─────────────────────────────────────────────────────────────────────────────
# Optional restriction on which Study_Group levels to compare; None = all
COMPARISON_GROUPS = None

# Optional restriction on which lineages to model; None = all in LINEAGE_CONFIGS
FOCUS_CELL_TYPES = None

# ─────────────────────────────────────────────────────────────────────────────
# Modeling parameters
# ─────────────────────────────────────────────────────────────────────────────
MIN_CELLS_PER_DONOR = 20
MIN_DONORS_PER_CT   = 10
FDR_THRESHOLD       = 0.05
SEED                = 42

GLMM_CONFIG = {
    "optimizer":  "bobyqa",
    "maxfun":     100000,
    "nAGQ":       1,
    "min_cells":  100,
    "min_donors": 15,
}

VALIDATION = {
    "min_cells_per_group": 10,
    "min_paired_donors":   10,
}

MANUAL_VALIDATION_CTS_BY_TISSUE = {
    "brain": ["Microglia", "Astrocyte", "OPC", "Oligodendrocyte"],
    "pbmc":  ["cd8t", "cd4t", "cd14mono", "cd16mono"],
    "csf":   ["cd8t", "cd4t", "monocyte"],
}
MANUAL_VALIDATION_CTS = MANUAL_VALIDATION_CTS_BY_TISSUE[TISSUE]

# ─────────────────────────────────────────────────────────────────────────────
# Sloan gene-list reference (for §5c)
# ─────────────────────────────────────────────────────────────────────────────
MARKERS_DIR = _env("SENESCENCE_REF") + "/markers"
SLOAN_FILE  = f"{MARKERS_DIR}/1-s2.0-S2666979X25003830-mmc10.xlsx"

SLOAN_LISTS = {
    "p53 Targets":       {"col": 0, "type": "hallmark"},
    "Cell Cycle Arrest": {"col": 1, "type": "hallmark"},
    "SASP":              {"col": 2, "type": "hallmark"},
    "Anti-apoptosis":    {"col": 3, "type": "hallmark"},
    "DDR":               {"col": 4, "type": "hallmark"},
    "Surface Markers":   {"col": 5, "type": "hallmark"},
    "Lysosomal":         {"col": 6, "type": "hallmark"},
    "San Diego TMC":     {"col": 7, "type": "multi"},
    "SenMayo":           {"col": 8, "type": "multi"},
    "Fridman":           {"col": 9, "type": "multi"},
}

# ─────────────────────────────────────────────────────────────────────────────
# Cell cycle gene reference (for §5d) — Tirosh et al. 2016
# ─────────────────────────────────────────────────────────────────────────────
CELL_CYCLE_GENES = {
    "s_genes": [
        "MCM5","PCNA","TYMS","FEN1","MCM2","MCM4","RRM1","UNG",
        "GINS2","MCM6","CDCA7","DTL","PRIM1","UHRF1","MLF1IP",
        "HELLS","RFC2","RPA2","NASP","RAD51AP1","GMNN","WDR76",
        "SLBP","CCNE2","UBR7","POLD3","MSH2","ATAD2","RAD51",
        "RRM2","CDC45","CDC6","EXO1","TIPIN","DSCC1","BLM",
        "CASP8AP2","USP1","CLSPN","POLA1","CHAF1B","BRIP1","E2F8",
    ],
    "g2m_genes": [
        "HMGB2","CDK1","NUSAP1","UBE2C","BIRC5","TPX2","TOP2A",
        "NDC80","CKS2","NUF2","CKS1B","MKI67","TMPO","CENPF",
        "TACC3","FAM64A","SMC4","CCNB2","CKAP2L","CKAP2",
        "AURKB","BUB1","KIF11","ANP32E","TUBB4B","GTSE1",
        "KIF20B","HJURP","CDCA3","HN1","CDC20","TTK",
        "CDC25C","KIF2C","RANGAP1","NCAPD2","DLGAP5","CDCA2",
        "CDCA8","ECT2","KIF23","HMMR","AURKA","PSRC1",
        "ANLN","LBR","CKAP5","CENPE","CTCF","NEK2",
        "G2E3","GAS2L3","CBX5","CENPA",
    ],
}

# ─────────────────────────────────────────────────────────────────────────────
# Plot style
# ─────────────────────────────────────────────────────────────────────────────
PLOT_STYLE = {
    "dpi":        150,
    "dpi_save":   300,
    "font_size":  10,
    "title_size": 11,
    "fig_ext":    "pdf",
    "formats":    ["pdf", "png", "svg"],
}

# ─────────────────────────────────────────────────────────────────────────────
# Derived flags
# ─────────────────────────────────────────────────────────────────────────────
HAS_COHORT = META["cohort"] is not None
N_LINEAGES = len(LINEAGE_CONFIGS)

# ─────────────────────────────────────────────────────────────────────────────
# Summary
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 64)
print(f"MODULE 03 — Senescence Burden Modeling")
print("=" * 64)
print(f"  TISSUE          : {TISSUE}")
print(f"  STUDY_TYPE      : {STUDY_TYPE}")
if IS_DISEASE:
    print(f"  DISEASE         : {DISEASE}")
print(f"  DATASET         : {DATASET}")
print(f"  REFERENCE_GROUP : {REFERENCE_GROUP}")
print()
print(f"  Cell-type col   : {CELLTYPE_COL}")
print(f"  Lineages ({N_LINEAGES})    : {list(LINEAGE_CONFIGS.keys())}")
print(f"  Manual val. CTs : {MANUAL_VALIDATION_CTS}")
print(f"  Min cells/donor : {MIN_CELLS_PER_DONOR}")
print(f"  Min donors/CT   : {MIN_DONORS_PER_CT}")
print(f"  FDR α           : {FDR_THRESHOLD}")
print()
print(f"  Section toggles:")
print(f"    RUN_PATHOLOGY_SECTIONS : {RUN_PATHOLOGY_SECTIONS}")
print(f"      §7a/§7b/§7c gated on this + §1.4b assignment")
print(f"    §3/§4/§5/§6/§8.x       : always run")
print()
print(f"  Pathology concepts to discover in §1.4a:")
for concept, info in PATHOLOGY_CONCEPTS.items():
    print(f"    {concept:18s} {info['description']}")
print()
print(f"  Core META mapping (user-facing column names):")
for k, v in META.items():
    flag = "  ⚠ None" if v is None else ""
    print(f"      {k:14s} → {v}{flag}")
print()
print(f"  Input  : {PATHS['input_data']}")
print(f"  Output : {BASE_OUTPUT}")
for key in ["data", "figures", "results"]:
    print(f"      {key:8s} → {PATHS[key]}")
print("=" * 64)

---
## 02 · Setup and load

**Why.** Reads the scored `.h5ad` from module 03, harmonizes cell-type names, and derives `log10_total_counts` if the object does not already carry it.

In [ ]:
# =============================================================================
# MODULE 03 — Senescence Burden Modeling  ·  SETUP + LOAD (Cell 1)
# =============================================================================
# Imports, output directories, plot style, helper functions, and h5ad load
# with core obs column standardization. Pathology-specific column discovery
# happens in §1.4a; pathology assignment in §1.4b; pathology rename apply
# in §1.5.
# =============================================================================

import os
import warnings
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.gridspec as gridspec
import matplotlib.patheffects as pe
from matplotlib.patches import Polygon
from matplotlib.lines  import Line2D
from pathlib import Path

from scipy import stats
from scipy.stats import (
    mannwhitneyu, kruskal, spearmanr, chi2_contingency, gaussian_kde, wilcoxon,
)
from scipy.sparse import issparse
from scipy.spatial.distance import pdist, squareform

from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
np.random.seed(SEED)

# ─────────────────────────────────────────────────────────────────────────────
# Output directories
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 64)
print(f"§1 — SETUP + LOAD  |  {DATASET}")
print("=" * 64)

print(f"\n▸ Output directories")
for key in ["data", "figures", "results"]:
    Path(PATHS[key]).mkdir(parents=True, exist_ok=True)
    print(f"  ✓ {key:8s} → {PATHS[key]}")

# ─────────────────────────────────────────────────────────────────────────────
# Plot style (matplotlib defaults)
# ─────────────────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi":       PLOT_STYLE["dpi"],
    "savefig.dpi":      PLOT_STYLE["dpi_save"],
    "font.size":        PLOT_STYLE["font_size"],
    "axes.titlesize":   PLOT_STYLE["title_size"],
    "axes.labelsize":   PLOT_STYLE["font_size"],
    "xtick.labelsize":  PLOT_STYLE["font_size"] - 1,
    "ytick.labelsize":  PLOT_STYLE["font_size"] - 1,
    "legend.fontsize":  PLOT_STYLE["font_size"] - 1,
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "savefig.bbox":      "tight",
    "savefig.facecolor": "white",
})

# ─────────────────────────────────────────────────────────────────────────────
# Helper functions
# ─────────────────────────────────────────────────────────────────────────────

def fmt_p(p):
    """Format p-value: '<0.001' or scientific notation for very small."""
    if pd.isna(p):
        return "NA"
    if p < 0.001:
        return f"{p:.2e}"
    return f"{p:.3f}"

def sig_stars(p):
    """Asterisk significance encoding."""
    if pd.isna(p):
        return ""
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return ""

def bh_correction(pvals):
    """Benjamini–Hochberg FDR correction."""
    arr  = np.asarray(pvals, dtype=float)
    mask = ~np.isnan(arr)
    out  = np.full_like(arr, np.nan)
    if mask.sum() == 0:
        return out
    _, q, _, _ = multipletests(arr[mask], method="fdr_bh")
    out[mask] = q
    return out

def find_umi_col(obs):
    """Find a UMI total-counts column in priority order."""
    candidates = [
        META.get("n_counts"),
        "nCount_RNA", "total_counts", "n_counts", "nUMI",
    ]
    for c in candidates:
        if c is not None and c in obs.columns:
            return c
    return None

def _snc_label_str(series):
    """Coerce SnC label encoding to canonical strings: Senescent / Non-senescent."""
    if series.dtype == bool:
        return np.where(series, "Senescent", "Non-senescent")
    if pd.api.types.is_numeric_dtype(series):
        return np.where(series.astype(int) == 1, "Senescent", "Non-senescent")
    s = series.astype(str).str.strip().str.lower()
    out = np.where(
        s.isin(["senescent", "snc", "true", "1", "yes"]),
        "Senescent", "Non-senescent",
    )
    return out

def display_table(df, title=None, max_rows=30):
    """Print a tabular summary."""
    if title:
        print(f"\n  {title}")
        print(f"  {'─' * len(title)}")
    print(df.head(max_rows).to_string(index=False))
    if len(df) > max_rows:
        print(f"  ... ({len(df) - max_rows} more rows)")

def save_table(df, slug, index=False):
    """Save dataframe to results/ as csv.

    Parameters
    ----------
    df : DataFrame
        The table to save.
    slug : str
        Filename stem (no extension). Saved as {PATHS['results']}/{slug}.csv.
    index : bool, default False
        Whether to write the row index as the first column. Set True for
        square pairwise matrices (e.g. config × config Jaccard) where the
        row labels carry information; leave False (default) for ordinary
        long-form tables where the index is just an integer position.
    """
    path = f"{PATHS['results']}/{slug}.csv"
    df.to_csv(path, index=index)
    print(f"  ✓ saved → {Path(path).name}  ({len(df)} rows)")

def save_figure(fig, slug):
    """Save figure to figures/ in all PLOT_STYLE['formats'] extensions."""
    for ext in PLOT_STYLE["formats"]:
        path = f"{PATHS['figures']}/{slug}.{ext}"
        fig.savefig(path, dpi=PLOT_STYLE["dpi_save"], bbox_inches="tight",
                    facecolor="white")
    print(f"  ✓ saved → {slug}.{{{','.join(PLOT_STYLE['formats'])}}}")

def _ct_sort_key(ct):
    """Sort key putting OVERALL first, then CELLTYPE_ORDER_PLOT order, then alpha."""
    if ct == "OVERALL":
        return (0, 0, "")
    if "CELLTYPE_ORDER_PLOT" in globals() and ct in CELLTYPE_ORDER_PLOT:
        return (1, CELLTYPE_ORDER_PLOT.index(ct), ct)
    return (2, 0, ct)

# ─────────────────────────────────────────────────────────────────────────────
# §1.1 Load h5ad
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n▸ §1.1 Load scored h5ad from Module 01")

h5ad_path = os.path.join(
    PATHS["input_data"],
    f"all_{CONDITION_TAG}_{DATASET}_scored.h5ad",
)
if not os.path.exists(h5ad_path):
    raise FileNotFoundError(
        f"  ✗ Not found: {h5ad_path}\n"
        f"  Re-run Module 01 to generate it, or check CONDITION_TAG/DATASET in §0."
    )

adata = sc.read_h5ad(h5ad_path)

# Drop duplicated obs columns (from prior runs)
n_dup = adata.obs.columns.duplicated().sum()
if n_dup > 0:
    print(f"  ⚠ Dropped {n_dup} duplicate obs column(s) on load")
    adata.obs = adata.obs.loc[:, ~adata.obs.columns.duplicated(keep="first")]

print(f"  ✓ Loaded   : {adata.n_obs:,} cells × {adata.n_vars:,} genes")
print(f"  Donors      : {adata.obs[META['donor']].nunique()}")
print(f"  obs columns : {len(adata.obs.columns)}")

# ─────────────────────────────────────────────────────────────────────────────
# §1.2 Standardize CORE obs columns
# Pathology columns (tau, amyloid, etc.) are handled by §1.4a + §1.4b + §1.5.
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n▸ §1.2 Standardize core obs column names")

rename_map = {}

# Required: Donor
if META["donor"] not in adata.obs.columns:
    raise KeyError(f"  ✗ META['donor']='{META['donor']}' not in obs columns")
if META["donor"] != "Donor":
    rename_map[META["donor"]] = "Donor"
print(f"  Donor       ← {META['donor']!r}")

# Required: Cell_Type
if META["celltype"] not in adata.obs.columns:
    raise KeyError(f"  ✗ META['celltype']='{META['celltype']}' not in obs columns")
if META["celltype"] != "Cell_Type":
    rename_map[META["celltype"]] = "Cell_Type"
print(f"  Cell_Type   ← {META['celltype']!r}")

# Required: Study_Group
if META["study_group"] not in adata.obs.columns:
    raise KeyError(
        f"  ✗ META['study_group']='{META['study_group']}' not in obs.\n"
        f"  Available: {sorted(adata.obs.columns.tolist())[:30]}..."
    )
if META["study_group"] != "Study_Group":
    rename_map[META["study_group"]] = "Study_Group"
print(f"  Study_Group ← {META['study_group']!r}")

# Optional: Age, Sex, Cohort
optional_cols = {
    "age":    ("Age",    np.nan),
    "sex":    ("Sex",    "Unknown"),
    "cohort": ("Cohort", DATASET),
}
for meta_key, (std_name, default_val) in optional_cols.items():
    src_col       = META.get(meta_key)
    target_exists = std_name in adata.obs.columns

    if src_col is not None and src_col in adata.obs.columns:
        if src_col == std_name:
            print(f"  {std_name:10s}  (already canonical)")
        elif target_exists:
            print(f"  ⚠ {std_name} exists AND {src_col!r} also present "
                  f"→ dropping existing {std_name!r}")
            adata.obs = adata.obs.drop(columns=[std_name])
            rename_map[src_col] = std_name
            print(f"  {std_name:10s}← {src_col!r}")
        else:
            rename_map[src_col] = std_name
            print(f"  {std_name:10s}← {src_col!r}")
    else:
        if target_exists:
            print(f"  {std_name:10s}  (already present; META src missing)")
        else:
            adata.obs[std_name] = default_val
            reason = "None in META" if src_col is None else "not in obs"
            print(f"  ⚠ {std_name:10s}= {default_val!r}  ({reason})")

if rename_map:
    adata.obs = adata.obs.rename(columns=rename_map)

# Final dup sweep after rename
n_dup2 = adata.obs.columns.duplicated().sum()
if n_dup2 > 0:
    print(f"  ⚠ Dropped {n_dup2} duplicate(s) after rename")
    adata.obs = adata.obs.loc[:, ~adata.obs.columns.duplicated(keep="first")]

# ─────────────────────────────────────────────────────────────────────────────
# §1.3 Canonical senescence columns (encoding-robust)
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n▸ §1.3 Canonical senescence columns")

if META["sen_score"] not in adata.obs.columns:
    raise KeyError(f"  ✗ META['sen_score']='{META['sen_score']}' not in obs")
adata.obs["sen_score"] = pd.to_numeric(
    adata.obs[META["sen_score"]], errors="coerce"
).values
print(f"  sen_score     ← {META['sen_score']!r}  "
      f"(range [{adata.obs['sen_score'].min():.3f}, "
      f"{adata.obs['sen_score'].max():.3f}])")

if META["sen_label"] not in adata.obs.columns:
    raise KeyError(f"  ✗ META['sen_label']='{META['sen_label']}' not in obs")
snc_str = _snc_label_str(adata.obs[META["sen_label"]])
adata.obs["is_senescent"] = (snc_str == "Senescent").astype(int)
adata.obs["SnC_Label"]    = snc_str
n_snc   = int((adata.obs["is_senescent"] == 1).sum())
pct_snc = n_snc / adata.n_obs * 100
print(f"  is_senescent  ← {META['sen_label']!r}  "
      f"({n_snc:,} of {adata.n_obs:,}  =  {pct_snc:.2f}% SnC)")

# ─────────────────────────────────────────────────────────────────────────────
# §1.4 log10_total_counts
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n▸ §1.4 log10_total_counts")

if "log10_total_counts" in adata.obs.columns:
    print(f"  ✓ log10_total_counts already present")
else:
    umi_col = find_umi_col(adata.obs)
    if umi_col is None:
        raise KeyError(
            f"  ✗ Cannot find UMI column. Tried: "
            f"{[META.get('n_counts'), 'nCount_RNA', 'total_counts', 'n_counts']}"
        )
    umi_vals = pd.to_numeric(adata.obs[umi_col], errors="coerce") \
                  .fillna(1).clip(lower=1).values
    adata.obs["log10_total_counts"] = np.log10(umi_vals)
    print(f"  ✓ log10_total_counts ← log10({umi_col!r})  "
          f"(range [{adata.obs['log10_total_counts'].min():.2f}, "
          f"{adata.obs['log10_total_counts'].max():.2f}])")

print(f"\n✓ §1 Setup + core load complete")
print(f"  Next: §1.4a will discover candidate pathology columns")
print(f"        §1.4b will assign canonical names from discovery")
print(f"        §1.5  will apply the rename and set HAS_* flags")

---
## 03 · Pathology column discovery

**Why.** Scans `adata.obs` for tau / amyloid / Braak / CERAD columns. Disease mode only — in aging mode it finds nothing and the module proceeds on `Study_Group` alone.

In [ ]:
# =============================================================================
# §1.4a — PATHOLOGY COLUMN DISCOVERY
# =============================================================================
# Scans adata.obs.columns against PATHOLOGY_CONCEPTS (defined in §0) and prints
# candidate columns for each concept. Purely diagnostic — no side effects.
#
# For each concept:
#   • Substring-match column names against the concept's patterns
#   • Profile each match: dtype, n_unique, range/levels, donor-level invariance
#   • Print a categorized report
#   • Suggest a copy-pasteable PATHOLOGY_META dict for §1.4b
#
# Donor-level invariance check: pathology variables should be constant within
# each donor. A column that varies within donor isn't donor-level metadata.
# =============================================================================

print("=" * 80)
print(f"§1.4a — PATHOLOGY COLUMN DISCOVERY")
print("=" * 80)

# ─────────────────────────────────────────────────────────────────────────────
# Helpers (local to discovery)
# ─────────────────────────────────────────────────────────────────────────────

def _is_donor_level(series_obs, donor_series):
    """Check if a column is constant within each donor."""
    try:
        # Build a (donor, value) frame and check unique values per donor
        df_check = pd.DataFrame({
            "donor": donor_series.astype(str).values,
            "val":   series_obs.values,
        })
        # NaN-tolerant: count distinct non-NaN values per donor
        per_donor_nuniq = df_check.groupby("donor")["val"].nunique(dropna=True)
        # Donor-level if max distinct values per donor ≤ 1
        return bool(per_donor_nuniq.max() <= 1)
    except Exception:
        return False

def _profile_column(col_name, obs, donor_series):
    """Return a dict describing the column."""
    s = obs[col_name]
    dtype_str = str(s.dtype)
    is_numeric = pd.api.types.is_numeric_dtype(s)
    n_unique  = s.nunique(dropna=True)
    n_missing = s.isna().sum()
    pct_missing = 100.0 * n_missing / len(s)
    
    profile = {
        "name":        col_name,
        "dtype":       dtype_str,
        "is_numeric":  is_numeric,
        "n_unique":    n_unique,
        "n_missing":   int(n_missing),
        "pct_missing": pct_missing,
        "donor_level": _is_donor_level(s, donor_series),
    }
    
    # Range or levels summary
    if is_numeric:
        s_clean = pd.to_numeric(s, errors="coerce").dropna()
        if len(s_clean) > 0:
            profile["min"]  = float(s_clean.min())
            profile["max"]  = float(s_clean.max())
            profile["mean"] = float(s_clean.mean())
            profile["summary"] = f"range [{profile['min']:.2f}, {profile['max']:.2f}]"
        else:
            profile["summary"] = "(all NaN)"
    else:
        levels = sorted(s.dropna().astype(str).unique().tolist())
        if len(levels) <= 10:
            profile["levels"]  = levels
            profile["summary"] = "{" + ", ".join(levels) + "}"
        else:
            profile["levels"]  = levels[:5]
            profile["summary"] = f"{n_unique} unique (first 5: {levels[:5]})"
    
    return profile

def _match_concept(obs_columns, patterns):
    """Return obs columns whose name contains any of the patterns (case-insensitive).
    Excludes columns that look like Module 01 senescence outputs."""
    matches = []
    excluded_substrings = ["sen_score", "senescence_score", "is_senescent",
                           "snc_label", "_snc"]
    for col in obs_columns:
        col_lower = col.lower()
        if any(excl in col_lower for excl in excluded_substrings):
            continue
        for pat in patterns:
            if pat.lower() in col_lower:
                matches.append(col)
                break
    return matches

def _format_profile_line(p):
    """One-line summary of a column profile for printing."""
    dl_flag = "✓" if p["donor_level"] else "✗"
    miss_flag = ""
    if p["pct_missing"] > 50:
        miss_flag = f"  ⚠ {p['pct_missing']:.0f}% missing"
    elif p["pct_missing"] > 0:
        miss_flag = f"  ({p['pct_missing']:.0f}% missing)"
    return (f"    {p['name']:<25s} {p['dtype']:<10s} "
            f"{p['n_unique']:>4d} unique  donor-level:{dl_flag}  "
            f"{p['summary']}{miss_flag}")

# ─────────────────────────────────────────────────────────────────────────────
# Scan
# ─────────────────────────────────────────────────────────────────────────────
all_obs_cols = list(adata.obs.columns)
donor_series = adata.obs["Donor"]

print(f"\nScanning {len(all_obs_cols)} obs columns against "
      f"{len(PATHOLOGY_CONCEPTS)} pathology concepts...\n")

# Per-concept results
concept_matches = {}        # concept → list of profile dicts
suggested_pick  = {}        # concept → str (best column) or None

for concept, info in PATHOLOGY_CONCEPTS.items():
    print(f"▸ Concept: {concept}  ({info['description']})")
    
    matched_cols = _match_concept(all_obs_cols, info["patterns"])
    
    if not matched_cols:
        print(f"    (no candidates found)\n")
        concept_matches[concept] = []
        suggested_pick[concept]  = None
        continue
    
    # Profile each match
    profiles = []
    for col in matched_cols:
        p = _profile_column(col, adata.obs, donor_series)
        profiles.append(p)
    
    # Print all matches
    for p in profiles:
        print(_format_profile_line(p))
    
    # Pick the "best" candidate for the suggestion:
    #   1. Must be donor-level
    #   2. Must have <50% missing
    #   3. Must have ≥2 unique values
    #   4. Prefer exact-name match (e.g. concept "tau" ← column "tau")
    valid = [p for p in profiles
             if p["donor_level"] and p["pct_missing"] < 50 and p["n_unique"] >= 2]
    
    if not valid:
        print(f"    ⊘ No candidate passes filters (donor-level, <50% missing, ≥2 levels)")
        suggested_pick[concept] = None
    else:
        # Prefer exact name match
        exact = [p for p in valid if p["name"].lower() == concept.lower()]
        if exact:
            suggested_pick[concept] = exact[0]["name"]
        else:
            # Otherwise prefer column whose name starts with a pattern
            for pat in info["patterns"]:
                starts_with = [p for p in valid if p["name"].lower().startswith(pat.lower())]
                if starts_with:
                    suggested_pick[concept] = starts_with[0]["name"]
                    break
            else:
                # Fall back to first valid
                suggested_pick[concept] = valid[0]["name"]
    
    concept_matches[concept] = profiles
    print()

# ─────────────────────────────────────────────────────────────────────────────
# Print suggested PATHOLOGY_META
# ─────────────────────────────────────────────────────────────────────────────
print("─" * 80)
print(f"  SUGGESTED §1.4b ASSIGNMENT — copy into next cell, edit as needed:")
print("─" * 80)

print()
print("PATHOLOGY_META = {")
for concept in PATHOLOGY_CONCEPTS.keys():
    pick = suggested_pick.get(concept)
    if pick is None:
        print(f'    {repr(concept) + ":":<22s} None,')
    else:
        # Find the profile so we can annotate
        prof = next((p for p in concept_matches[concept] if p["name"] == pick), None)
        comment = ""
        if prof is not None:
            comment = f"  # {prof['summary']}"
        print(f'    {repr(concept) + ":":<22s} {repr(pick) + ",":<25s}{comment}')
print("}")
print()

# Suggest pathology_group reference + ordering when relevant
pg_pick = suggested_pick.get("pathology_group")
if pg_pick is not None:
    pg_levels = sorted(adata.obs[pg_pick].dropna().astype(str).unique().tolist())
    print(f"# pathology_group levels in data: {pg_levels}")
    
    # Heuristic ordering: known patterns first, else as-is
    natural_orderings = [
        ["no-pathology", "early-pathology", "late-pathology"],
        ["low", "intermediate", "high"],
        ["none", "mild", "moderate", "severe"],
        ["0", "1", "2", "3", "4", "5", "6"],   # Braak-like
    ]
    suggested_order = None
    for nat in natural_orderings:
        if all(lv in pg_levels for lv in nat) or set(pg_levels).issubset(set(nat)):
            suggested_order = [lv for lv in nat if lv in pg_levels]
            break
    if suggested_order is None:
        suggested_order = pg_levels
    
    suggested_ref = suggested_order[0] if suggested_order else None
    
    print(f'PATHOLOGY_GROUP_REFERENCE   = {repr(suggested_ref)}')
    print(f'PATHOLOGY_GROUP_LEVEL_ORDER = {suggested_order}')
else:
    print(f"# No pathology_group candidate found")
    print(f"PATHOLOGY_GROUP_REFERENCE   = None")
    print(f"PATHOLOGY_GROUP_LEVEL_ORDER = []")

print()
print("─" * 80)
print(f"\n✓ §1.4a Discovery complete")
print(f"  Next: edit the suggestion above, paste into §1.4b, run §1.4b to validate")

---
## 04 · Pathology assignment

**Why.** Builds `pathology_group` from whatever the previous section found, and sets its reference level and ordering.

In [ ]:
# =============================================================================
# §1.4b — PATHOLOGY ASSIGNMENT
# =============================================================================
# User-edited cell. Paste the suggestion from §1.4a here, then edit as needed:
#   • Change a column choice (e.g. swap 'amyloid' → 'plaq_n' for Mathys)
#   • Set unused concepts to None
#   • Edit PATHOLOGY_GROUP_REFERENCE if you want a different baseline
#   • Edit PATHOLOGY_GROUP_LEVEL_ORDER for adjacent contrasts in §5
#
# After editing, re-run this cell. It validates and reports any issues. Then
# §1.5 applies the rename to canonical names and sets HAS_* flags downstream
# sections key off.
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# EDIT THESE — paste from §1.4a, then customize
# ─────────────────────────────────────────────────────────────────────────────

PATHOLOGY_META = {
    'pathology_group':     None,
    'tau':                 'Braak',                   # {Stage I, Stage II, missing or unknown}
    'amyloid':             'amyCerad',                # {None/No AD/C0, missing or unknown}
    'cognition':           None,
    'apoe':                'apoeGenotype',            # {22, 23, 24, 33, 34, 44, missing or unknown}
    'pmi':                 'PMI',                     # range [2.33, 46.50]
}

# No pathology_group candidate found
PATHOLOGY_GROUP_REFERENCE   = None
PATHOLOGY_GROUP_LEVEL_ORDER = []

# ─────────────────────────────────────────────────────────────────────────────
# Validate
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 80)
print(f"§1.4b — PATHOLOGY ASSIGNMENT VALIDATION")
print("=" * 80)

# Check assignments are well-formed
all_valid = True
problems  = []

# Check PATHOLOGY_META keys match PATHOLOGY_CONCEPTS keys
known_concepts = set(PATHOLOGY_CONCEPTS.keys())
assigned_keys  = set(PATHOLOGY_META.keys())

unknown_keys = assigned_keys - known_concepts
missing_keys = known_concepts - assigned_keys

if unknown_keys:
    problems.append(f"Unknown concepts in PATHOLOGY_META: {sorted(unknown_keys)}")
    all_valid = False
if missing_keys:
    problems.append(f"Missing concepts (set to None if unused): {sorted(missing_keys)}")
    all_valid = False

# Check each assigned column actually exists in obs
print(f"\nValidating column assignments against adata.obs:\n")
for concept in PATHOLOGY_CONCEPTS.keys():
    source = PATHOLOGY_META.get(concept)
    if source is None:
        print(f"  ⊘ {concept:18s}  not assigned (downstream sections will skip)")
        continue
    if source not in adata.obs.columns:
        print(f"  ✗ {concept:18s}  ← {source!r}  NOT IN OBS")
        problems.append(f"{concept!r} → {source!r} not in adata.obs")
        all_valid = False
        continue
    # Profile and report
    s = adata.obs[source]
    n_uniq    = s.nunique(dropna=True)
    n_missing = int(s.isna().sum())
    pct_miss  = 100.0 * n_missing / len(s)
    
    if n_uniq < 2:
        print(f"  ⚠ {concept:18s}  ← {source!r}  only {n_uniq} unique value(s)")
        problems.append(f"{concept!r} → {source!r} has <2 unique values")
        all_valid = False
        continue
    
    miss_str = f"  ({pct_miss:.0f}% missing)" if pct_miss > 0 else ""
    if pd.api.types.is_numeric_dtype(s):
        s_clean = pd.to_numeric(s, errors="coerce").dropna()
        rng = f"range [{s_clean.min():.2f}, {s_clean.max():.2f}]"
        print(f"  ✓ {concept:18s}  ← {source!r}  ({n_uniq} unique, {rng}){miss_str}")
    else:
        levels = sorted(s.dropna().astype(str).unique().tolist())
        if len(levels) <= 6:
            lv_str = "{" + ", ".join(levels) + "}"
        else:
            lv_str = f"{n_uniq} levels"
        print(f"  ✓ {concept:18s}  ← {source!r}  {lv_str}{miss_str}")

# Validate pathology_group reference + ordering
print()
pg_source = PATHOLOGY_META.get("pathology_group")
if pg_source is not None and pg_source in adata.obs.columns:
    pg_levels_in_data = sorted(adata.obs[pg_source].dropna().astype(str).unique().tolist())
    
    print(f"  pathology_group levels in data: {pg_levels_in_data}")
    
    # Check reference
    if PATHOLOGY_GROUP_REFERENCE is None:
        print(f"  ⚠ PATHOLOGY_GROUP_REFERENCE is None but pathology_group is assigned")
        problems.append("PATHOLOGY_GROUP_REFERENCE must be set when pathology_group is assigned")
        all_valid = False
    elif PATHOLOGY_GROUP_REFERENCE not in pg_levels_in_data:
        print(f"  ✗ PATHOLOGY_GROUP_REFERENCE={PATHOLOGY_GROUP_REFERENCE!r} "
              f"NOT in data levels {pg_levels_in_data}")
        problems.append(f"PATHOLOGY_GROUP_REFERENCE {PATHOLOGY_GROUP_REFERENCE!r} not in data")
        all_valid = False
    else:
        n_ref = (adata.obs[pg_source].astype(str) == PATHOLOGY_GROUP_REFERENCE).sum()
        print(f"  ✓ PATHOLOGY_GROUP_REFERENCE   = {PATHOLOGY_GROUP_REFERENCE!r}  "
              f"({n_ref:,} cells)")
    
    # Check ordering
    if not PATHOLOGY_GROUP_LEVEL_ORDER:
        print(f"  ⚠ PATHOLOGY_GROUP_LEVEL_ORDER is empty — defaulting to alphabetical")
    else:
        order_in_data = [lv for lv in PATHOLOGY_GROUP_LEVEL_ORDER
                         if lv in pg_levels_in_data]
        missing_from_order = [lv for lv in pg_levels_in_data
                              if lv not in PATHOLOGY_GROUP_LEVEL_ORDER]
        extra_in_order = [lv for lv in PATHOLOGY_GROUP_LEVEL_ORDER
                          if lv not in pg_levels_in_data]
        
        print(f"  ✓ PATHOLOGY_GROUP_LEVEL_ORDER = {PATHOLOGY_GROUP_LEVEL_ORDER}")
        if missing_from_order:
            print(f"     ⚠ levels in data NOT in ordering "
                  f"(will be appended alpha): {missing_from_order}")
        if extra_in_order:
            print(f"     ⚠ levels in ordering NOT in data (ignored): {extra_in_order}")
else:
    if PATHOLOGY_GROUP_REFERENCE is not None or PATHOLOGY_GROUP_LEVEL_ORDER:
        print(f"  ⚠ pathology_group not assigned but REFERENCE/LEVEL_ORDER set "
              f"(will be ignored)")

# ─────────────────────────────────────────────────────────────────────────────
# Final verdict
# ─────────────────────────────────────────────────────────────────────────────
print()
print("─" * 80)
if all_valid:
    n_assigned = sum(1 for v in PATHOLOGY_META.values() if v is not None)
    print(f"  ✓ §1.4b ASSIGNMENT VALID — {n_assigned}/{len(PATHOLOGY_CONCEPTS)} concepts assigned")
    print(f"  Run §1.5 next to apply the rename.")
else:
    print(f"  ✗ §1.4b VALIDATION FAILED — {len(problems)} problem(s):")
    for p in problems:
        print(f"      • {p}")
    print(f"\n  Fix the issues above, then re-run this cell.")
print("─" * 80)

---
## 05 · Apply pathology rename, set flags

**Why.** Writes the harmonized labels back and sets `HAS_PATHOLOGY_GROUP` / `STRATIFY_VAR`, which decide what sections 23-28 iterate over.

In [ ]:
# =============================================================================
# §1.5 — APPLY PATHOLOGY RENAME + SET HAS_* FLAGS + STRATIFY_VAR
# =============================================================================
# Reads PATHOLOGY_META from §1.4b. For each concept whose source column exists
# in obs, renames it to the canonical concept name (e.g. 'Final_NFTs_per_mm2_Mario'
# → 'tau'). Then sets HAS_* flags that downstream sections key off.
#
# Flag-setting principle:
#   A canonical column is considered USABLE only if it both exists AND carries
#   enough non-null, non-placeholder values to actually model with. A column
#   that exists but is all-NaN (e.g. PsychAD aging's empty 'tau' column), or
#   has only one level, or is composed entirely of placeholder strings like
#   'Not Evaluated', is treated as ABSENT for gating purposes. This prevents
#   §7 sections from entering and crashing on empty data.
#
#   HAS_PATHOLOGY_GROUP — categorical staging available?
#   HAS_TAU             — continuous tau biomarker available?
#   HAS_AMYLOID         — continuous amyloid biomarker available?
#   HAS_COGNITION       — cognitive score available?
#   HAS_APOE            — APOE genotype available?
#   HAS_PMI             — postmortem interval available?
#   HAS_BIOMARKERS      — at least one of (tau, amyloid) available
#                          (gates §7a pathology_index)
#
# Also derives:
#   STRATIFY_VAR        — categorical stratification variable for §7b plots
#                         and §7c interaction tests:
#                            'pathology_group' if HAS_PATHOLOGY_GROUP
#                            else 'Study_Group' (clinical diagnosis fallback)
#                         §3/§4/§5 are NOT affected — they always use Study_Group.
#                         Only §7b/§7c switch behavior on STRATIFY_VAR.
#
#   STRATIFY_LEVELS     — declared order of levels (for ordered plots)
#   STRATIFY_REFERENCE  — baseline level for contrasts
#
# After this cell, every downstream section refers ONLY to canonical names
# (tau, amyloid, pathology_group, etc.) — never to the raw source names.
# =============================================================================

print("=" * 80)
print(f"§1.5 — APPLY PATHOLOGY RENAME + SET FLAGS + STRATIFY_VAR")
print("=" * 80)

# ─────────────────────────────────────────────────────────────────────────────
# Build rename map from PATHOLOGY_META (skip None entries)
# ─────────────────────────────────────────────────────────────────────────────
rename_map = {}
print(f"\nApplying rename:")
for concept in PATHOLOGY_CONCEPTS.keys():
    source = PATHOLOGY_META.get(concept)

    if source is None:
        print(f"  ⊘ {concept:18s}  not assigned — skipping")
        continue

    if source not in adata.obs.columns:
        print(f"  ✗ {concept:18s}  ← {source!r}  NOT IN OBS  (re-run §1.4b)")
        continue

    if source == concept:
        print(f"  ✓ {concept:18s}  (already canonical)")
    else:
        # Check for collision: canonical name already exists in obs
        if concept in adata.obs.columns:
            print(f"  ⚠ {concept:18s}  ← {source!r}  "
                  f"canonical name already in obs — dropping existing first")
            adata.obs = adata.obs.drop(columns=[concept])
        rename_map[source] = concept
        print(f"  ✓ {concept:18s}  ← {source!r}")

if rename_map:
    adata.obs = adata.obs.rename(columns=rename_map)

# Final dup sweep
n_dup = adata.obs.columns.duplicated().sum()
if n_dup > 0:
    print(f"\n  ⚠ Dropped {n_dup} duplicate column(s) after rename")
    adata.obs = adata.obs.loc[:, ~adata.obs.columns.duplicated(keep="first")]

# ─────────────────────────────────────────────────────────────────────────────
# Set HAS_* flags from canonical names
#
# Each flag answers two questions, in this order:
#   (1) Does the canonical column exist in adata.obs (post-rename)?
#   (2) Does it carry enough usable values to model with?
#
# Donor-level views: pathology metadata is donor-invariant, so we deduplicate
# to one value per donor before counting valid entries. This avoids cells from
# large donors swamping the count.
# ─────────────────────────────────────────────────────────────────────────────
MIN_DONORS_FOR_FLAG = 10   # require ≥ this many donors with usable values

# Placeholder strings filtered out of categorical level counts
CATEGORICAL_PLACEHOLDERS = {
    "not evaluated", "not assessed", "not reported", "unevaluated",
    "unknown", "missing", "n/a", "na", "none", "",
}

def _donor_values(col):
    """Return per-donor (deduplicated) values for a canonical column.
    Returns None if column missing."""
    if col not in adata.obs.columns:
        return None
    return (adata.obs[["Donor", col]]
                  .drop_duplicates(subset="Donor")
                  .set_index("Donor")[col])

def _check_numeric(col, min_donors=MIN_DONORS_FOR_FLAG):
    """Numeric column with ≥min_donors non-null values AND variance > 0.
    Returns (flag, reason)."""
    s = _donor_values(col)
    if s is None:
        return False, "column missing"
    s_num = pd.to_numeric(s, errors="coerce")
    n_valid = int(s_num.notna().sum())
    if n_valid == 0:
        return False, "0 donors with values"
    if n_valid < min_donors:
        return False, f"only {n_valid} donor(s) with values (<{min_donors})"
    if s_num.dropna().nunique() < 2:
        return False, f"all {n_valid} values identical ({s_num.dropna().iloc[0]!r})"
    return True, (f"{n_valid} donors, "
                  f"range [{s_num.min():.3g}, {s_num.max():.3g}]")

def _check_categorical(col, min_donors=MIN_DONORS_FOR_FLAG, min_levels=2):
    """Categorical column with ≥min_levels levels, each having ≥min_donors,
    after filtering out placeholder strings.
    Returns (flag, reason)."""
    s = _donor_values(col)
    if s is None:
        return False, "column missing"
    s_str = (s.astype(str).str.strip()
              .replace({lv: np.nan for lv in CATEGORICAL_PLACEHOLDERS},
                       regex=False))
    # Also catch case-variants of placeholders
    s_str_lower = s_str.dropna().str.lower()
    placeholder_mask = s_str_lower.isin(CATEGORICAL_PLACEHOLDERS)
    s_clean = s_str.dropna()[~placeholder_mask.values]

    counts = s_clean.value_counts()
    if len(counts) == 0:
        return False, "0 informative donors (all placeholders or NaN)"

    informative_levels = counts[counts >= min_donors]
    if len(informative_levels) < min_levels:
        return False, (f"only {len(informative_levels)} level(s) "
                       f"with ≥{min_donors} donors. All levels: {counts.to_dict()}")
    return True, f"{len(informative_levels)} levels, counts: {dict(counts)}"

def _check_genotype(col=None, min_donors=MIN_DONORS_FOR_FLAG):
    """APOE-style genotype: column exists with ≥min_donors non-null values."""
    col = col or "apoe"
    s = _donor_values(col)
    if s is None:
        return False, "column missing"
    n_valid = int(s.dropna().shape[0])
    if n_valid < min_donors:
        return False, f"only {n_valid} donor(s) with genotype"
    return True, f"{n_valid} donors with genotype"

# Apply checks
_pg_ok,   _pg_reason   = _check_categorical("pathology_group")
_tau_ok,  _tau_reason  = _check_numeric("tau")
_amy_ok,  _amy_reason  = _check_numeric("amyloid")
_cog_ok,  _cog_reason  = _check_numeric("cognition")
_apoe_ok, _apoe_reason = _check_genotype("apoe")
_pmi_ok,  _pmi_reason  = _check_numeric("pmi")

HAS_PATHOLOGY_GROUP = _pg_ok
HAS_TAU             = _tau_ok
HAS_AMYLOID         = _amy_ok
HAS_COGNITION       = _cog_ok
HAS_APOE            = _apoe_ok
HAS_PMI             = _pmi_ok

HAS_BIOMARKERS    = HAS_TAU or HAS_AMYLOID
HAS_ANY_PATHOLOGY = HAS_PATHOLOGY_GROUP or HAS_BIOMARKERS or HAS_COGNITION

# Bundle reasons for the status panel below
_flag_reasons = {
    "pathology_group": _pg_reason,
    "tau":             _tau_reason,
    "amyloid":         _amy_reason,
    "cognition":       _cog_reason,
    "apoe":            _apoe_reason,
    "pmi":             _pmi_reason,
}

# ─────────────────────────────────────────────────────────────────────────────
# Apply pathology_group categorical ordering (if usable)
# Uses PATHOLOGY_GROUP_LEVEL_ORDER from §1.4b. Levels not in the order get
# appended alphabetically.
# ─────────────────────────────────────────────────────────────────────────────
pg_order_final = []
if HAS_PATHOLOGY_GROUP:
    pg_levels_in_data = sorted(adata.obs["pathology_group"].dropna()
                                .astype(str).unique().tolist())

    declared = [lv for lv in PATHOLOGY_GROUP_LEVEL_ORDER
                if lv in pg_levels_in_data]
    extras   = sorted([lv for lv in pg_levels_in_data
                       if lv not in PATHOLOGY_GROUP_LEVEL_ORDER])
    pg_order_final = declared + extras

    adata.obs["pathology_group"] = pd.Categorical(
        adata.obs["pathology_group"].astype(str),
        categories=pg_order_final,
        ordered=True,
    )
    print(f"\n  pathology_group categorical order: {pg_order_final}")

# ─────────────────────────────────────────────────────────────────────────────
# Derive STRATIFY_VAR for §7b plots / §7c interactions
# Option B fallback: pathology_group → Study_Group when pathology_group absent.
# §3/§4/§5/§8.x are NOT affected by this — they iterate PRIMARY_SPECS only.
# ─────────────────────────────────────────────────────────────────────────────
if HAS_PATHOLOGY_GROUP:
    STRATIFY_VAR       = "pathology_group"
    STRATIFY_LEVELS    = list(pg_order_final)
    STRATIFY_REFERENCE = (PATHOLOGY_GROUP_REFERENCE
                          if PATHOLOGY_GROUP_REFERENCE in pg_order_final
                          else (pg_order_final[0] if pg_order_final else None))
    STRATIFY_SOURCE    = "pathology_group (categorical staging from §1.4b)"
else:
    STRATIFY_VAR       = "Study_Group"
    # Levels/reference are finalized in §2.3 after Study_Group ordering is set.
    # Use REFERENCE_GROUP from §0 as the baseline; levels populated later.
    STRATIFY_LEVELS    = None
    STRATIFY_REFERENCE = REFERENCE_GROUP
    STRATIFY_SOURCE    = ("Study_Group fallback — pathology_group not assigned, "
                          "using clinical diagnosis as staging axis")

# ─────────────────────────────────────────────────────────────────────────────
# Resolve RUN_PATHOLOGY_SECTIONS toggle
# (auto/on/off from §0; final boolean for §7 gating)
# ─────────────────────────────────────────────────────────────────────────────
if RUN_PATHOLOGY_SECTIONS == "off":
    RUN_PATHOLOGY    = False
    pathology_status = "OFF (forced via RUN_PATHOLOGY_SECTIONS='off')"
elif RUN_PATHOLOGY_SECTIONS == "on":
    if not HAS_ANY_PATHOLOGY:
        raise RuntimeError(
            f"RUN_PATHOLOGY_SECTIONS='on' but no usable pathology data "
            f"detected.\n"
            f"  Either assign tau / amyloid / pathology_group / cognition in §1.4b\n"
            f"  with columns that have ≥{MIN_DONORS_FOR_FLAG} populated donors,\n"
            f"  or set RUN_PATHOLOGY_SECTIONS='auto' in §0."
        )
    RUN_PATHOLOGY    = True
    pathology_status = "ON (forced via RUN_PATHOLOGY_SECTIONS='on')"
else:  # "auto"
    RUN_PATHOLOGY    = HAS_ANY_PATHOLOGY
    pathology_status = (f"AUTO → "
                        f"{'enabled' if RUN_PATHOLOGY else 'skipped (no usable data)'}")

# ─────────────────────────────────────────────────────────────────────────────
# Status panel
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'─' * 80}")
print(f"  §1.5 STATUS PANEL  (min_donors_for_flag = {MIN_DONORS_FOR_FLAG})")
print(f"{'─' * 80}")

print(f"\n  Canonical pathology columns — flag + reason:")
flags = [
    ("pathology_group", HAS_PATHOLOGY_GROUP, "categorical staging"),
    ("tau",             HAS_TAU,             "continuous tau / NFT"),
    ("amyloid",         HAS_AMYLOID,         "continuous amyloid / Aβ"),
    ("cognition",       HAS_COGNITION,       "cognitive score"),
    ("apoe",            HAS_APOE,            "APOE genotype"),
    ("pmi",             HAS_PMI,             "postmortem interval"),
]
for canonical, flag, descr in flags:
    mark = "✓" if flag else "⊘"
    reason = _flag_reasons.get(canonical, "")
    print(f"    {mark} {canonical:18s} {descr:25s}  {reason}")

print(f"\n  Aggregate flags:")
print(f"    HAS_BIOMARKERS    = {HAS_BIOMARKERS}    (HAS_TAU or HAS_AMYLOID)")
print(f"    HAS_ANY_PATHOLOGY = {HAS_ANY_PATHOLOGY}    "
      f"(group or biomarkers or cognition)")

print(f"\n  Stratification variable for §7b/§7c (descriptive plots + interactions):")
print(f"    STRATIFY_VAR       = {STRATIFY_VAR!r}")
print(f"    STRATIFY_REFERENCE = {STRATIFY_REFERENCE!r}")
if STRATIFY_LEVELS:
    print(f"    STRATIFY_LEVELS    = {STRATIFY_LEVELS}")
else:
    print(f"    STRATIFY_LEVELS    = (set in §2.3 after Study_Group ordering)")
print(f"    source             : {STRATIFY_SOURCE}")

print(f"\n  Section gating:")
print(f"    §3, §4, §5, §6, §8.x  : will run (always)")
print(f"    §7a (pathology_index) : {pathology_status}")
print(f"    §7b (sen ~ pathology) : {'will run' if RUN_PATHOLOGY else 'will skip'}"
      f"  (requires §7a + at least one biomarker)")
print(f"    §7c (interaction)     : {'will run' if RUN_PATHOLOGY else 'will skip'}"
      f"  (requires §7a/§7b)")

if RUN_PATHOLOGY and not HAS_BIOMARKERS:
    print(f"\n  ⚠ Pathology sections enabled but no continuous biomarkers (tau/amyloid)")
    print(f"    §7a will skip pathology_index construction")
    print(f"    §7b will skip continuous models — only categorical models on STRATIFY_VAR")

if RUN_PATHOLOGY and not HAS_PATHOLOGY_GROUP:
    print(f"\n  ℹ §7b/§7c will use Study_Group as stratification axis "
          f"(pathology_group not assigned)")
    print(f"     Continuous biomarker models (tau, amyloid, pathology_index) "
          f"are unaffected — they don't need a grouping variable.")

print(f"\n✓ §1.5 Apply complete")
print(f"  Next: §1.6 cell-type filter → §1.7-1.9 build df_cells/df_donor_ct/df_donor")
print(f"        → §1.10 validate primary variables → §1.11 run plan summary")

---
## 06 · Cell-type inventory

**Why.** Counts cells and donors per cell type. No filtering — this is the census that later minimum-cell rules are judged against.

In [ ]:
# =============================================================================
# §1.6 — CELL-TYPE INVENTORY  (no filtering)
# =============================================================================
# Inventory pass — does not drop cells. Each downstream section decides
# its own coverage:
#   §3/§4/§5/§7/§8.x : iterate all CTs present in df_cells (with their own
#                        per-CT MIN_CELLS_PER_DONOR check)
#   §5c/§5d Sloan / cell-cycle validation : iterate CTs in LINEAGE_CONFIGS
#                        (CTs without a validation panel just don't get a
#                        validation row — not an error)
#
# We only drop cells with an empty/missing Cell_Type label, because those
# can't enter any per-CT model.
# =============================================================================

print("=" * 80)
print(f"§1.6 — CELL-TYPE INVENTORY")
print("=" * 80)

# ─────────────────────────────────────────────────────────────────────────────
# Drop unannotated cells (empty-string or NaN Cell_Type only)
# ─────────────────────────────────────────────────────────────────────────────
n_before = adata.n_obs
ct_str = adata.obs["Cell_Type"].astype(str)
unannot_mask = ct_str.isin(["", "nan", "NaN", "None"]) | adata.obs["Cell_Type"].isna()
n_unannot = int(unannot_mask.sum())

if n_unannot > 0:
    print(f"  Dropping {n_unannot:,} cell(s) with missing/empty Cell_Type label")
    adata = adata[~unannot_mask, :].copy()

# ─────────────────────────────────────────────────────────────────────────────
# Inventory
# ─────────────────────────────────────────────────────────────────────────────
ct_counts = adata.obs["Cell_Type"].value_counts()
all_cts_present = ct_counts.index.tolist()

# Optional FOCUS_CELL_TYPES from §0 — applied here as advisory only,
# users who set it explicitly want to restrict
if "FOCUS_CELL_TYPES" in dir() and FOCUS_CELL_TYPES:
    typos = [ct for ct in FOCUS_CELL_TYPES if ct not in all_cts_present]
    if typos:
        print(f"  ⚠ FOCUS_CELL_TYPES contains CTs not in data: {typos}")
    keep_mask = adata.obs["Cell_Type"].astype(str).isin(FOCUS_CELL_TYPES)
    n_focus_drop = int((~keep_mask).sum())
    if n_focus_drop > 0:
        print(f"  Applying FOCUS_CELL_TYPES → keeping {(~keep_mask).sum():,} of "
              f"{n_focus_drop:,} non-focus cells dropped")
        adata = adata[keep_mask, :].copy()
    ct_counts = adata.obs["Cell_Type"].value_counts()
    all_cts_present = ct_counts.index.tolist()

# ─────────────────────────────────────────────────────────────────────────────
# Print inventory with validation-panel availability
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n  Cells: {n_before:,} → {adata.n_obs:,}  "
      f"(dropped {n_before - adata.n_obs:,})")

has_lineage_config = "LINEAGE_CONFIGS" in dir() or "LINEAGE_CONFIGS" in globals()
configured_cts = set(LINEAGE_CONFIGS.keys()) if has_lineage_config else set()

print(f"\n  Cell-type inventory (post unannotated-drop):")
print(f"  {'Cell Type':<25} {'N cells':>10}  {'§5c/§5d':<8}")
print(f"  {'─'*47}")
for ct, n in ct_counts.items():
    val_panel = "✓ panel" if ct in configured_cts else "⊘ skip"
    print(f"    {ct:<23} {n:>10,}  {val_panel}")

# ─────────────────────────────────────────────────────────────────────────────
# Set ACTIVE_CELL_TYPES + CELLTYPE_ORDER_PLOT for downstream
# ─────────────────────────────────────────────────────────────────────────────
ACTIVE_CELL_TYPES = list(all_cts_present)

# CELLTYPE_ORDER_PLOT: prefer the order from §0 if it covers active CTs,
# then append remaining alphabetically
if "CELLTYPE_ORDER_PLOT" in dir() and CELLTYPE_ORDER_PLOT:
    declared = [ct for ct in CELLTYPE_ORDER_PLOT if ct in ACTIVE_CELL_TYPES]
    extras   = sorted([ct for ct in ACTIVE_CELL_TYPES if ct not in declared])
    CELLTYPE_ORDER_PLOT = declared + extras
else:
    CELLTYPE_ORDER_PLOT = sorted(ACTIVE_CELL_TYPES)

print(f"\n  ACTIVE_CELL_TYPES   : {len(ACTIVE_CELL_TYPES)} CT(s)")
print(f"  CELLTYPE_ORDER_PLOT : {CELLTYPE_ORDER_PLOT}")

# Validation coverage breakdown
n_with_panel = sum(1 for ct in ACTIVE_CELL_TYPES if ct in configured_cts)
n_without    = len(ACTIVE_CELL_TYPES) - n_with_panel
print(f"\n  Validation panel coverage: {n_with_panel}/{len(ACTIVE_CELL_TYPES)} CTs have "
      f"LINEAGE_CONFIGS entries (§5c/§5d will fit on these only)")
if n_without > 0:
    no_panel = [ct for ct in ACTIVE_CELL_TYPES if ct not in configured_cts]
    print(f"  CTs without panel : {no_panel}")
    print(f"  These run in §3/§4/§5/§7/§8.x but skip §5c/§5d validation rows.")

print(f"\n✓ §1.6 cell-type inventory complete (no filtering applied)")

---
## 07 · Cell-level frame

**Why.** `df_cells`: one row per cell, carrying donor, cell type, senescence call, score, group, and depth. Everything cell-level reads from this.

In [ ]:
# =============================================================================
# §1.7 — BUILD df_cells (cell-level frame)
# =============================================================================
# Cell-level frame for §5 GLMM and §8.x.
#
# This cell is self-sufficient: defines the per-dataset Cell_Type rename map
# locally, applies it to df_cells, and refreshes CELLTYPE_ORDER_PLOT /
# ACTIVE_CELL_TYPES from the harmonized data. Drops in cleanly even if §0
# and §1.2 weren't patched.
#
# After harmonization, every Cell_Type in df_cells matches a LINEAGE_CONFIGS
# key when possible. CTs that don't (e.g. VLMC) still run in §3/§4/§5/§7/§8
# but skip §5c/§5d validation overlay.
# =============================================================================

print("=" * 80)
print(f"§1.7 — BUILD df_cells")
print("=" * 80)

# ─────────────────────────────────────────────────────────────────────────────
# Per-dataset Cell_Type rename map (defined locally — no §0 dependency)
# ─────────────────────────────────────────────────────────────────────────────
CELLTYPE_RENAME_BY_DATASET = {
    "australian": {
        "Glutamatergic":   "Excitatory",
        "GABAergic":       "Inhibitory",
        "Microglia-PVM":   "Microglia",
    },
    "mathys":          {},
    "psychad_ad":      {},
    "psychad_aging":   {},
    "terekhova":       {},
    "yazar":           {},
    "ostkamp":         {},
    "gate":            {},
    "brase":           {},
}
CELLTYPE_RENAME = CELLTYPE_RENAME_BY_DATASET.get(DATASET, {})

# ─────────────────────────────────────────────────────────────────────────────
# Helper: refresh ACTIVE_CELL_TYPES + CELLTYPE_ORDER_PLOT from current df_cells
# Defined here so all subsequent cells (§1.8 onward) can call it.
# ─────────────────────────────────────────────────────────────────────────────
def _refresh_celltype_order():
    """Refresh ACTIVE_CELL_TYPES and CELLTYPE_ORDER_PLOT from df_cells."""
    global ACTIVE_CELL_TYPES, CELLTYPE_ORDER_PLOT
    if "df_cells" not in dir() and "df_cells" not in globals():
        return
    present_cts = df_cells["Cell_Type"].value_counts().index.tolist()
    ACTIVE_CELL_TYPES = list(present_cts)
    declared = []
    if ("CELLTYPE_ORDER_PLOT" in dir()) or ("CELLTYPE_ORDER_PLOT" in globals()):
        try:
            declared = [ct for ct in CELLTYPE_ORDER_PLOT if ct in ACTIVE_CELL_TYPES]
        except Exception:
            declared = []
    extras = [ct for ct in ACTIVE_CELL_TYPES if ct not in declared]
    CELLTYPE_ORDER_PLOT = declared + sorted(extras)

# ─────────────────────────────────────────────────────────────────────────────
# Pathology + core columns
# ─────────────────────────────────────────────────────────────────────────────
PATHOLOGY_PASSTHROUGH_COLS = [
    canonical for canonical in
    ["pathology_group", "tau", "amyloid", "cognition", "apoe", "pmi"]
    if canonical in adata.obs.columns
]
print(f"  Pathology passthrough columns: {PATHOLOGY_PASSTHROUGH_COLS}")

CORE_COLS = ["Donor", "Cell_Type", "is_senescent", "sen_score",
             "Study_Group", "Age", "Sex"]
if "Cohort" in adata.obs.columns:
    CORE_COLS.append("Cohort")
if "log10_total_counts" in adata.obs.columns:
    CORE_COLS.append("log10_total_counts")

cell_cols = [c for c in CORE_COLS + PATHOLOGY_PASSTHROUGH_COLS
             if c in adata.obs.columns]
print(f"  df_cells columns: {cell_cols}")

# ─────────────────────────────────────────────────────────────────────────────
# Materialize from adata.obs
# ─────────────────────────────────────────────────────────────────────────────
df_cells = adata.obs[cell_cols].copy()

df_cells["Donor"]        = df_cells["Donor"].astype(str)
df_cells["Cell_Type"]    = df_cells["Cell_Type"].astype(str)
df_cells["is_senescent"] = df_cells["is_senescent"].astype(int)
df_cells["sen_score"]    = pd.to_numeric(df_cells["sen_score"], errors="coerce")
df_cells["Study_Group"]  = df_cells["Study_Group"].astype(str)
df_cells["Age"]          = pd.to_numeric(df_cells["Age"], errors="coerce")
df_cells["Sex"]          = df_cells["Sex"].astype(str)
if "Cohort" in df_cells.columns:
    df_cells["Cohort"]   = df_cells["Cohort"].astype(str)

for c in ["tau", "amyloid", "cognition", "pmi"]:
    if c in df_cells.columns:
        df_cells[c] = pd.to_numeric(df_cells[c], errors="coerce")

for c in ["pathology_group", "apoe"]:
    if c in df_cells.columns:
        df_cells[c] = df_cells[c].astype(str).replace({"nan": np.nan})

if "log10_total_counts" not in df_cells.columns:
    umi_col = find_umi_col(adata.obs)
    if umi_col is not None:
        df_cells["log10_total_counts"] = np.log10(
            pd.to_numeric(adata.obs[umi_col], errors="coerce") + 1
        )
        print(f"  Derived log10_total_counts from {umi_col!r}")

# ─────────────────────────────────────────────────────────────────────────────
# BEFORE-state inventory
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n  Cell_Type values BEFORE harmonization:")
ct_before = df_cells["Cell_Type"].value_counts()
for ct, n in ct_before.items():
    will_rename = ct in CELLTYPE_RENAME
    arrow = f"  → {CELLTYPE_RENAME[ct]!r}" if will_rename else ""
    print(f"    {ct:<25s}  {n:>10,}{arrow}")

# ─────────────────────────────────────────────────────────────────────────────
# Apply harmonization rename
# ─────────────────────────────────────────────────────────────────────────────
if CELLTYPE_RENAME:
    cells_to_rename = df_cells["Cell_Type"].isin(CELLTYPE_RENAME.keys())
    n_to_rename = int(cells_to_rename.sum())
    if n_to_rename > 0:
        print(f"\n  Applying harmonization for {DATASET}:")
        for src, tgt in CELLTYPE_RENAME.items():
            n_src = int((df_cells["Cell_Type"] == src).sum())
            if n_src > 0:
                print(f"    {src!r:<25s} → {tgt!r:<15s} ({n_src:>10,} cells)")
        df_cells["Cell_Type"] = df_cells["Cell_Type"].replace(CELLTYPE_RENAME)
        print(f"  ✓ {n_to_rename:,} cells harmonized")
    else:
        print(f"\n  ⊘ No source CT names matched CELLTYPE_RENAME — already harmonized")
else:
    print(f"\n  ⊘ CELLTYPE_RENAME empty for {DATASET} — no harmonization needed")

# ─────────────────────────────────────────────────────────────────────────────
# Refresh ACTIVE_CELL_TYPES + CELLTYPE_ORDER_PLOT from harmonized data
# ─────────────────────────────────────────────────────────────────────────────
_refresh_celltype_order()

print(f"\n  CELLTYPE_ORDER_PLOT (post-harmonization, {len(CELLTYPE_ORDER_PLOT)} CTs):")
print(f"    {CELLTYPE_ORDER_PLOT}")

# ─────────────────────────────────────────────────────────────────────────────
# Verify against LINEAGE_CONFIGS (informational, not gating)
# ─────────────────────────────────────────────────────────────────────────────
configured_cts = set(LINEAGE_CONFIGS.keys()) if "LINEAGE_CONFIGS" in dir() else set()
unmapped_cts = [ct for ct in ACTIVE_CELL_TYPES if ct not in configured_cts]

if unmapped_cts:
    print(f"\n  ⚠ {len(unmapped_cts)} CT(s) not in LINEAGE_CONFIGS:")
    for ct in unmapped_cts:
        n = int((df_cells["Cell_Type"] == ct).sum())
        print(f"      {ct!r:<25s} ({n:,} cells)  → §3/§4/§5/§7/§8 run, §5c/§5d skip")
    print(f"  To fix: add CELLTYPE_RENAME_BY_DATASET[{DATASET!r}][source_name]")
else:
    print(f"  ✓ All CTs in LINEAGE_CONFIGS — full §5c/§5d coverage")

# ─────────────────────────────────────────────────────────────────────────────
# Distribution summary
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n  df_cells shape : {df_cells.shape}")
print(f"  SnC rate overall: {df_cells['is_senescent'].mean()*100:.2f}%")

print(f"\n  SnC rate per Cell_Type (post-harmonization):")
print(f"  {'Cell_Type':<25} {'n_cells':>10} {'%SnC':>8}")
print(f"  {'─'*45}")
for ct in CELLTYPE_ORDER_PLOT:
    sub = df_cells[df_cells["Cell_Type"] == ct]
    n = len(sub)
    pct_snc = sub["is_senescent"].mean() * 100
    flag = "" if ct in configured_cts else "  ⚠ no panel"
    print(f"  {ct:<25} {n:>10,} {pct_snc:>7.2f}%{flag}")

print(f"\n✓ §1.7 df_cells complete  ({len(df_cells):,} rows × {df_cells.shape[1]} cols)")

---
## 08 · Donor × cell-type frame

**Why.** `df_donor_ct`: one row per donor per cell type, with `prop_snc` = the donor's senescent fraction within that type. This is the frame the donor-level models in sections 23-24 fit on.

In [ ]:
# =============================================================================
# §1.8 — BUILD df_donor_ct (donor × Cell_Type frame)
# =============================================================================
# Donor × Cell_Type pseudo-bulk for §3 (donor-level OLS) and §4 (RLR).
# Drops donor × CT groups with fewer than MIN_CELLS_PER_DONOR cells.
#
# Self-sufficient: re-applies CELLTYPE_RENAME against df_cells AND refreshes
# CELLTYPE_ORDER_PLOT from harmonized data before any operations.
# =============================================================================

print("=" * 80)
print(f"§1.8 — BUILD df_donor_ct")
print("=" * 80)

# ─────────────────────────────────────────────────────────────────────────────
# Defensive harmonization
# ─────────────────────────────────────────────────────────────────────────────
if "CELLTYPE_RENAME" in dir() and CELLTYPE_RENAME:
    n_to_rename = int(df_cells["Cell_Type"].isin(CELLTYPE_RENAME.keys()).sum())
    if n_to_rename > 0:
        print(f"  ⚠ Re-applying harmonization ({n_to_rename:,} cells)")
        df_cells["Cell_Type"] = df_cells["Cell_Type"].replace(CELLTYPE_RENAME)
        print(f"  ✓ harmonization restored")

# Refresh ACTIVE_CELL_TYPES / CELLTYPE_ORDER_PLOT from harmonized df_cells
_refresh_celltype_order()
print(f"  CELLTYPE_ORDER_PLOT: {CELLTYPE_ORDER_PLOT}")

# ─────────────────────────────────────────────────────────────────────────────
# Aggregate cell-level → donor × CT
# ─────────────────────────────────────────────────────────────────────────────
agg = (
    df_cells.groupby(["Donor", "Cell_Type"])
    .agg(
        n_cells=("is_senescent", "size"),
        n_snc=("is_senescent", "sum"),
        prop_snc=("is_senescent", "mean"),
        mean_sen_score=("sen_score", "mean"),
    )
    .reset_index()
)

print(f"\n  Pre-filter donor × CT groups : {len(agg):,}")
print(f"  n_cells distribution         : "
      f"min={agg['n_cells'].min()}  median={int(agg['n_cells'].median())}  "
      f"max={agg['n_cells'].max():,}")

# ─────────────────────────────────────────────────────────────────────────────
# Apply MIN_CELLS_PER_DONOR filter (donor × CT level)
# ─────────────────────────────────────────────────────────────────────────────
keep_mask = agg["n_cells"] >= MIN_CELLS_PER_DONOR
n_dropped = int((~keep_mask).sum())
print(f"\n  MIN_CELLS_PER_DONOR = {MIN_CELLS_PER_DONOR}")
print(f"  Groups passing filter        : {keep_mask.sum():,}  "
      f"(dropped {n_dropped:,})")

if n_dropped > 0:
    dropped_summary = (
        agg[~keep_mask].groupby("Cell_Type").size().sort_values(ascending=False)
    )
    print(f"\n  Dropped donor × CT groups by Cell_Type:")
    for ct, n in dropped_summary.items():
        print(f"    {ct:<25} {n:>4} group(s)")

df_donor_ct = agg[keep_mask].reset_index(drop=True)

# ─────────────────────────────────────────────────────────────────────────────
# Attach donor-level metadata
# ─────────────────────────────────────────────────────────────────────────────
donor_meta_cols = [c for c in ["Study_Group", "Age", "Sex", "Cohort"]
                   if c in df_cells.columns]
donor_meta_cols += [c for c in PATHOLOGY_PASSTHROUGH_COLS
                    if c in df_cells.columns]

donor_meta = (
    df_cells[["Donor"] + donor_meta_cols]
    .drop_duplicates(subset=["Donor"])
    .set_index("Donor")
)
df_donor_ct = df_donor_ct.merge(donor_meta, on="Donor", how="left")

# ─────────────────────────────────────────────────────────────────────────────
# Per-CT inventory (using harmonized order)
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n  Per-CT donor counts (passing MIN_CELLS_PER_DONOR):")
print(f"  {'Cell_Type':<25} {'donors':>6}  {'median cells':>14}  {'%SnC':>7}")
print(f"  {'─'*60}")
for ct in CELLTYPE_ORDER_PLOT:
    sub = df_donor_ct[df_donor_ct["Cell_Type"] == ct]
    n_donors = sub["Donor"].nunique()
    if len(sub) > 0:
        median_n = int(sub["n_cells"].median())
        mean_pct = sub["prop_snc"].mean() * 100
        print(f"  {ct:<25} {n_donors:>6}  {median_n:>14}  {mean_pct:>6.2f}%")
    else:
        print(f"  {ct:<25} {n_donors:>6}  {'—':>14}  {'—':>7}  ⚠ all donors below MIN_CELLS")

print(f"\n  df_donor_ct shape : {df_donor_ct.shape}")
print(f"\n✓ §1.8 df_donor_ct complete")

---
## 09 · Donor frame

**Why.** `df_donor`: one row per donor. Age, sex, cohort, group, and the donor-wide senescent fraction.

In [ ]:
# =============================================================================
# §1.9 — BUILD df_donor (one row per donor)
# =============================================================================
# Pooled across all CTs in df_cells. Used by §6 cross-model summary and
# any donor-as-statistical-unit ablation.
#
# Self-sufficient: re-applies CELLTYPE_RENAME and refreshes order before use.
# =============================================================================

print("=" * 80)
print(f"§1.9 — BUILD df_donor")
print("=" * 80)

# ─────────────────────────────────────────────────────────────────────────────
# Defensive harmonization
# ─────────────────────────────────────────────────────────────────────────────
if "CELLTYPE_RENAME" in dir() and CELLTYPE_RENAME:
    n_to_rename = int(df_cells["Cell_Type"].isin(CELLTYPE_RENAME.keys()).sum())
    if n_to_rename > 0:
        print(f"  ⚠ Re-applying harmonization ({n_to_rename:,} cells)")
        df_cells["Cell_Type"] = df_cells["Cell_Type"].replace(CELLTYPE_RENAME)

_refresh_celltype_order()

# ─────────────────────────────────────────────────────────────────────────────
# Aggregate cell-level → donor (pooled across CTs)
# ─────────────────────────────────────────────────────────────────────────────
donor_agg = (
    df_cells.groupby("Donor")
    .agg(
        n_cells_total=("is_senescent", "size"),
        n_snc_total=("is_senescent", "sum"),
        prop_snc_donor=("is_senescent", "mean"),
        mean_sen_score_donor=("sen_score", "mean"),
    )
    .reset_index()
)

# Build donor_meta (re-derive in case it wasn't built in §1.8 yet, or it was
# built before harmonization)
donor_meta_cols = [c for c in ["Study_Group", "Age", "Sex", "Cohort"]
                   if c in df_cells.columns]
donor_meta_cols += [c for c in PATHOLOGY_PASSTHROUGH_COLS
                    if c in df_cells.columns]

donor_meta = (
    df_cells[["Donor"] + donor_meta_cols]
    .drop_duplicates(subset=["Donor"])
    .set_index("Donor")
)

df_donor = donor_agg.merge(donor_meta, on="Donor", how="left")

# ─────────────────────────────────────────────────────────────────────────────
# Summary
# ─────────────────────────────────────────────────────────────────────────────
print(f"  df_donor shape   : {df_donor.shape}")
print(f"  Donors           : {df_donor['Donor'].nunique():,}")
print(f"  Total cells      : {df_donor['n_cells_total'].sum():,}")
print(f"  Mean cells/donor : {df_donor['n_cells_total'].mean():.0f}")
print(f"  Median cells/donor: {df_donor['n_cells_total'].median():.0f}")
print(f"  Mean prop_snc    : {df_donor['prop_snc_donor'].mean()*100:.2f}%")
print(f"  prop_snc range   : "
      f"{df_donor['prop_snc_donor'].min()*100:.2f}% to "
      f"{df_donor['prop_snc_donor'].max()*100:.2f}%")

print(f"\n  Donor counts by Study_Group:")
sg_counts = df_donor.groupby("Study_Group").size().sort_values(ascending=False)
for sg, n in sg_counts.items():
    print(f"    {sg:<25} {n:>4}")

# Pathology biomarker availability per donor
biomarker_cols = [c for c in ["tau", "amyloid", "cognition", "pmi"]
                  if c in df_donor.columns]
if biomarker_cols:
    print(f"\n  Donor-level biomarker availability:")
    for bm in biomarker_cols:
        n_with = int(df_donor[bm].notna().sum())
        print(f"    {bm:<15} {n_with:>3} / {len(df_donor)} donors with data")

if "apoe" in df_donor.columns:
    print(f"\n  APOE genotype distribution (donor-level):")
    apoe_counts = df_donor["apoe"].value_counts(dropna=False)
    for ge, n in apoe_counts.items():
        print(f"    {str(ge):<10} {n:>4}")

print(f"\n✓ §1.9 df_donor complete")

---
## 10 · Primary variables

**Why.** Decides which categorical variables the group-contrast sections iterate over.

**Worth noting:** if `REFERENCE_GROUP` is not among the observed levels, the variable is dropped with a printed `✗` rather than silently defaulting to the alphabetically-first level. That is the correct behaviour and it is why every contrast below has a stated reference.

In [ ]:
# =============================================================================
# §1.10 — BUILD PRIMARY_VARS_AVAILABLE
# =============================================================================
# Decides which categorical primary variables §3/§4/§5/§8.x will iterate over.
#
#   • Study_Group is ALWAYS included.
#   • pathology_group is included only when HAS_PATHOLOGY_GROUP = True
#     AND its reference / level order are valid.
# =============================================================================

print("=" * 80)
print(f"§1.10 — BUILD PRIMARY_VARS_AVAILABLE")
print("=" * 80)

# ─────────────────────────────────────────────────────────────────────────────
# Defensive harmonization
# ─────────────────────────────────────────────────────────────────────────────
if "CELLTYPE_RENAME" in dir() and CELLTYPE_RENAME:
    n_to_rename = int(df_cells["Cell_Type"].isin(CELLTYPE_RENAME.keys()).sum())
    if n_to_rename > 0:
        print(f"  ⚠ Re-applying harmonization ({n_to_rename:,} cells)")
        df_cells["Cell_Type"] = df_cells["Cell_Type"].replace(CELLTYPE_RENAME)

_refresh_celltype_order()

PRIMARY_VARS_AVAILABLE = []
primary_vars_meta = {}

# ─────────────────────────────────────────────────────────────────────────────
# Always: Study_Group
# ─────────────────────────────────────────────────────────────────────────────
sg_levels = df_cells["Study_Group"].dropna().unique().tolist()
if len(sg_levels) < 2:
    print(f"  ✗ Study_Group has only {len(sg_levels)} level — cannot use as primary")
elif REFERENCE_GROUP not in sg_levels:
    print(f"  ✗ REFERENCE_GROUP={REFERENCE_GROUP!r} not in Study_Group levels {sg_levels}")
else:
    PRIMARY_VARS_AVAILABLE.append("Study_Group")
    primary_vars_meta["Study_Group"] = {
        "levels":     sg_levels,
        "reference":  REFERENCE_GROUP,
        "n_levels":   len(sg_levels),
    }
    print(f"  ✓ Study_Group     reference={REFERENCE_GROUP!r}   "
          f"levels={sg_levels}")

# ─────────────────────────────────────────────────────────────────────────────
# Conditional: pathology_group
# ─────────────────────────────────────────────────────────────────────────────
if HAS_PATHOLOGY_GROUP:
    pg_levels = df_cells["pathology_group"].dropna().astype(str).unique().tolist()
    if len(pg_levels) < 2:
        print(f"  ⚠ pathology_group has {len(pg_levels)} level — skipping as primary")
    elif PATHOLOGY_GROUP_REFERENCE is None:
        print(f"  ⚠ PATHOLOGY_GROUP_REFERENCE not set — skipping pathology_group")
    elif PATHOLOGY_GROUP_REFERENCE not in pg_levels:
        print(f"  ✗ PATHOLOGY_GROUP_REFERENCE={PATHOLOGY_GROUP_REFERENCE!r} "
              f"not in data levels {pg_levels} — skipping")
    else:
        PRIMARY_VARS_AVAILABLE.append("pathology_group")
        primary_vars_meta["pathology_group"] = {
            "levels":     PATHOLOGY_GROUP_LEVEL_ORDER or sorted(pg_levels),
            "reference":  PATHOLOGY_GROUP_REFERENCE,
            "n_levels":   len(pg_levels),
        }
        print(f"  ✓ pathology_group reference={PATHOLOGY_GROUP_REFERENCE!r}   "
              f"levels={PATHOLOGY_GROUP_LEVEL_ORDER}")
else:
    print(f"  ⊘ pathology_group not assigned — primary will use Study_Group only")

# ─────────────────────────────────────────────────────────────────────────────
# Final
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n  PRIMARY_VARS_AVAILABLE = {PRIMARY_VARS_AVAILABLE}")
print(f"\n✓ §1.10 primary variable validation complete")

---
## 11 · Run plan

**Why.** Prints what is about to run — which specs, which cell types, which covariates — before anything is fit.

In [ ]:
# =============================================================================
# §1.11 — RUN-PLAN SUMMARY
# =============================================================================
# One-screen dashboard cross-checking all §1 decisions.
# =============================================================================

print("=" * 80)
print(f"§1.11 — RUN-PLAN SUMMARY  |  {DATASET}")
print("=" * 80)

# ─────────────────────────────────────────────────────────────────────────────
# Defensive harmonization
# ─────────────────────────────────────────────────────────────────────────────
if "CELLTYPE_RENAME" in dir() and CELLTYPE_RENAME:
    n_to_rename = int(df_cells["Cell_Type"].isin(CELLTYPE_RENAME.keys()).sum())
    if n_to_rename > 0:
        print(f"  ⚠ Re-applying harmonization ({n_to_rename:,} cells)")
        df_cells["Cell_Type"] = df_cells["Cell_Type"].replace(CELLTYPE_RENAME)

_refresh_celltype_order()

# ─────────────────────────────────────────────────────────────────────────────
# Dataset block
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n  DATASET")
print(f"  {'─' * 60}")
print(f"    tissue            : {TISSUE}")
print(f"    study_type        : {STUDY_TYPE}{('  ('+DISEASE+')') if IS_DISEASE else ''}")
print(f"    cohort key        : {DATASET}")
print(f"    cells             : {len(df_cells):>10,}")
print(f"    donors            : {df_cells['Donor'].nunique():>10,}")
print(f"    cell types        : {len(ACTIVE_CELL_TYPES):>10}")

# ─────────────────────────────────────────────────────────────────────────────
# Cell_Type harmonization summary
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n  CELL_TYPE HARMONIZATION  (per-dataset rename map)")
print(f"  {'─' * 60}")
if CELLTYPE_RENAME:
    for src, tgt in CELLTYPE_RENAME.items():
        print(f"    {src!r:<25s} → {tgt!r}")
else:
    print(f"    (none — {DATASET} CT names match LINEAGE_CONFIGS as-is)")

print(f"\n    Active CTs ({len(ACTIVE_CELL_TYPES)}): {ACTIVE_CELL_TYPES}")

configured_cts = set(LINEAGE_CONFIGS.keys()) if "LINEAGE_CONFIGS" in dir() else set()
unmapped = [ct for ct in ACTIVE_CELL_TYPES if ct not in configured_cts]
if unmapped:
    print(f"    ⚠ CTs not in LINEAGE_CONFIGS (skip §5c/§5d): {unmapped}")
else:
    print(f"    ✓ All CTs covered by LINEAGE_CONFIGS panels")

# ─────────────────────────────────────────────────────────────────────────────
# Canonical pathology columns
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n  CANONICAL COLUMNS IN OBS (post §1.5 rename)")
print(f"  {'─' * 60}")
canonical_state = [
    ("Study_Group",     True,                 "clinical diagnosis"),
    ("pathology_group", HAS_PATHOLOGY_GROUP,  "categorical staging"),
    ("tau",             HAS_TAU,              "continuous tau / NFT"),
    ("amyloid",         HAS_AMYLOID,          "continuous amyloid / Aβ"),
    ("cognition",       HAS_COGNITION,        "cognitive score"),
    ("apoe",            HAS_APOE,             "APOE genotype"),
    ("pmi",             HAS_PMI,              "postmortem interval"),
]
for canonical, flag, descr in canonical_state:
    mark = "✓" if flag else "⊘"
    print(f"    {mark} {canonical:18s} {descr}")

# ─────────────────────────────────────────────────────────────────────────────
# Primary variables
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n  PRIMARY VARIABLES (for §3 / §4 / §5 / §8.x)")
print(f"  {'─' * 60}")
for pv in PRIMARY_VARS_AVAILABLE:
    meta = primary_vars_meta[pv]
    print(f"    • {pv:18s}  ref={meta['reference']!r}  "
          f"{meta['n_levels']} level(s)")

# ─────────────────────────────────────────────────────────────────────────────
# Stratification variable for §7b/§7c
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n  STRATIFICATION VARIABLE (for §7b / §7c)")
print(f"  {'─' * 60}")
print(f"    STRATIFY_VAR       = {STRATIFY_VAR!r}")
print(f"    STRATIFY_REFERENCE = {STRATIFY_REFERENCE!r}")
if STRATIFY_LEVELS:
    print(f"    STRATIFY_LEVELS    = {STRATIFY_LEVELS}")
else:
    print(f"    STRATIFY_LEVELS    = (will finalize in §2.3)")

# ─────────────────────────────────────────────────────────────────────────────
# Section gating
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n  SECTION GATING")
print(f"  {'─' * 60}")
gating = [
    ("§3 donor-level OLS",         True,           ""),
    ("§4 donor-level RLR",         True,           ""),
    ("§5 cell-level GLMM (lme4)",  True,           ""),
    ("§5c Sloan validation",       True,           "covers CTs in LINEAGE_CONFIGS"),
    ("§5d cell cycle validation",  True,           "covers CTs in LINEAGE_CONFIGS"),
    ("§6 cross-model summary",     True,           ""),
    ("§7a pathology_index",        RUN_PATHOLOGY and HAS_BIOMARKERS,
        f"requires biomarkers (HAS_TAU={HAS_TAU}, HAS_AMYLOID={HAS_AMYLOID})"),
    ("§7b sen ~ pathology",        RUN_PATHOLOGY,  "requires §7a or biomarkers"),
    ("§7c interaction + mediation",RUN_PATHOLOGY,  "requires §7a/§7b"),
    ("§8.1 continuous LMM",        True,           ""),
    ("§8.2 threshold sensitivity", True,           ""),
    ("§8.3 alt scoring methods",   True,           ""),
    ("§8.4 negative control",      True,           ""),
]
for label, will_run, note in gating:
    mark = "✓" if will_run else "⊘"
    note_str = f"  ({note})" if note else ""
    status = "run" if will_run else "skip"
    print(f"    {mark} {label:<32s}  {status}{note_str}")

# ─────────────────────────────────────────────────────────────────────────────
# Covariate availability
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n  COVARIATE AVAILABILITY")
print(f"  {'─' * 60}")
covariate_state = [
    ("Age",                "Age" in df_cells.columns),
    ("Sex",                "Sex" in df_cells.columns),
    ("Cohort",             "Cohort" in df_cells.columns),
    ("log10_total_counts", "log10_total_counts" in df_cells.columns),
]
for cov, has_it in covariate_state:
    mark = "✓" if has_it else "⊘"
    print(f"    {mark} {cov}")

# ─────────────────────────────────────────────────────────────────────────────
# Output paths
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n  OUTPUT PATHS")
print(f"  {'─' * 60}")
for k in ["data", "results", "figures"]:
    print(f"    {k:<10s} → {PATHS[k]}")

# ─────────────────────────────────────────────────────────────────────────────
# Edit points
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n  EDIT POINTS")
print(f"  {'─' * 60}")
print(f"    §0     RUN_PATHOLOGY_SECTIONS  (auto/on/off)")
print(f"    §0     FOCUS_CELL_TYPES         (subset, or None)")
print(f"    §0     MIN_CELLS_PER_DONOR      = {MIN_CELLS_PER_DONOR}")
print(f"    §1.4b  PATHOLOGY_META")
print(f"    §1.4b  PATHOLOGY_GROUP_REFERENCE / PATHOLOGY_GROUP_LEVEL_ORDER")
print(f"    §1.7   CELLTYPE_RENAME_BY_DATASET[{DATASET!r}]")

print(f"\n{'=' * 80}")
print(f"✓ §1.11 RUN PLAN COMPLETE — proceed to §2 (data preparation)")
print(f"{'=' * 80}")

---
## 12 · Data preparation

**Why.** Scales age and depth, builds `df_cells_model`. `SCALE_UMI = True` z-scores `log10_total_counts` into `log10_total_counts_scaled`.

In [ ]:
# =============================================================================
# §2 — DATA PREPARATION
# =============================================================================
# Build the modeling-ready frame `df_cells_model`. Same shape as df_cells but
# with:
#   - Age_scaled (z-scored Age, mean 0 / sd 1)
#   - log10_total_counts_scaled (z-scored)
#   - rows with NA in any modeling-relevant column dropped
#
# This is the frame §3/§4/§5/§8.x consume directly (after their own per-spec
# subset). Keeping a separate "_model" frame from raw df_cells means the raw
# data stays available for §7 pathology work which needs un-scaled biomarkers.
# =============================================================================

# §2.1 Setup + scaling
# §2.2 Build df_cells_model

print("=" * 80)
print(f"§2.1 — SETUP + SCALING  |  {DATASET}")
print("=" * 80)

# ─────────────────────────────────────────────────────────────────────────────
# Defensive harmonization (every cell does this)
# ─────────────────────────────────────────────────────────────────────────────
if "CELLTYPE_RENAME" in dir() and CELLTYPE_RENAME:
    n_to_rename = int(df_cells["Cell_Type"].isin(CELLTYPE_RENAME.keys()).sum())
    if n_to_rename > 0:
        print(f"  ⚠ Re-applying harmonization ({n_to_rename:,} cells)")
        df_cells["Cell_Type"] = df_cells["Cell_Type"].replace(CELLTYPE_RENAME)
_refresh_celltype_order()

# ─────────────────────────────────────────────────────────────────────────────
# Modeling configuration (lives here, not §0 — it's data-prep-specific)
# ─────────────────────────────────────────────────────────────────────────────
SCALE_AGE              = True       # z-score Age
SCALE_UMI              = True       # z-score log10_total_counts
DROP_NA_IN_MODEL_FRAME = True       # drop rows with NA in modeling-relevant cols

print(f"\n  Scaling configuration:")
print(f"    SCALE_AGE              = {SCALE_AGE}  (z-score Age → Age_scaled)")
print(f"    SCALE_UMI              = {SCALE_UMI}  (z-score log10_total_counts → ..._scaled)")
print(f"    DROP_NA_IN_MODEL_FRAME = {DROP_NA_IN_MODEL_FRAME}")

# ─────────────────────────────────────────────────────────────────────────────
# Build df_cells_model
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n  Building df_cells_model from df_cells...")
df_cells_model = df_cells.copy()
print(f"    starting shape: {df_cells_model.shape}")

# §2.1.1 Scale Age
if SCALE_AGE and "Age" in df_cells_model.columns:
    age_vals = df_cells_model["Age"].dropna().values
    if len(age_vals) > 0:
        age_mean = float(np.mean(age_vals))
        age_std  = float(np.std(age_vals, ddof=1))
        if age_std == 0:
            print(f"    ⚠ Age has zero std — skipping scaling")
            df_cells_model["Age_scaled"] = df_cells_model["Age"]
        else:
            df_cells_model["Age_scaled"] = (
                (df_cells_model["Age"] - age_mean) / age_std
            )
            print(f"    Age:               μ={age_mean:.2f}  σ={age_std:.2f}  → Age_scaled")
    else:
        print(f"    ⚠ Age has no valid values — skipping")

# §2.1.2 Scale log10_total_counts
if SCALE_UMI and "log10_total_counts" in df_cells_model.columns:
    umi_vals = df_cells_model["log10_total_counts"].dropna().values
    if len(umi_vals) > 0:
        umi_mean = float(np.mean(umi_vals))
        umi_std  = float(np.std(umi_vals, ddof=1))
        if umi_std == 0:
            print(f"    ⚠ log10_total_counts has zero std — skipping")
            df_cells_model["log10_total_counts_scaled"] = df_cells_model["log10_total_counts"]
        else:
            df_cells_model["log10_total_counts_scaled"] = (
                (df_cells_model["log10_total_counts"] - umi_mean) / umi_std
            )
            print(f"    log10_total_counts: μ={umi_mean:.2f}  σ={umi_std:.2f}  → log10_total_counts_scaled")

# §2.1.3 Decide which columns are essential for modeling (NA-drop scope)
print("=" * 80)
print(f"§2.2 — DF_CELLS_MODEL (drop NA + final inventory)")
print("=" * 80)

essential_cols = ["Donor", "Cell_Type", "is_senescent", "sen_score", "Study_Group"]
if "Age_scaled" in df_cells_model.columns:
    essential_cols.append("Age_scaled")
if "Sex" in df_cells_model.columns:
    essential_cols.append("Sex")
if "log10_total_counts_scaled" in df_cells_model.columns:
    essential_cols.append("log10_total_counts_scaled")

# Pathology columns — only require non-NA when actually used as primary
# (i.e., for pathology_group → require non-NA only on rows where pathology_group
# is a primary). Don't require tau/amyloid non-NA at this stage; §7 handles that.
if HAS_PATHOLOGY_GROUP and "pathology_group" in df_cells_model.columns:
    essential_cols.append("pathology_group")

print(f"  Essential cols (require non-NA): {essential_cols}")

# §2.1.4 Drop NA
if DROP_NA_IN_MODEL_FRAME:
    n_before = len(df_cells_model)
    na_per_col = df_cells_model[essential_cols].isna().sum()
    cols_with_na = na_per_col[na_per_col > 0]
    if len(cols_with_na) > 0:
        print(f"\n  NA counts in essential columns:")
        for col, n in cols_with_na.items():
            print(f"    {col:<30s} {n:>8,}  ({n/n_before*100:.2f}%)")
    df_cells_model = df_cells_model.dropna(subset=essential_cols).reset_index(drop=True)
    n_after = len(df_cells_model)
    print(f"\n  After NA-drop: {n_before:,} → {n_after:,}  (dropped {n_before - n_after:,})")

# §2.1.5 Coerce final dtypes
df_cells_model["Donor"]        = df_cells_model["Donor"].astype(str)
df_cells_model["Cell_Type"]    = df_cells_model["Cell_Type"].astype(str)
df_cells_model["Study_Group"]  = df_cells_model["Study_Group"].astype(str)
df_cells_model["is_senescent"] = df_cells_model["is_senescent"].astype(int)
df_cells_model["sen_score"]    = df_cells_model["sen_score"].astype(float)

if "Age_scaled" in df_cells_model.columns:
    df_cells_model["Age_scaled"] = df_cells_model["Age_scaled"].astype(float)
if "log10_total_counts_scaled" in df_cells_model.columns:
    df_cells_model["log10_total_counts_scaled"] = (
        df_cells_model["log10_total_counts_scaled"].astype(float)
    )
if "Sex" in df_cells_model.columns:
    df_cells_model["Sex"] = df_cells_model["Sex"].astype(str)
if "Cohort" in df_cells_model.columns:
    df_cells_model["Cohort"] = df_cells_model["Cohort"].astype(str)
if "pathology_group" in df_cells_model.columns:
    df_cells_model["pathology_group"] = df_cells_model["pathology_group"].astype(str)

# Refresh CT order from the (potentially smaller) df_cells_model
present_cts_model = df_cells_model["Cell_Type"].value_counts().index.tolist()
print(f"\n  CT inventory in df_cells_model:")
print(f"  {'Cell_Type':<25} {'n_cells':>10} {'%SnC':>8}")
print(f"  {'─'*45}")
for ct in CELLTYPE_ORDER_PLOT:
    if ct not in present_cts_model:
        print(f"  {ct:<25} {'0':>10} {'—':>8}  ⚠ no rows after filter")
        continue
    sub = df_cells_model[df_cells_model["Cell_Type"] == ct]
    n = len(sub)
    pct = sub["is_senescent"].mean() * 100
    print(f"  {ct:<25} {n:>10,} {pct:>7.2f}%")

print(f"\n  df_cells_model shape: {df_cells_model.shape}")
print(f"  donors: {df_cells_model['Donor'].nunique():,}")

print(f"\n✓ §2.1+§2.2 df_cells_model complete")

---
## 13 · Categorical orderings

**Why.** Fixes the level order for `Study_Group` and `pathology_group` so every forest plot and every stacked bar reads in the same direction.

In [ ]:
# =============================================================================
# §2.3 — CATEGORICAL ORDERINGS (Study_Group + pathology_group)
# §2.4 — LEVEL VALIDATION
# §2.5 — COLOR DISPATCHERS
# =============================================================================
# Sets ordered Categoricals on df_cells_model so §3/§4/§5 contrasts run
# in the right reference direction. Reads from §1.4b configuration —
# no hardcoded pathology level lists.
#
# Then validates each primary variable's levels are populated (no zero-count
# levels) and finalizes STRATIFY_LEVELS for the Study_Group fallback case.
#
# Finally: builds COLOR_DISPATCH dict mapping each primary variable name to
# its color palette. §3/§4/§5 plot helpers read from this — no inline
# `if primary == "pathology_group"` branches.
# =============================================================================

print("=" * 80)
print(f"§2.3 — CATEGORICAL ORDERINGS")
print("=" * 80)

# ─────────────────────────────────────────────────────────────────────────────
# Defensive harmonization
# ─────────────────────────────────────────────────────────────────────────────
if "CELLTYPE_RENAME" in dir() and CELLTYPE_RENAME:
    n_to_rename = int(df_cells_model["Cell_Type"].isin(CELLTYPE_RENAME.keys()).sum())
    if n_to_rename > 0:
        print(f"  ⚠ Re-applying harmonization to df_cells_model ({n_to_rename:,} cells)")
        df_cells_model["Cell_Type"] = df_cells_model["Cell_Type"].replace(CELLTYPE_RENAME)

# ─────────────────────────────────────────────────────────────────────────────
# §2.3.1 Study_Group ordering
# REFERENCE_GROUP first, others alphabetical
# ─────────────────────────────────────────────────────────────────────────────
sg_levels_in_data = df_cells_model["Study_Group"].dropna().unique().tolist()
if REFERENCE_GROUP in sg_levels_in_data:
    others = sorted([lv for lv in sg_levels_in_data if lv != REFERENCE_GROUP])
    study_group_order = [REFERENCE_GROUP] + others
else:
    study_group_order = sorted(sg_levels_in_data)
    print(f"  ⚠ REFERENCE_GROUP={REFERENCE_GROUP!r} not in Study_Group levels — "
          f"using alphabetical")

df_cells_model["Study_Group"] = pd.Categorical(
    df_cells_model["Study_Group"].astype(str),
    categories=study_group_order,
    ordered=True,
)
print(f"\n  Study_Group ordering: {study_group_order}")

# ─────────────────────────────────────────────────────────────────────────────
# §2.3.2 pathology_group ordering (only if HAS_PATHOLOGY_GROUP)
# Reads PATHOLOGY_GROUP_LEVEL_ORDER from §1.4b — no hardcoded lists
# ─────────────────────────────────────────────────────────────────────────────
if HAS_PATHOLOGY_GROUP and "pathology_group" in df_cells_model.columns:
    pg_levels_in_data = (
        df_cells_model["pathology_group"].dropna().astype(str).unique().tolist()
    )

    declared_order = [lv for lv in PATHOLOGY_GROUP_LEVEL_ORDER
                      if lv in pg_levels_in_data]
    extras = sorted([lv for lv in pg_levels_in_data
                     if lv not in PATHOLOGY_GROUP_LEVEL_ORDER])
    pg_order_final = declared_order + extras

    if PATHOLOGY_GROUP_REFERENCE not in pg_order_final:
        print(f"  ⚠ PATHOLOGY_GROUP_REFERENCE={PATHOLOGY_GROUP_REFERENCE!r} "
              f"not in data — pathology_group spec will be skipped in §2.6")
    else:
        # Move reference to front
        pg_order_final = [PATHOLOGY_GROUP_REFERENCE] + [
            lv for lv in pg_order_final if lv != PATHOLOGY_GROUP_REFERENCE
        ]

    df_cells_model["pathology_group"] = pd.Categorical(
        df_cells_model["pathology_group"].astype(str),
        categories=pg_order_final,
        ordered=True,
    )
    print(f"  pathology_group ordering: {pg_order_final}")
else:
    print(f"  ⊘ pathology_group not assigned — skip ordering")

# ─────────────────────────────────────────────────────────────────────────────
# §2.3.3 Finalize STRATIFY_LEVELS / STRATIFY_REFERENCE for fallback case
# When pathology_group exists: already set in §1.5 from PATHOLOGY_GROUP_LEVEL_ORDER
# When fallback to Study_Group: set here once study_group_order is finalized
# ─────────────────────────────────────────────────────────────────────────────
if STRATIFY_VAR == "Study_Group":
    STRATIFY_LEVELS    = list(study_group_order)
    STRATIFY_REFERENCE = REFERENCE_GROUP
    print(f"\n  STRATIFY_VAR fallback finalized:")
    print(f"    STRATIFY_LEVELS    = {STRATIFY_LEVELS}")
    print(f"    STRATIFY_REFERENCE = {STRATIFY_REFERENCE!r}")
else:
    print(f"\n  STRATIFY_VAR={STRATIFY_VAR!r} (already set in §1.5)")

# ─────────────────────────────────────────────────────────────────────────────
# §2.4 — LEVEL VALIDATION
# Ensure each primary's levels are all populated (no zero-count levels)
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print(f"§2.4 — LEVEL VALIDATION")
print("=" * 80)

problems = []

print(f"\n  Study_Group level counts:")
sg_counts = df_cells_model["Study_Group"].value_counts().reindex(study_group_order)
for lv, n in sg_counts.items():
    n_donors = (
        df_cells_model[df_cells_model["Study_Group"] == lv]["Donor"]
        .nunique()
    )
    flag = "" if n > 0 else "  ⚠ ZERO CELLS"
    flag_donor = "" if n_donors >= 2 else "  ⚠ <2 DONORS"
    print(f"    {lv:<20s} {n:>10,} cells  ({n_donors} donors){flag}{flag_donor}")
    if n == 0:
        problems.append(f"Study_Group level {lv!r} has zero cells")

if HAS_PATHOLOGY_GROUP and "pathology_group" in df_cells_model.columns:
    print(f"\n  pathology_group level counts:")
    pg_counts = df_cells_model["pathology_group"].value_counts()
    pg_order_now = list(df_cells_model["pathology_group"].cat.categories)
    for lv in pg_order_now:
        n = int(pg_counts.get(lv, 0))
        n_donors = (
            df_cells_model[df_cells_model["pathology_group"] == lv]["Donor"]
            .nunique()
        )
        flag = "" if n > 0 else "  ⚠ ZERO CELLS"
        flag_donor = "" if n_donors >= 2 else "  ⚠ <2 DONORS"
        print(f"    {lv:<20s} {n:>10,} cells  ({n_donors} donors){flag}{flag_donor}")
        if n == 0:
            problems.append(f"pathology_group level {lv!r} has zero cells")

if problems:
    print(f"\n  ⚠ {len(problems)} validation problem(s):")
    for p in problems:
        print(f"    - {p}")
else:
    print(f"\n  ✓ All primary variable levels populated")

# ─────────────────────────────────────────────────────────────────────────────
# §2.5 — COLOR DISPATCHERS
# COLOR_DISPATCH[primary_name] returns a dict mapping each level → hex color.
# §3/§4/§5 plot helpers read from this — no inline if-branches per primary.
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "=" * 80)
print(f"§2.5 — COLOR DISPATCHERS")
print("=" * 80)

# Default palettes — fallback if §0 didn't define them
_DEFAULT_STUDY_GROUP_COLORS = {
    "Control": "#4F8EC4",
    "AD":      "#D85A30",
    "MCI":     "#EF9F27",
    "Other":   "#A6AAB5",
}
_DEFAULT_PATHOLOGY_GROUP_COLORS = {
    "no-pathology":    "#7BB37D",   # green
    "early-pathology": "#EFC93C",   # yellow
    "late-pathology":  "#D85A30",   # red
}

# Use §0-defined palettes if present, otherwise fall back
SG_COLORS_USED = (
    STUDY_GROUP_COLORS if "STUDY_GROUP_COLORS" in dir()
    else _DEFAULT_STUDY_GROUP_COLORS
)
PG_COLORS_USED = (
    PATHOLOGY_GROUP_COLORS if "PATHOLOGY_GROUP_COLORS" in dir()
    else _DEFAULT_PATHOLOGY_GROUP_COLORS
)

# Build per-primary color dict (level → hex)
COLOR_DISPATCH = {}

# Study_Group dispatch
sg_dispatch = {}
for lv in study_group_order:
    sg_dispatch[lv] = SG_COLORS_USED.get(lv, "#888888")
COLOR_DISPATCH["Study_Group"] = sg_dispatch

# pathology_group dispatch (when present)
if HAS_PATHOLOGY_GROUP and "pathology_group" in df_cells_model.columns:
    pg_order_now = list(df_cells_model["pathology_group"].cat.categories)
    pg_dispatch = {}
    for lv in pg_order_now:
        pg_dispatch[lv] = PG_COLORS_USED.get(lv, "#888888")
    COLOR_DISPATCH["pathology_group"] = pg_dispatch

print(f"\n  COLOR_DISPATCH built for {len(COLOR_DISPATCH)} primary variable(s):")
for primary, palette in COLOR_DISPATCH.items():
    print(f"    {primary}:")
    for lv, color in palette.items():
        print(f"      {lv:<25s} {color}")

print(f"\n✓ §2.3+§2.4+§2.5 ordering, validation, color dispatch complete")

---
## 14 · Primary specs

**Why.** Assembles `PRIMARY_SPECS` — for each primary variable: its reference, its level order, its donor-level covariates and its cell-level covariates.

**Two things to be aware of, both carried through as they ran:**

1. `covariates_donor_default` appends `Age` unconditionally. In *aging* mode the primary variable is the age bin, so sections 23-24 fit the age bin and continuous age in the same model. The bin coefficients are therefore what remains after `Age` has taken the variance, and they come out near-null. In *disease* mode this is correct — age is a genuine covariate there.
2. `covariates_cell_default` appends `log10_total_counts_scaled`, so the cell-level GLMM in section 26 adjusts for depth while the per-decade GLMMs in sections 30-32 do not.

In [ ]:
# =============================================================================
# §2.6 — BUILD PRIMARY_SPECS
# §2.7 — SANITY DASHBOARD
# =============================================================================
# PRIMARY_SPECS is the canonical list §3/§4/§5/§8.x iterate over. Each spec:
#   {
#     "name":             primary variable name (e.g. "Study_Group")
#     "slug":             filename-safe slug
#     "label":            display label
#     "type":             "categorical" | "continuous"
#     "var_cell":         column name in df_cells_model
#     "var_donor":        column name in df_donor_ct (usually same)
#     "reference":        baseline level (categorical) or None (continuous)
#     "level_order":      ordered list of non-reference levels for vs-ref + adjacent contrasts
#                         (excludes reference; the reference is implicit)
#     "covariates_cell":  list of cell-level covariate column names
#     "covariates_donor": list of donor-level covariate column names
#     "color_dispatch":   dict {level: hex} from §2.5 (categorical only)
#   }
#
# Reads:
#   - PRIMARY_VARS_AVAILABLE from §1.10
#   - primary_vars_meta from §1.10
#   - COLOR_DISPATCH from §2.5
#   - PATHOLOGY_GROUP_LEVEL_ORDER / PATHOLOGY_GROUP_REFERENCE from §1.4b
# =============================================================================

print("=" * 80)
print(f"§2.6 — BUILD PRIMARY_SPECS")
print("=" * 80)

# ─────────────────────────────────────────────────────────────────────────────
# Defensive harmonization
# ─────────────────────────────────────────────────────────────────────────────
if "CELLTYPE_RENAME" in dir() and CELLTYPE_RENAME:
    n_to_rename = int(df_cells_model["Cell_Type"].isin(CELLTYPE_RENAME.keys()).sum())
    if n_to_rename > 0:
        print(f"  ⚠ Re-applying harmonization to df_cells_model ({n_to_rename:,} cells)")
        df_cells_model["Cell_Type"] = df_cells_model["Cell_Type"].replace(CELLTYPE_RENAME)

# ─────────────────────────────────────────────────────────────────────────────
# Choose covariate column names based on what's available
# (cell-level uses scaled versions; donor-level uses unscaled — donor-level
# regressions z-score internally if needed)
# ─────────────────────────────────────────────────────────────────────────────
covariates_cell_default = []
if "Age_scaled" in df_cells_model.columns:
    covariates_cell_default.append("Age_scaled")
if "Sex" in df_cells_model.columns:
    covariates_cell_default.append("Sex")
if "Cohort" in df_cells_model.columns:
    covariates_cell_default.append("Cohort")
if "log10_total_counts_scaled" in df_cells_model.columns:
    covariates_cell_default.append("log10_total_counts_scaled")

covariates_donor_default = []
if "Age" in df_donor_ct.columns:
    covariates_donor_default.append("Age")
if "Sex" in df_donor_ct.columns:
    covariates_donor_default.append("Sex")
if "Cohort" in df_donor_ct.columns:
    covariates_donor_default.append("Cohort")

print(f"\n  Default covariates:")
print(f"    cell-level  : {covariates_cell_default}")
print(f"    donor-level : {covariates_donor_default}")

# ─────────────────────────────────────────────────────────────────────────────
# Build specs from PRIMARY_VARS_AVAILABLE — no hardcoded list
# ─────────────────────────────────────────────────────────────────────────────
PRIMARY_SPECS = []

for primary in PRIMARY_VARS_AVAILABLE:
    meta = primary_vars_meta[primary]
    reference = meta["reference"]
    levels_all = meta["levels"]

    # level_order = non-reference levels, in the order specified by §1.10
    # (which itself reads from PATHOLOGY_GROUP_LEVEL_ORDER for pathology_group,
    # and study_group_order for Study_Group)
    if primary == "Study_Group":
        # use study_group_order from §2.3
        ordered_levels = list(study_group_order)
    elif primary == "pathology_group":
        # use the categorical order set in §2.3
        ordered_levels = list(df_cells_model["pathology_group"].cat.categories)
    else:
        ordered_levels = list(levels_all)

    non_ref_levels = [lv for lv in ordered_levels if lv != reference]

    spec = {
        "name":             primary,
        "slug":             primary.lower().replace(" ", "_"),
        "label":            primary.replace("_", " "),
        "type":             "categorical",
        "var_cell":         primary,
        "var_donor":        primary,
        "reference":        reference,
        "level_order":      non_ref_levels,
        "covariates_cell":  list(covariates_cell_default),
        "covariates_donor": list(covariates_donor_default),
        "color_dispatch":   COLOR_DISPATCH.get(primary, {}),
    }

    PRIMARY_SPECS.append(spec)

# ─────────────────────────────────────────────────────────────────────────────
# Print the specs
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n  Built {len(PRIMARY_SPECS)} spec(s):\n")
for i, spec in enumerate(PRIMARY_SPECS, 1):
    print(f"  [{i}] {spec['label']}")
    print(f"      name             : {spec['name']!r}")
    print(f"      slug             : {spec['slug']!r}")
    print(f"      type             : {spec['type']}")
    print(f"      reference        : {spec['reference']!r}")
    print(f"      level_order      : {spec['level_order']}  (non-reference levels)")
    print(f"      covariates_cell  : {spec['covariates_cell']}")
    print(f"      covariates_donor : {spec['covariates_donor']}")
    print(f"      color_dispatch   : {spec['color_dispatch']}")
    print()

# ─────────────────────────────────────────────────────────────────────────────
# §2.7 — SANITY DASHBOARD
# Quick checklist before §3 starts fitting models.
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 80)
print(f"§2.7 — SANITY DASHBOARD")
print("=" * 80)

print(f"\n  FRAMES")
print(f"  {'─'*60}")
print(f"    df_cells          : {df_cells.shape}              (raw cell-level)")
print(f"    df_cells_model    : {df_cells_model.shape}        (modeling-ready, scaled, NA-dropped)")
print(f"    df_donor_ct       : {df_donor_ct.shape}           (donor × CT pseudobulk)")
print(f"    df_donor          : {df_donor.shape}              (one row per donor)")

print(f"\n  PRIMARY SPECS")
print(f"  {'─'*60}")
for spec in PRIMARY_SPECS:
    print(f"    {spec['name']:<20} ref={spec['reference']!r:<15} "
          f"contrasts={spec['level_order']}")

print(f"\n  CELL TYPES (in df_cells_model)")
print(f"  {'─'*60}")
configured_cts = set(LINEAGE_CONFIGS.keys()) if "LINEAGE_CONFIGS" in dir() else set()
for ct in CELLTYPE_ORDER_PLOT:
    sub = df_cells_model[df_cells_model["Cell_Type"] == ct]
    if len(sub) == 0:
        continue
    n_cells = len(sub)
    n_donors = sub["Donor"].nunique()
    pct_snc = sub["is_senescent"].mean() * 100
    panel_flag = "✓" if ct in configured_cts else "⊘"
    print(f"    {panel_flag} {ct:<22} {n_cells:>10,} cells  "
          f"{n_donors:>3} donors  {pct_snc:>6.2f}% SnC")

print(f"\n  COVARIATES")
print(f"  {'─'*60}")
print(f"    cell-level  (scaled): {covariates_cell_default}")
print(f"    donor-level (raw)  : {covariates_donor_default}")

print(f"\n  STRATIFICATION (for §7b/§7c)")
print(f"  {'─'*60}")
print(f"    STRATIFY_VAR       = {STRATIFY_VAR!r}")
print(f"    STRATIFY_REFERENCE = {STRATIFY_REFERENCE!r}")
print(f"    STRATIFY_LEVELS    = {STRATIFY_LEVELS}")

print(f"\n  PATHOLOGY (for §7)")
print(f"  {'─'*60}")
print(f"    HAS_PATHOLOGY_GROUP = {HAS_PATHOLOGY_GROUP}")
print(f"    HAS_TAU             = {HAS_TAU}")
print(f"    HAS_AMYLOID         = {HAS_AMYLOID}")
print(f"    HAS_BIOMARKERS      = {HAS_BIOMARKERS}")
print(f"    RUN_PATHOLOGY       = {RUN_PATHOLOGY}")

# ─────────────────────────────────────────────────────────────────────────────
# Final OK
# ─────────────────────────────────────────────────────────────────────────────
ready_to_fit = (
    len(PRIMARY_SPECS) > 0
    and len(df_cells_model) > 0
    and len(df_donor_ct) > 0
)

print(f"\n{'='*80}")
if ready_to_fit:
    print(f"✓ §2 DATA PREP COMPLETE — ready for §3 (donor-level OLS)")
else:
    print(f"✗ §2 NOT READY — check problems above")
print(f"{'='*80}")

---
## 15 · Burden build and composition

**Why.** `Burden_ct(donor) = n_snc_ct / n_total_cells(donor)`, with the denominator being *all* of the donor's cells including dropped types — so the burdens are comparable across donors regardless of which types were excluded.

`RENORMALIZE_TO_KEPT = False` keeps it a true burden. Setting it True turns it into a composition among kept types, which is a different quantity.

**Produces.** `CellProp` (abundance), `SenFrac` (susceptibility), `SenBurden` (their product). `SenFrac` is NaN below `BURDEN_MIN_CELLS = 10` rather than being a ratio of small integers.

In [ ]:
# =============================================================================
# §9.1 — SENESCENCE BURDEN (abundance-weighted) · BUILD + COMPOSITION
# =============================================================================
# Burden_ct(donor) = n_senescent_ct / n_total_cells(donor)  [ ≡ CellProp × SenFrac ]
# Denominator = ALL of the donor's cells (true burden), even for dropped types.
# Grouping read from PRIMARY_SPECS. Stages: 1 DATA · 2 PREP · 3 STATS · 4 RESULTS · 5 VIZ
# =============================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_columns", None); pd.set_option("display.width", 200)
print("=" * 80); print("§9.1 — BURDEN: BUILD + COMPOSITION"); print("=" * 80)

# ── config ──
BURDEN_SPEC_NAME = None
BURDEN_MIN_CELLS = 10
DROP_CELL_TYPES  = ["PVM", "VLMC", "VSMC", "Adaptive", "Endothelial", "Pericyte"]  # edit
RENORMALIZE_TO_KEPT = False   # False = burden (% of all cells); True = composition among kept types

_cat = [s for s in PRIMARY_SPECS if s["type"] == "categorical"]
GROUP_SPEC = (next((s for s in _cat if s["name"] == BURDEN_SPEC_NAME), None)
              if BURDEN_SPEC_NAME else (_cat[0] if _cat else None))
assert GROUP_SPEC is not None, "no categorical primary spec available for burden grouping"
GROUP_VAR   = GROUP_SPEC["name"]
GROUP_ORDER = [GROUP_SPEC["reference"]] + list(GROUP_SPEC["level_order"])
GROUP_FROM, GROUP_TO = GROUP_ORDER[0], GROUP_ORDER[-1]

# ── 1. DATA ──────────────────────────────────────────────────────────────────
print("\n" + "─"*70 + "\n1. DATA — df_cells (cell-level input)\n" + "─"*70)
print(f"  shape: {df_cells.shape} | donors: {df_cells['Donor'].nunique()} | cell types: {df_cells['Cell_Type'].nunique()}")
print(f"  grouping: {GROUP_VAR}  ({GROUP_FROM} → … → {GROUP_TO}, {len(GROUP_ORDER)} levels)")
print(df_cells[["Donor","Cell_Type","is_senescent","sen_score",GROUP_VAR]].head().to_string(index=False))

# ── 2. PREP ──────────────────────────────────────────────────────────────────
print("\n" + "─"*70 + "\n2. PREP — df_burden (donor × cell type)\n" + "─"*70)
active_cts = [c for c in CELLTYPE_ORDER_PLOT
              if c in df_cells["Cell_Type"].unique() and c not in DROP_CELL_TYPES]
print(f"  kept cell types ({len(active_cts)}): {active_cts}")
if DROP_CELL_TYPES:
    print(f"  dropped: {[c for c in DROP_CELL_TYPES if c in df_cells['Cell_Type'].unique()]}")

# denominator: all cells (burden) OR kept-only (composition)
if RENORMALIZE_TO_KEPT:
    donor_tot = (df_cells[df_cells["Cell_Type"].isin(active_cts)]
                 .groupby("Donor").size().rename("donor_total"))
    print("  denominator: KEPT cell types only (renormalized composition)")
else:
    donor_tot = df_cells.groupby("Donor").size().rename("donor_total")
    print("  denominator: ALL cells (true burden)")

g = (df_cells[df_cells["Cell_Type"].isin(active_cts)]
     .groupby(["Donor","Cell_Type"])
     .agg(n_cells=("is_senescent","size"), n_snc=("is_senescent","sum")).reset_index())
full_idx = pd.MultiIndex.from_product([donor_tot.index, active_cts], names=["Donor","Cell_Type"])
df_burden = g.set_index(["Donor","Cell_Type"]).reindex(full_idx, fill_value=0).reset_index()
df_burden = df_burden.merge(donor_tot, on="Donor", how="left")
df_burden["CellProp"]  = df_burden["n_cells"] / df_burden["donor_total"]
df_burden["SenBurden"] = df_burden["n_snc"]   / df_burden["donor_total"]
df_burden["SenFrac"]   = np.where(df_burden["n_cells"] >= BURDEN_MIN_CELLS,
                                  df_burden["n_snc"] / df_burden["n_cells"].replace(0, np.nan), np.nan)
donor_meta = (df_cells[["Donor", GROUP_VAR] +
                       [c for c in GROUP_SPEC.get("covariates_donor", []) if c in df_cells.columns]]
              .drop_duplicates("Donor"))
df_burden = df_burden.merge(donor_meta, on="Donor", how="left")
df_burden = df_burden[df_burden[GROUP_VAR].isin(GROUP_ORDER)].copy()
df_burden[GROUP_VAR]   = pd.Categorical(df_burden[GROUP_VAR], categories=GROUP_ORDER, ordered=True)
df_burden["Cell_Type"] = pd.Categorical(df_burden["Cell_Type"], categories=active_cts, ordered=True)
print(f"\n  shape: {df_burden.shape} | donors × CT = {df_burden['Donor'].nunique()} × {len(active_cts)}")
print(df_burden.head().to_string(index=False))

# ── 3. STATS (definitions / integrity) ────────────────────────────────────────
print("\n" + "─"*70 + "\n3. STATS — definitions & integrity check\n" + "─"*70)
print("  SenBurden_ct(donor) = n_snc_ct / donor_total   (≡ CellProp_ct × SenFrac_ct)")
exp_sum = "1.000" if RENORMALIZE_TO_KEPT else "≤1.000 (dropped types' share excluded)"
print(f"  Σ_ct CellProp  per donor  expected = {exp_sum}")
print("  Σ_ct SenBurden per donor  = senescent fraction among KEPT types")

# ── 4. RESULTS ─────────────────────────────────────────────────────────────────
print("\n" + "─"*70 + "\n4. RESULTS\n" + "─"*70)
chk = df_burden.groupby("Donor").agg(prop_sum=("CellProp","sum"), burden_sum=("SenBurden","sum"))
print(f"  CellProp Σ range: [{chk.prop_sum.min():.3f}, {chk.prop_sum.max():.3f}]  (expected {exp_sum})")
print(f"  senescent fraction over kept types (mean of Σ burden): {chk.burden_sum.mean()*100:.2f}%\n")
burden_summary = (df_burden.groupby([GROUP_VAR,"Cell_Type"], observed=True)
                  .agg(mean_burden=("SenBurden","mean"), mean_prop=("CellProp","mean"),
                       mean_senfrac=("SenFrac","mean"), n_donors=("Donor","nunique")).reset_index())
print("  mean SenBurden (% of all cells) by group × cell type:")
print((burden_summary.assign(v=lambda d:(d.mean_burden*100).round(3))
       .pivot(index="Cell_Type", columns=GROUP_VAR, values="v")).to_string())
save_table(df_burden, f"burden_donor_celltype_{CONDITION_TAG}")
save_table(burden_summary, f"burden_summary_{CONDITION_TAG}")

# ── 5. VISUALIZE — stacked burden by group ─────────────────────────────────────
print("\n" + "─"*70 + "\n5. VISUALIZE\n" + "─"*70)
pivot = (burden_summary.pivot(index=GROUP_VAR, columns="Cell_Type", values="mean_burden")
         .reindex(GROUP_ORDER)[active_cts] * 100)
fig, ax = plt.subplots(figsize=(1.1*len(GROUP_ORDER)+2.5, 4.2))
bottom = np.zeros(len(GROUP_ORDER))
for c in active_cts:
    ax.bar(range(len(GROUP_ORDER)), pivot[c].values, bottom=bottom,
           color=CELL_TYPE_COLORS.get(c, "#999999"), width=0.7, edgecolor="white", linewidth=0.4, label=c)
    bottom += pivot[c].values
ax.set_xticks(range(len(GROUP_ORDER))); ax.set_xticklabels(GROUP_ORDER, rotation=25, ha="right", fontsize=8)
ax.set_ylabel("Senescence burden (% of all cells)" + ("" if not RENORMALIZE_TO_KEPT else " (kept-type composition)"))
ax.set_title(f"Senescent-cell load by cell type — {CONDITION_TAG}")
ax.spines[["top","right"]].set_visible(False)
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", frameon=False, fontsize=8, title="Cell type")
fig.tight_layout(); save_figure(fig, f"burden_stacked_{CONDITION_TAG}"); plt.show()
print("\n✓ §9.1 complete")

---
## 16 · Composition — per donor

**Why.** Horizontal stacked burden per donor, faceted by group. The point is to see donor heterogeneity before any model averages over it.

In [ ]:
# =============================================================================
# §9.1 — VISUALIZE (1/2): PER-DONOR horizontal stacked burden, faceted by group
# =============================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
print("="*80); print("§9.1 VIZ — per-donor stacked burden"); print("="*80)

CT_PALETTE = {  # established cell-type colors (matches meta/main fig)
    "Astrocyte":"#E41A1C","Excitatory":"#377EB8","Inhibitory":"#4DAF4A",
    "Microglia":"#984EA3","OPC":"#FF7F00","Oligodendrocyte":"#A65628",
    "Endothelial":"#F781BF","Pericyte":"#999999","Adaptive":"#A6CEE3",
    "PVM":"#FDBF6F","VLMC":"#CAB2D6","VSMC":"#B15928",
}
CT_PALETTE = {**CT_PALETTE, **{c: CELL_TYPE_COLORS.get(c, CT_PALETTE.get(c,"#999")) for c in active_cts}}

# wide donor × cell-type burden matrix (%), donor group + total
wide = (df_burden.assign(burden_pct=lambda d: d.SenBurden*100)
        .pivot_table(index=["Donor", GROUP_VAR], columns="Cell_Type",
                     values="burden_pct", observed=True)[active_cts].fillna(0))
wide["__total__"] = wide[active_cts].sum(axis=1)
wide = wide.reset_index()

groups = [g for g in GROUP_ORDER if g in wide[GROUP_VAR].unique()]
n_g = len(groups)
counts = wide.groupby(GROUP_VAR, observed=True).size()

fig, axes = plt.subplots(n_g, 1, figsize=(7.2, max(4.5, 0.045*len(wide) + 0.5*n_g)),
                         sharex=True, gridspec_kw={"height_ratios":[counts[g] for g in groups]})
if n_g == 1: axes = [axes]
xmax = wide["__total__"].max()*1.05

for ax, grp in zip(axes, groups):
    sub = wide[wide[GROUP_VAR]==grp].sort_values("__total__", ascending=True).reset_index(drop=True)
    y = np.arange(len(sub)); left = np.zeros(len(sub))
    for c in active_cts:
        ax.barh(y, sub[c].values, left=left, height=0.9,
                color=CT_PALETTE[c], edgecolor="none", label=c)
        left += sub[c].values
    ax.set_xlim(0, xmax); ax.set_ylim(-0.6, len(sub)-0.4)
    ax.set_yticks([]); ax.set_ylabel(grp.replace("Age_","").replace("_","–"),
                                     rotation=0, ha="right", va="center", fontsize=8.5, labelpad=8)
    ax.tick_params(labelsize=8)
    for s in ["top","right","left"]: ax.spines[s].set_visible(False)
    ax.text(0.99, 0.92, f"n={len(sub)}", transform=ax.transAxes, ha="right", va="top",
            fontsize=7.5, color="#666")

axes[-1].set_xlabel("Senescence burden (% of all cells)", fontsize=9)
handles = [plt.Rectangle((0,0),1,1, color=CT_PALETTE[c]) for c in active_cts]
fig.legend(handles, active_cts, loc="lower center", ncol=min(len(active_cts),6),
           frameon=False, fontsize=8, bbox_to_anchor=(0.5, -0.04))
fig.suptitle(f"Per-donor microglial-to-glial senescence burden by age — {CONDITION_TAG}",
             fontsize=10.5, y=0.995)
fig.tight_layout(rect=[0,0.03,1,0.97])
save_figure(fig, f"burden_stacked_perdonor_{CONDITION_TAG}"); plt.show()
print("\n✓ per-donor stacked burden complete")

---
## 17 · Composition — per group

**Why.** The same stack collapsed to group means.

In [ ]:
# =============================================================================
# §9.1 — VISUALIZE (2/2): PER-GROUP horizontal stacked burden (group means)
# =============================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
print("="*80); print("§9.1 VIZ — per-group stacked burden"); print("="*80)

# group × cell-type mean burden (%) — wide
gm = (burden_summary.assign(v=lambda d: d.mean_burden*100)
      .pivot(index=GROUP_VAR, columns="Cell_Type", values="v")
      .reindex(GROUP_ORDER)[active_cts].fillna(0))
groups = list(gm.index)
y = np.arange(len(groups))[::-1]            # youngest at top

fig, ax = plt.subplots(figsize=(7.6, 0.62*len(groups)+1.6))
left = np.zeros(len(groups))
for c in active_cts:
    vals = gm[c].values
    ax.barh(y, vals, left=left, height=0.68, color=CT_PALETTE[c],
            edgecolor="white", linewidth=0.6, label=c)
    for yi, (v, l) in enumerate(zip(vals, left)):
        if v >= 0.30:                       # label only segments wide enough to read
            ax.text(l + v/2, y[yi], f"{v:.2f}", ha="center", va="center",
                    fontsize=7.5, color="white", fontweight="bold")
    left += vals

ax.set_yticks(y); ax.set_yticklabels([g.replace("Age_","").replace("_","–") for g in groups], fontsize=9)
ax.set_xlabel("Senescence burden (% of all cells)", fontsize=9)
ax.set_xlim(0, left.max()*1.04)
ax.tick_params(labelsize=8)
for s in ["top","right","left"]: ax.spines[s].set_visible(False)
# total at bar end
for yi, tot in zip(y, gm[active_cts].sum(axis=1).values):
    ax.text(tot+left.max()*0.008, yi, f"{tot:.2f}%", va="center", ha="left", fontsize=8, color="#444")

handles = [plt.Rectangle((0,0),1,1, color=CT_PALETTE[c]) for c in active_cts]
ax.legend(handles, active_cts, loc="lower center", ncol=min(len(active_cts),6),
          frameon=False, fontsize=8, bbox_to_anchor=(0.5, -0.22))
ax.set_title(f"Senescence burden composition by age group — {CONDITION_TAG}", fontsize=10.5, pad=8)
fig.tight_layout()
save_figure(fig, f"burden_stacked_pergroup_{CONDITION_TAG}"); plt.show()
print("\n✓ per-group stacked burden complete")

---
## 18 · Δ-decomposition

**Why.** The endpoint contrast, split into the two things that can drive it.

**`ContributionPct`** = share of the total burden increase attributable to this cell type. Abundance-weighted — this is population impact.

**`ΔSenFrac`** = change in per-cell propensity. Abundance-independent — this is intrinsic susceptibility.

A common type can dominate contribution while barely moving in ΔSenFrac, and a rare type can do the reverse. Descriptive endpoint contrast; the inferential test is sections 20-22.

In [ ]:
# =============================================================================
# §9.2 — BURDEN: Δ-DECOMPOSITION (contribution + intrinsic susceptibility)
# =============================================================================
# FROM→TO endpoint contrast. Two complementary axes:
#   ContributionPct = share of the total burden increase (abundance-weighted)
#   ΔSenFrac        = per-cell propensity change (abundance-independent)
# Stages: 1 DATA · 2 PREP · 3 STATS(defn) · 4 RESULTS · 5 VISUALIZE
# =============================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
print("=" * 80); print("§9.2 — BURDEN: Δ-DECOMPOSITION"); print("=" * 80)

# ── 1. DATA ──────────────────────────────────────────────────────────────────
print("\n" + "─"*70 + "\n1. DATA — burden_summary endpoints\n" + "─"*70)
print(f"  contrast: {GROUP_FROM}  →  {GROUP_TO}")
ends = burden_summary[burden_summary[GROUP_VAR].isin([GROUP_FROM, GROUP_TO])]
print(ends.assign(mean_burden=lambda d:(d.mean_burden*100).round(3),
                  mean_senfrac=lambda d:(d.mean_senfrac*100).round(2))
      [[GROUP_VAR,"Cell_Type","mean_burden","mean_senfrac","n_donors"]].to_string(index=False))

# ── 2. PREP ──────────────────────────────────────────────────────────────────
print("\n" + "─"*70 + "\n2. PREP — contrib frame\n" + "─"*70)
piv_b = burden_summary.pivot(index="Cell_Type", columns=GROUP_VAR, values="mean_burden")[[GROUP_FROM, GROUP_TO]]
piv_f = burden_summary.pivot(index="Cell_Type", columns=GROUP_VAR, values="mean_senfrac")[[GROUP_FROM, GROUP_TO]]
contrib = pd.DataFrame({
    "Cell_Type":    piv_b.index,
    "Burden_from":  piv_b[GROUP_FROM], "Burden_to": piv_b[GROUP_TO],
    "DeltaBurden":  piv_b[GROUP_TO] - piv_b[GROUP_FROM],
    "DeltaSenFrac": piv_f[GROUP_TO] - piv_f[GROUP_FROM],
}).reset_index(drop=True)
tot = contrib["DeltaBurden"].sum()
contrib["ContributionPct"] = np.where(abs(tot) > 1e-9, 100*contrib["DeltaBurden"]/tot, np.nan)
contrib = contrib.sort_values("DeltaBurden", ascending=False).reset_index(drop=True)
print(contrib.round({"Burden_from":4,"Burden_to":4,"DeltaBurden":4,"DeltaSenFrac":4,"ContributionPct":1}).to_string(index=False))

# ── 3. STATS (definitions) ─────────────────────────────────────────────────────
print("\n" + "─"*70 + "\n3. STATS — definitions\n" + "─"*70)
print(f"  DeltaBurden_ct   = mean_burden[{GROUP_TO}] − mean_burden[{GROUP_FROM}]")
print(f"  ContributionPct  = 100 · DeltaBurden_ct / Σ_ct DeltaBurden   (population-impact share)")
print(f"  DeltaSenFrac_ct  = mean_senfrac[{GROUP_TO}] − mean_senfrac[{GROUP_FROM}]   (abundance-independent)")
print("  NOTE: descriptive endpoint contrast; inferential trajectory test is §9.3.")
# worked example
ex = contrib.iloc[0]
print(f"\n  ▸ WORKED EXAMPLE — {ex.Cell_Type} (top contributor):")
print(f"    DeltaBurden  = {ex.Burden_to*100:.3f} − {ex.Burden_from*100:.3f} = {ex.DeltaBurden*100:.3f} pp")
print(f"    Contribution = 100 · {ex.DeltaBurden*100:.3f} / {tot*100:.3f} = {ex.ContributionPct:.1f}%")

# ── 4. RESULTS ─────────────────────────────────────────────────────────────────
print("\n" + "─"*70 + "\n4. RESULTS\n" + "─"*70)
print(f"  Σ DeltaBurden ({GROUP_FROM}→{GROUP_TO}) = {tot*100:.3f} pp"
      + ("" if abs(tot) > 1e-9 else "  ⚠ ≈0 — ContributionPct undefined"))
print(f"  senescent fraction (kept types): {piv_b[GROUP_FROM].sum()*100:.2f}% → {piv_b[GROUP_TO].sum()*100:.2f}%")
print("\n  top contributors to the increase:")
print(contrib.head(5).assign(ContributionPct=lambda d:d.ContributionPct.round(1))
      [["Cell_Type","DeltaBurden","ContributionPct"]].to_string(index=False))
print("\n  highest intrinsic susceptibility (ΔSenFrac):")
print(contrib.sort_values("DeltaSenFrac", ascending=False).head(5)
      .assign(DeltaSenFrac=lambda d:(d.DeltaSenFrac*100).round(2))[["Cell_Type","DeltaSenFrac"]].to_string(index=False))
save_table(contrib, f"burden_contribution_{CONDITION_TAG}")

# ── 5. VISUALIZE — contribution (left) + ΔSenFrac (right) ──────────────────────
print("\n" + "─"*70 + "\n5. VISUALIZE\n" + "─"*70)
fig, axes = plt.subplots(1, 2, figsize=(11, 0.42*len(contrib)+1.8))
panels = [("ContributionPct", f"Contribution to Δburden ({GROUP_FROM}→{GROUP_TO})", "% of total increase", 1.0),
          ("DeltaSenFrac",    "Intrinsic susceptibility (ΔSenFrac)", "Δ senescent fraction (pp)", 100.0)]
for ax, (col, ttl, xl, scale) in zip(axes, panels):
    d = contrib.sort_values(col)
    ax.barh(range(len(d)), d[col]*scale,
            color=[CT_PALETTE.get(c, "#999999") for c in d["Cell_Type"]],
            edgecolor="#333333", linewidth=0.3)
    ax.axvline(0, color="grey", lw=0.5)
    ax.set_yticks(range(len(d))); ax.set_yticklabels(d["Cell_Type"], fontsize=8)
    ax.set_title(ttl, fontsize=10); ax.set_xlabel(xl, fontsize=8)
    ax.spines[["top","right"]].set_visible(False)
fig.suptitle(f"Senescence-burden decomposition — {CONDITION_TAG}", fontsize=11, y=1.02)
fig.tight_layout(); save_figure(fig, f"burden_contribution_susceptibility_{CONDITION_TAG}"); plt.show()
print("\n✓ §9.2 complete")

---
## 19 · Susceptibility index

**Why.** `SusceptibilityIndex = ΔBurden / MeanProp`. Reported alongside ΔSenFrac because the two coincide when abundance is stable and diverge when it is not. The cell prints its own caveat: the index inflates for rare types, so read the columns together.

In [ ]:
# =============================================================================
# §9.2b — add literal SusceptibilityIndex (ΔBurden / MeanProp) per collaborator
# =============================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
print("=" * 80); print("§9.2b — SusceptibilityIndex"); print("=" * 80)

# MeanProp = mean CellProp across donors, per cell type (collaborator's mean(CellProp))
mean_prop = df_burden.groupby("Cell_Type", observed=True)["CellProp"].mean()
contrib["MeanProp"] = contrib["Cell_Type"].map(mean_prop).astype(float)
contrib["SusceptibilityIndex"] = np.where(contrib["MeanProp"] > 1e-9,
                                          contrib["DeltaBurden"] / contrib["MeanProp"], np.nan)

# ── definition ──
print("\n  SusceptibilityIndex = DeltaBurden / MeanProp")
print(f"    MeanProp = mean CellProp across donors (per cell type)")
print(f"    DeltaBurden = SenBurden[{GROUP_TO}] − SenBurden[{GROUP_FROM}]")
print("    ≈ ΔSenFrac, but divided by MeanProp (inflates for rare types — compare columns)\n")

# ── ranked table: both susceptibility measures side by side ──
show = (contrib.sort_values("SusceptibilityIndex", ascending=False)
        .assign(MeanProp=lambda d:(d.MeanProp*100).round(2),
                DeltaBurden=lambda d:(d.DeltaBurden*100).round(4),
                SusceptibilityIndex=lambda d:(d.SusceptibilityIndex*100).round(3),
                DeltaSenFrac=lambda d:(d.DeltaSenFrac*100).round(2))
        [["Cell_Type","MeanProp","DeltaBurden","SusceptibilityIndex","DeltaSenFrac"]])
print("  ranked by SusceptibilityIndex (all ×100):")
print("  MeanProp=%cells · DeltaBurden=pp · SuscIndex=DeltaBurden/MeanProp ×100 · DeltaSenFrac=pp")
print(show.to_string(index=False))
save_table(contrib, f"burden_contribution_{CONDITION_TAG}")   # re-save with new cols

# ── viz: Contribution | SusceptibilityIndex (collaborator's metric) ──
fig, axes = plt.subplots(1, 2, figsize=(11, 0.42*len(contrib)+1.8))
panels = [("ContributionPct",     f"Contribution to Δburden ({GROUP_FROM}→{GROUP_TO})", "% of total increase", 1.0),
          ("SusceptibilityIndex", "Susceptibility index (ΔBurden / MeanProp)",          "index ×100",          100.0)]
for ax, (col, ttl, xl, scale) in zip(axes, panels):
    d = contrib.sort_values(col)
    ax.barh(range(len(d)), d[col]*scale,
            color=[CELL_TYPE_COLORS.get(c, "#999999") for c in d["Cell_Type"]],
            edgecolor="#333333", linewidth=0.3)
    ax.axvline(0, color="grey", lw=0.5)
    ax.set_yticks(range(len(d))); ax.set_yticklabels(d["Cell_Type"], fontsize=8)
    ax.set_title(ttl, fontsize=10); ax.set_xlabel(xl, fontsize=8)
    ax.spines[["top","right"]].set_visible(False)
fig.suptitle(f"Burden contribution vs susceptibility index — {CONDITION_TAG}", fontsize=11, y=1.02)
fig.tight_layout(); save_figure(fig, f"burden_contribution_susceptibilityindex_{CONDITION_TAG}"); plt.show()
print("\n✓ §9.2b complete")

---
## 20 · Burden trajectory — OLS

**Why.** Continuous age as the predictor rather than an endpoint contrast. HC3 standard errors, donor-level, per cell type. Effect is per decade.

In [ ]:
# =============================================================================
# §9.3a — BURDEN TRAJECTORY · OLS (donor-level, HC3) · predictor = Age/decade
# =============================================================================
# Predictor switched from age-bin ordinal → continuous Age/10 (per decade).
# Interpretable slope: burden pp change per 10 years. Age is the EXPOSURE here,
# so it is NOT also a covariate (covariates = Sex, Cohort only).
# Stages: 1 DATA · 2 PREP · 3 FORMULA · 4 RESULTS · 5 VISUALIZE
# =============================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from matplotlib.patches import Polygon
print("=" * 80); print("§9.3a — OLS (Age per decade)"); print("=" * 80)

CT_COL = CELL_TYPE_COLORS
assert "Age" in df_burden.columns, "continuous Age not in df_burden — add it to GROUP_SPEC covariates_donor or merge from df_cells"

# ── 1. DATA ──────────────────────────────────────────────────────────────────
print("\n" + "─"*70 + "\n1. DATA — df_burden (donor × CT)\n" + "─"*70)
print(f"  shape: {df_burden.shape} | donors: {df_burden['Donor'].nunique()} | cell types: {len(active_cts)}")
print(df_burden[["Donor","Cell_Type","SenBurden","Age",GROUP_VAR]].head().to_string(index=False))

# ── 2. PREP ──────────────────────────────────────────────────────────────────
print("\n" + "─"*70 + "\n2. PREP — continuous age predictor + covariates\n" + "─"*70)
df_burden["age_dec"] = df_burden["Age"].astype(float) / 10.0          # per-decade
covs = [c for c in GROUP_SPEC.get("covariates_donor", [])
        if c in df_burden.columns and c not in (GROUP_VAR, "Age", "Age_scaled")]
print(f"  predictor  : age_dec = Age/10  (range {df_burden['age_dec'].min():.1f}–{df_burden['age_dec'].max():.1f} decades)")
print(f"  covariates : {covs or '(none)'}   (Age excluded — it's the exposure)")
exb = df_burden[df_burden.Donor == EX_DONOR][["Cell_Type","SenBurden","Age","age_dec"]]
print(f"  ▸ WORKED EXAMPLE — donor {EX_DONOR}:")
print(exb.assign(SenBurden=lambda d:(d.SenBurden*100).round(3)).to_string(index=False))

# ── 3. FORMULA ─────────────────────────────────────────────────────────────────
print("\n" + "─"*70 + "\n3. STATS — formula\n" + "─"*70)
rhs = " + ".join(["age_dec"] + [f"C({c})" if df_burden[c].dtype == object else c for c in covs])
FORMULA = f"SenBurden ~ {rhs}"
print(f"  {FORMULA}")
print(f"  estimator: OLS, HC3 robust SE | unit = donor | test = slope of age_dec")
print(f"  slope interpretation: burden pp change per DECADE | BH across {len(active_cts)} cell types")

# ── 4. RESULTS ─────────────────────────────────────────────────────────────────
print("\n" + "─"*70 + "\n4. RESULTS\n" + "─"*70)
rows = []
for ct in active_cts:
    sub = df_burden[df_burden.Cell_Type == ct].dropna(subset=["SenBurden","age_dec"])
    if sub.Donor.nunique() < MIN_DONORS_PER_CT or sub.SenBurden.std() < 1e-9:
        continue
    m = smf.ols(FORMULA, data=sub).fit(cov_type="HC3"); ci = m.conf_int().loc["age_dec"]
    rows.append(dict(Cell_Type=ct, n_donors=sub.Donor.nunique(), slope=m.params["age_dec"],
                     se=m.bse["age_dec"], lo=ci[0], hi=ci[1], p=m.pvalues["age_dec"]))
ols = pd.DataFrame(rows); ols["padj"] = bh_correction(ols["p"])
ols = ols.sort_values("slope", ascending=False).reset_index(drop=True)
print(ols.assign(slope=lambda d:(d.slope*100).round(3), se=lambda d:(d.se*100).round(3),
                 p=lambda d:d.p.map(fmt_p), padj=lambda d:d.padj.map(fmt_p)+d.padj.map(sig_stars))
      [["Cell_Type","n_donors","slope","se","p","padj"]].to_string(index=False))
print("  (slope/se ×100 = burden pp per DECADE)")
save_table(ols, f"burden_ols_{CONDITION_TAG}")

# ── 5. VISUALIZE — COMPACT table-style forest ──────────────────────────────────
print("\n" + "─"*70 + "\n5. VISUALIZE\n" + "─"*70)
dfp = ols.copy(); n_rows = len(dfp); EST = "#08519C"
ROW_H = 0.46
fig = plt.figure(figsize=(7.8, max(2.6, n_rows*ROW_H + 1.4))); ax = fig.add_subplot(111)
FH, FC, FD, FS = 9, 9, 8.5, 7.5
x = {"ct":0.005, "fL":0.20, "fR":0.56}
x["beta"], x["ci"], x["p"], x["fdr"] = 0.66, 0.80, 0.90, 0.985

pe = np.concatenate([(dfp.slope*100).values,(dfp.lo*100).values,(dfp.hi*100).values])
pmin, pmax = pe.min(), pe.max(); pad = max((pmax-pmin)*0.5, 0.05); xmin, xmax = pmin-pad, pmax+pad
def to_x(v): v = np.clip(v, xmin, xmax); return x["fL"] + (v-xmin)/(xmax-xmin)*(x["fR"]-x["fL"])
AR = 0.006
y_first = n_rows-1; y_hdr = y_first+0.55; y_line = y_hdr-0.10; y_axis = -0.5

for tv in np.arange(np.ceil(xmin), np.floor(xmax)+1):
    if xmin <= tv <= xmax and abs(tv) > 1e-9:
        ax.plot([to_x(tv)]*2, [y_axis, y_line], color="#eeeeee", lw=0.5, zorder=0)
for idx in range(n_rows):
    if idx % 2 == 0:
        yy = n_rows-idx-1; ax.axhspan(yy-0.5, yy+0.5, xmin=0.004, xmax=0.996, color="#f7f7f7", zorder=0)
ax.axvline(to_x(0), color="#888888", ls="--", lw=0.8, ymin=0.05, ymax=0.92, alpha=0.6, zorder=1)

ax.text(x["ct"], y_hdr, "Cell Type", fontsize=FH, fontweight="bold", ha="left", va="bottom")
for lab, k, ha in [("β (SE)","beta","center"), ("95% CI","ci","center"), ("p","p","right"), ("FDR","fdr","right")]:
    ax.text(x[k], y_hdr, lab, fontsize=FH, fontweight="bold", ha=ha, va="bottom")
ax.plot([0.005, 0.99], [y_line, y_line], "k-", lw=1, clip_on=False)

for idx, row in dfp.iterrows():
    y = n_rows-idx-1; sig = row.padj < FDR_THRESHOLD; rawp = row.p < 0.05
    ctc = CT_COL.get(row.Cell_Type, "#808080"); pre = "★ " if sig else ("* " if rawp else "")
    ax.text(x["ct"], y, pre+row.Cell_Type, fontsize=FC, fontweight="bold" if rawp else "normal",
            ha="left", va="center", color=ctc if sig else "#444444")
    b, lo, hi = row.slope*100, row.lo*100, row.hi*100; xb, xl, xh = to_x(b), to_x(lo), to_x(hi)
    ax.plot([xl, xh], [y, y], color=ctc, lw=1.8, alpha=0.75, zorder=2)
    if lo < xmin: ax.add_patch(Polygon([[xl,y],[xl+AR,y+0.10],[xl+AR,y-0.10]], fc=ctc, ec="none", alpha=0.75, zorder=2))
    if hi > xmax: ax.add_patch(Polygon([[xh,y],[xh-AR,y+0.10],[xh-AR,y-0.10]], fc=ctc, ec="none", alpha=0.75, zorder=2))
    ax.plot(xb, y, "o", ms=6, color=ctc, mec="black", mew=0.5, zorder=3)
    ax.text(x["beta"], y, f"{b:.3f} ({row.se*100:.3f})", fontsize=FD, ha="center", va="center",
            fontweight="bold" if rawp else "normal")
    ax.text(x["ci"],   y, f"[{lo:.2f}, {hi:.2f}]", fontsize=FD, ha="center", va="center", color="#444")
    ax.text(x["p"],    y, fmt_p(row.p), fontsize=FD, ha="right", va="center", fontweight="bold" if rawp else "normal")
    ax.text(x["fdr"],  y, fmt_p(row.padj)+sig_stars(row.padj), fontsize=FD, ha="right", va="center",
            fontweight="bold" if sig else "normal", color=EST if sig else "#333")

ax.plot([x["fL"], x["fR"]], [y_axis, y_axis], "k-", lw=1)
for tv in np.arange(np.ceil(xmin), np.floor(xmax)+1):
    if xmin <= tv <= xmax:
        tx = to_x(tv); ax.plot([tx, tx], [y_axis, y_axis-0.10], "k-", lw=0.8)
        ax.text(tx, y_axis-0.17, f"{tv:g}", fontsize=FS, ha="center", va="top")
ax.text((x["fL"]+x["fR"])/2, y_axis-0.55, "Burden change per decade (pp)", fontsize=FD, fontweight="bold", ha="center")
ax.text(0.5, y_hdr+0.45, f"Burden trajectory — OLS (HC3, per decade) — {CONDITION_TAG}", fontsize=FH+1, fontweight="bold", ha="center")
ax.text(0.005, y_axis-0.55, "★ FDR<0.05   * p<0.05", fontsize=FS, ha="left", va="top", color="#666")
ax.set_xlim(-0.01, 1.01); ax.set_ylim(-1.25, y_hdr+0.55); ax.axis("off")
plt.tight_layout(); save_figure(fig, f"burden_forest_OLS_{CONDITION_TAG}"); plt.show()
print("\n✓ §9.3a complete")

---
## 21 · Burden trajectory — RLM

**Why.** Same design under Huber M-estimation, so a handful of extreme donors cannot set the slope.

In [ ]:
# =============================================================================
# §9.3b — BURDEN TRAJECTORY · RLM (robust HuberT) · predictor = Age/decade
# =============================================================================
# Robust counterpart to §9.3a: down-weights high-burden outlier donors.
# Compare to OLS — agreement = trend not outlier-driven; divergence = it was.
# Stages: 1 DATA · 2 PREP · 3 FORMULA · 4 RESULTS · 5 VISUALIZE
# =============================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm, statsmodels.formula.api as smf
from matplotlib.patches import Polygon
print("=" * 80); print("§9.3b — RLM (Age per decade)"); print("=" * 80)

CT_COL = CELL_TYPE_COLORS
assert "Age" in df_burden.columns, "continuous Age not in df_burden"

# ── 1. DATA ──────────────────────────────────────────────────────────────────
print("\n" + "─"*70 + "\n1. DATA — df_burden (donor × CT)\n" + "─"*70)
print(f"  shape: {df_burden.shape} | donors: {df_burden['Donor'].nunique()} | cell types: {len(active_cts)}")
print(df_burden[["Donor","Cell_Type","SenBurden","Age",GROUP_VAR]].head().to_string(index=False))

# ── 2. PREP ──────────────────────────────────────────────────────────────────
print("\n" + "─"*70 + "\n2. PREP — continuous age predictor + covariates\n" + "─"*70)
df_burden["age_dec"] = df_burden["Age"].astype(float) / 10.0
covs = [c for c in GROUP_SPEC.get("covariates_donor", [])
        if c in df_burden.columns and c not in (GROUP_VAR, "Age", "Age_scaled")]
print(f"  predictor  : age_dec = Age/10  (range {df_burden['age_dec'].min():.1f}–{df_burden['age_dec'].max():.1f} decades)")
print(f"  covariates : {covs or '(none)'}   (Age excluded — it's the exposure)")
exb = df_burden[df_burden.Donor == EX_DONOR][["Cell_Type","SenBurden","Age","age_dec"]]
print(f"  ▸ WORKED EXAMPLE — donor {EX_DONOR}:")
print(exb.assign(SenBurden=lambda d:(d.SenBurden*100).round(3)).to_string(index=False))

# ── 3. FORMULA ─────────────────────────────────────────────────────────────────
print("\n" + "─"*70 + "\n3. STATS — formula\n" + "─"*70)
rhs = " + ".join(["age_dec"] + [f"C({c})" if df_burden[c].dtype == object else c for c in covs])
FORMULA = f"SenBurden ~ {rhs}"
print(f"  {FORMULA}")
print(f"  estimator: RLM, HuberT M-estimator | unit = donor | test = slope of age_dec")
print(f"  slope interpretation: burden pp change per DECADE | BH across {len(active_cts)} cell types")

# ── 4. RESULTS ─────────────────────────────────────────────────────────────────
print("\n" + "─"*70 + "\n4. RESULTS\n" + "─"*70)
rows = []
for ct in active_cts:
    sub = df_burden[df_burden.Cell_Type == ct].dropna(subset=["SenBurden","age_dec"])
    if sub.Donor.nunique() < MIN_DONORS_PER_CT or sub.SenBurden.std() < 1e-9:
        continue
    m = smf.rlm(FORMULA, data=sub, M=sm.robust.norms.HuberT()).fit(); ci = m.conf_int().loc["age_dec"]
    rows.append(dict(Cell_Type=ct, n_donors=sub.Donor.nunique(), slope=m.params["age_dec"],
                     se=m.bse["age_dec"], lo=ci[0], hi=ci[1], p=m.pvalues["age_dec"]))
rlm = pd.DataFrame(rows); rlm["padj"] = bh_correction(rlm["p"])
rlm = rlm.sort_values("slope", ascending=False).reset_index(drop=True)
print(rlm.assign(slope=lambda d:(d.slope*100).round(3), se=lambda d:(d.se*100).round(3),
                 p=lambda d:d.p.map(fmt_p), padj=lambda d:d.padj.map(fmt_p)+d.padj.map(sig_stars))
      [["Cell_Type","n_donors","slope","se","p","padj"]].to_string(index=False))
print("  (slope/se ×100 = burden pp per DECADE)")
save_table(rlm, f"burden_rlm_{CONDITION_TAG}")

# ── 5. VISUALIZE — COMPACT table-style forest ──────────────────────────────────
print("\n" + "─"*70 + "\n5. VISUALIZE\n" + "─"*70)
dfp = rlm.copy(); n_rows = len(dfp); EST = "#A32D2D"
ROW_H = 0.46
fig = plt.figure(figsize=(7.8, max(2.6, n_rows*ROW_H + 1.4))); ax = fig.add_subplot(111)
FH, FC, FD, FS = 9, 9, 8.5, 7.5
x = {"ct":0.005, "fL":0.20, "fR":0.56}
x["beta"], x["ci"], x["p"], x["fdr"] = 0.66, 0.80, 0.90, 0.985

pe = np.concatenate([(dfp.slope*100).values,(dfp.lo*100).values,(dfp.hi*100).values])
pmin, pmax = pe.min(), pe.max(); pad = max((pmax-pmin)*0.5, 0.05); xmin, xmax = pmin-pad, pmax+pad
def to_x(v): v = np.clip(v, xmin, xmax); return x["fL"] + (v-xmin)/(xmax-xmin)*(x["fR"]-x["fL"])
AR = 0.006
y_first = n_rows-1; y_hdr = y_first+0.55; y_line = y_hdr-0.10; y_axis = -0.5

for tv in np.arange(np.ceil(xmin), np.floor(xmax)+1):
    if xmin <= tv <= xmax and abs(tv) > 1e-9:
        ax.plot([to_x(tv)]*2, [y_axis, y_line], color="#eeeeee", lw=0.5, zorder=0)
for idx in range(n_rows):
    if idx % 2 == 0:
        yy = n_rows-idx-1; ax.axhspan(yy-0.5, yy+0.5, xmin=0.004, xmax=0.996, color="#f7f7f7", zorder=0)
ax.axvline(to_x(0), color="#888888", ls="--", lw=0.8, ymin=0.05, ymax=0.92, alpha=0.6, zorder=1)

ax.text(x["ct"], y_hdr, "Cell Type", fontsize=FH, fontweight="bold", ha="left", va="bottom")
for lab, k, ha in [("β (SE)","beta","center"), ("95% CI","ci","center"), ("p","p","right"), ("FDR","fdr","right")]:
    ax.text(x[k], y_hdr, lab, fontsize=FH, fontweight="bold", ha=ha, va="bottom")
ax.plot([0.005, 0.99], [y_line, y_line], "k-", lw=1, clip_on=False)

for idx, row in dfp.iterrows():
    y = n_rows-idx-1; sig = row.padj < FDR_THRESHOLD; rawp = row.p < 0.05
    ctc = CT_COL.get(row.Cell_Type, "#808080"); pre = "★ " if sig else ("* " if rawp else "")
    ax.text(x["ct"], y, pre+row.Cell_Type, fontsize=FC, fontweight="bold" if rawp else "normal",
            ha="left", va="center", color=ctc if sig else "#444444")
    b, lo, hi = row.slope*100, row.lo*100, row.hi*100; xb, xl, xh = to_x(b), to_x(lo), to_x(hi)
    ax.plot([xl, xh], [y, y], color=ctc, lw=1.8, alpha=0.75, zorder=2)
    if lo < xmin: ax.add_patch(Polygon([[xl,y],[xl+AR,y+0.10],[xl+AR,y-0.10]], fc=ctc, ec="none", alpha=0.75, zorder=2))
    if hi > xmax: ax.add_patch(Polygon([[xh,y],[xh-AR,y+0.10],[xh-AR,y-0.10]], fc=ctc, ec="none", alpha=0.75, zorder=2))
    ax.plot(xb, y, "o", ms=6, color=ctc, mec="black", mew=0.5, zorder=3)
    ax.text(x["beta"], y, f"{b:.3f} ({row.se*100:.3f})", fontsize=FD, ha="center", va="center",
            fontweight="bold" if rawp else "normal")
    ax.text(x["ci"],   y, f"[{lo:.2f}, {hi:.2f}]", fontsize=FD, ha="center", va="center", color="#444")
    ax.text(x["p"],    y, fmt_p(row.p), fontsize=FD, ha="right", va="center", fontweight="bold" if rawp else "normal")
    ax.text(x["fdr"],  y, fmt_p(row.padj)+sig_stars(row.padj), fontsize=FD, ha="right", va="center",
            fontweight="bold" if sig else "normal", color=EST if sig else "#333")

ax.plot([x["fL"], x["fR"]], [y_axis, y_axis], "k-", lw=1)
for tv in np.arange(np.ceil(xmin), np.floor(xmax)+1):
    if xmin <= tv <= xmax:
        tx = to_x(tv); ax.plot([tx, tx], [y_axis, y_axis-0.10], "k-", lw=0.8)
        ax.text(tx, y_axis-0.17, f"{tv:g}", fontsize=FS, ha="center", va="top")
ax.text((x["fL"]+x["fR"])/2, y_axis-0.55, "Burden change per decade (pp)", fontsize=FD, fontweight="bold", ha="center")
ax.text(0.5, y_hdr+0.45, f"Burden trajectory — RLM (HuberT, per decade) — {CONDITION_TAG}", fontsize=FH+1, fontweight="bold", ha="center")
ax.text(0.005, y_axis-0.55, "★ FDR<0.05   * p<0.05", fontsize=FS, ha="left", va="top", color="#666")
ax.set_xlim(-0.01, 1.01); ax.set_ylim(-1.25, y_hdr+0.55); ax.axis("off")
plt.tight_layout(); save_figure(fig, f"burden_forest_RLM_{CONDITION_TAG}"); plt.show()
print("\n✓ §9.3b complete")

---
## 22 · Burden trajectory — GLMM

**Why.** The count form of the same question — binomial mixed model on the senescent / non-senescent split rather than on the proportion.

In [ ]:
# =============================================================================
# §9.3c — BURDEN TRAJECTORY · GLMM (binomial mixed) · predictor = Age/decade
# =============================================================================
#   cbind(n_snc, donor_total − n_snc) ~ age_dec + covs + (1|Donor)   per cell type
#   Fitted P = n_snc/donor_total = SenBurden  →  models BURDEN (abundance-weighted,
#   denominator = ALL donor cells, so CellProp is built in). Same outcome as
#   §9.3a/b, mixed binomial estimator. (1|Donor) = observation-level RE (over-
#   dispersion). Effect = OR per DECADE. Backend = lme4::glmer via rpy2 (§5).
#   Stages: 1 DATA · 2 PREP · 3 FORMULA · 4 RESULTS · 5 VISUALIZE
# =============================================================================
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from matplotlib.patches import Polygon
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
pandas_converter = ro.default_converter + pandas2ri.converter
print("=" * 80); print("§9.3c — GLMM (burden, Age per decade)"); print("=" * 80)

CT_COL = CELL_TYPE_COLORS
GLMM_CONFIG = globals().get("GLMM_CONFIG",
    {"optimizer":"bobyqa","maxfun":100000,"nAGQ":1,"min_cells":100,"min_donors":5})

# ── 1. DATA — donor × CT burden frame (n_snc + donor_total carry the proportion) ──
print("\n" + "─"*70 + "\n1. DATA — df_burden (donor × CT)\n" + "─"*70)
print(f"  shape: {df_burden.shape} | donors: {df_burden['Donor'].nunique()} | cell types: {len(active_cts)}")
print(df_burden[["Donor","Cell_Type","n_snc","donor_total","SenBurden","Age"]].head().to_string(index=False))

# ── 2. PREP — continuous age + binomial response columns + covariates ─────────
print("\n" + "─"*70 + "\n2. PREP — age_dec, binomial response, covariates\n" + "─"*70)
df_burden["age_dec"] = df_burden["Age"].astype(float) / 10.0
df_burden["n_snc"]    = df_burden["n_snc"].astype(int)
df_burden["n_other"]  = (df_burden["donor_total"] - df_burden["n_snc"]).astype(int)
covs = [c for c in GROUP_SPEC.get("covariates_donor", [])
        if c in df_burden.columns and c not in (GROUP_VAR, "Age", "Age_scaled")]
print(f"  predictor : age_dec = Age/10  (range {df_burden['age_dec'].min():.1f}–{df_burden['age_dec'].max():.1f} decades)")
print(f"  response  : cbind(n_snc, donor_total − n_snc)   →  fitted P = SenBurden")
print(f"  covariates: {covs or '(none)'}   (Age = exposure; log-UMI not used at donor level)")
exb = df_burden[df_burden.Donor == EX_DONOR][["Cell_Type","n_snc","donor_total","SenBurden","age_dec"]]
print(f"  ▸ WORKED EXAMPLE — donor {EX_DONOR}:")
print(exb.assign(SenBurden=lambda d:(d.SenBurden*100).round(3)).to_string(index=False))
print(f"    e.g. Astrocyte burden = n_snc/donor_total = {int(exb.iloc[0].n_snc)}/{int(exb.iloc[0].donor_total)} "
      f"= {exb.iloc[0].SenBurden:.4f}  ← this is the modeled probability")

# ── 3. FORMULA ─────────────────────────────────────────────────────────────────
print("\n" + "─"*70 + "\n3. STATS — formula\n" + "─"*70)
rhs = " + ".join(["age_dec"] + covs)
FORMULA = f"cbind(n_snc, n_other) ~ {rhs} + (1|Donor)"
print(f"  {FORMULA}")
print(f"  estimator: lme4::glmer binomial(logit), optimizer={GLMM_CONFIG['optimizer']}, nAGQ={GLMM_CONFIG['nAGQ']}")
print(f"  unit = donor × cell type | (1|Donor) = observation-level RE (over-dispersion)")
print(f"  effect = OR per DECADE on BURDEN | BH across {len(active_cts)} cell types")

# ── 4. RESULTS — per-CT binomial glmer via rpy2 ───────────────────────────────
print("\n" + "─"*70 + "\n4. RESULTS\n" + "─"*70)
def fit_glmer(ct):
    sub = df_burden[df_burden.Cell_Type == ct].dropna(subset=["n_snc","n_other","age_dec"] + covs)
    if sub.Donor.nunique() < GLMM_CONFIG["min_donors"] or sub["n_snc"].sum() < 5:
        print(f"  {ct:18s} ✗ insufficient donors / senescent cells"); return None
    cv = [c for c in covs if not (c in {"Sex","Cohort"} and sub[c].nunique() < 2)]
    f = f"cbind(n_snc, n_other) ~ {' + '.join(['age_dec'] + cv)} + (1|Donor)"
    with localconverter(pandas_converter):
        ro.globalenv["ct_data"] = sub
    rcode = f'''suppressMessages(library(lme4))
    tryCatch({{
        m <- glmer({f}, data=ct_data, family=binomial(link="logit"),
                   control=glmerControl(optimizer="{GLMM_CONFIG['optimizer']}",
                                        optCtrl=list(maxfun={GLMM_CONFIG['maxfun']})),
                   nAGQ={GLMM_CONFIG['nAGQ']})
        cs <- summary(m)$coefficients
        list(ok=TRUE, est=cs["age_dec","Estimate"], se=cs["age_dec","Std. Error"],
             p=cs["age_dec","Pr(>|z|)"])
    }}, error=function(e) list(ok=FALSE, err=as.character(e)))'''
    r = ro.r(rcode)
    if not r.rx2("ok")[0]:
        print(f"  {ct:18s} ✗ {str(r.rx2('err')[0])[:60]}"); return None
    b = float(r.rx2("est")[0]); se = float(r.rx2("se")[0])
    print(f"  {ct:18s} ✓ OR={np.exp(b):.3f} per decade")
    return dict(Cell_Type=ct, n_donors=sub.Donor.nunique(), n_snc_tot=int(sub["n_snc"].sum()),
                beta=b, se=se, lo=b-1.96*se, hi=b+1.96*se, p=float(r.rx2("p")[0]))

rows = [r for r in (fit_glmer(ct) for ct in active_cts) if r]
glmm = pd.DataFrame(rows); glmm["padj"] = bh_correction(glmm["p"])
glmm["OR"] = np.exp(glmm["beta"]); glmm["OR_lo"] = np.exp(glmm["lo"]); glmm["OR_hi"] = np.exp(glmm["hi"])
glmm = glmm.sort_values("beta", ascending=False).reset_index(drop=True)
print()
print(glmm.assign(OR=lambda d:d.OR.round(3), p=lambda d:d.p.map(fmt_p),
                  padj=lambda d:d.padj.map(fmt_p)+d.padj.map(sig_stars))
      [["Cell_Type","n_donors","n_snc_tot","OR","p","padj"]].to_string(index=False))
print("  (OR per decade on burden; >1 = senescent-cell load of this type rises with age)")
save_table(glmm, f"burden_glmm_{CONDITION_TAG}")

# ── 5. VISUALIZE — COMPACT forest, OR/log scale, reference at OR=1 ────────────
print("\n" + "─"*70 + "\n5. VISUALIZE\n" + "─"*70)
dfp = glmm.copy(); n_rows = len(dfp); EST = "#542788"
ROW_H = 0.46
fig = plt.figure(figsize=(7.8, max(2.6, n_rows*ROW_H + 1.4))); ax = fig.add_subplot(111)
FH, FC, FD, FS = 9, 9, 8.5, 7.5
x = {"ct":0.005, "fL":0.20, "fR":0.56}
x["beta"], x["ci"], x["p"], x["fdr"] = 0.66, 0.80, 0.90, 0.985

pe = np.concatenate([dfp.beta.values, dfp.lo.values, dfp.hi.values])
pmin, pmax = pe.min(), pe.max(); pad = max((pmax-pmin)*0.5, 0.05); xmin, xmax = pmin-pad, pmax+pad
def to_x(v): v = np.clip(v, xmin, xmax); return x["fL"] + (v-xmin)/(xmax-xmin)*(x["fR"]-x["fL"])
AR = 0.006
y_first = n_rows-1; y_hdr = y_first+0.55; y_line = y_hdr-0.10; y_axis = -0.5

or_ticks = [0.5, 0.75, 1.0, 1.5, 2.0, 3.0]
for orv in or_ticks:
    lv = np.log(orv)
    if xmin <= lv <= xmax and abs(lv) > 1e-9:
        ax.plot([to_x(lv)]*2, [y_axis, y_line], color="#eeeeee", lw=0.5, zorder=0)
for idx in range(n_rows):
    if idx % 2 == 0:
        yy = n_rows-idx-1; ax.axhspan(yy-0.5, yy+0.5, xmin=0.004, xmax=0.996, color="#f7f7f7", zorder=0)
ax.axvline(to_x(0), color="#888888", ls="--", lw=0.8, ymin=0.05, ymax=0.92, alpha=0.6, zorder=1)

ax.text(x["ct"], y_hdr, "Cell Type", fontsize=FH, fontweight="bold", ha="left", va="bottom")
for lab, k, ha in [("OR (SE)","beta","center"), ("95% CI","ci","center"), ("p","p","right"), ("FDR","fdr","right")]:
    ax.text(x[k], y_hdr, lab, fontsize=FH, fontweight="bold", ha=ha, va="bottom")
ax.plot([0.005, 0.99], [y_line, y_line], "k-", lw=1, clip_on=False)

for idx, row in dfp.iterrows():
    y = n_rows-idx-1; sig = row.padj < FDR_THRESHOLD; rawp = row.p < 0.05
    ctc = CT_COL.get(row.Cell_Type, "#808080"); pre = "★ " if sig else ("* " if rawp else "")
    ax.text(x["ct"], y, pre+row.Cell_Type, fontsize=FC, fontweight="bold" if rawp else "normal",
            ha="left", va="center", color=ctc if sig else "#444444")
    xb, xl, xh = to_x(row.beta), to_x(row.lo), to_x(row.hi)
    ax.plot([xl, xh], [y, y], color=ctc, lw=1.8, alpha=0.75, zorder=2)
    if row.lo < xmin: ax.add_patch(Polygon([[xl,y],[xl+AR,y+0.10],[xl+AR,y-0.10]], fc=ctc, ec="none", alpha=0.75, zorder=2))
    if row.hi > xmax: ax.add_patch(Polygon([[xh,y],[xh-AR,y+0.10],[xh-AR,y-0.10]], fc=ctc, ec="none", alpha=0.75, zorder=2))
    ax.plot(xb, y, "o", ms=6, color=ctc, mec="black", mew=0.5, zorder=3)
    ax.text(x["beta"], y, f"{row.OR:.3f} ({row.se:.3f})", fontsize=FD, ha="center", va="center",
            fontweight="bold" if rawp else "normal")
    ax.text(x["ci"],   y, f"[{row.OR_lo:.2f}, {row.OR_hi:.2f}]", fontsize=FD, ha="center", va="center", color="#444")
    ax.text(x["p"],    y, fmt_p(row.p), fontsize=FD, ha="right", va="center", fontweight="bold" if rawp else "normal")
    ax.text(x["fdr"],  y, fmt_p(row.padj)+sig_stars(row.padj), fontsize=FD, ha="right", va="center",
            fontweight="bold" if sig else "normal", color=EST if sig else "#333")

ax.plot([x["fL"], x["fR"]], [y_axis, y_axis], "k-", lw=1)
for orv in or_ticks:
    lv = np.log(orv)
    if xmin <= lv <= xmax:
        tx = to_x(lv); ax.plot([tx, tx], [y_axis, y_axis-0.10], "k-", lw=0.8)
        ax.text(tx, y_axis-0.17, f"{orv:g}", fontsize=FS, ha="center", va="top")
ax.text((x["fL"]+x["fR"])/2, y_axis-0.55, "OR per decade (log scale)", fontsize=FD, fontweight="bold", ha="center")
ax.text(0.5, y_hdr+0.45, f"Burden trajectory — GLMM (binomial, per decade) — {CONDITION_TAG}", fontsize=FH+1, fontweight="bold", ha="center")
ax.text(0.005, y_axis-0.55, "★ FDR<0.05   * p<0.05", fontsize=FS, ha="left", va="top", color="#666")
ax.set_xlim(-0.01, 1.01); ax.set_ylim(-1.25, y_hdr+0.55); ax.axis("off")
plt.tight_layout(); save_figure(fig, f"burden_forest_GLMM_{CONDITION_TAG}"); plt.show()
print("\n✓ §9.3c complete")

---
## 23 · Group contrast — donor-level OLS

**Why.** One model per cell type per primary variable. Outcome is `prop_snc`, the donor's senescent fraction within that cell type; each non-reference level gets its own contrast.

Donor is the unit, which is what makes this the estimator to read first under the pseudoreplication rule. Guard is `n_donors >= 5`.

**In aging mode, read section 14's note first** — the age bin and continuous `Age` are both in this formula, and the printed header shows it: `prop_snc ~ C(Study_Group, Treatment('Age_20_29')) + Age + C(Sex) + C(Cohort)`.

In [ ]:
# =============================================================================
# §3 — DONOR-LEVEL OLS (per primary × per cell type)
# =============================================================================
# For each spec in PRIMARY_SPECS, fit OLS:
#     prop_snc ~ C(primary, Treatment(ref)) + covariates_donor
# per cell type, with FDR within each (primary × CT pool).
#
# Outputs:
#   df_ols_by_spec   : dict {slug → DataFrame of per-CT contrasts}
#   df_ols           : concatenated across specs
#   results/donor_ols_{slug}.csv per spec
#   figures/donor_ols_{slug}.{pdf,png,svg} per spec
# =============================================================================

import statsmodels.formula.api as smf
import statsmodels.api as sm

print("=" * 80)
print(f"§3 — DONOR-LEVEL OLS  |  {DATASET}")
print("=" * 80)

# ─────────────────────────────────────────────────────────────────────────────
# Defensive harmonization
# ─────────────────────────────────────────────────────────────────────────────
if "CELLTYPE_RENAME" in dir() and CELLTYPE_RENAME:
    n_ren = int(df_donor_ct["Cell_Type"].isin(CELLTYPE_RENAME.keys()).sum())
    if n_ren > 0:
        print(f"  ⚠ Re-applying harmonization to df_donor_ct ({n_ren} rows)")
        df_donor_ct["Cell_Type"] = df_donor_ct["Cell_Type"].replace(CELLTYPE_RENAME)
_refresh_celltype_order()

# ─────────────────────────────────────────────────────────────────────────────
# Helper: format formula display
# ─────────────────────────────────────────────────────────────────────────────
def _format_formula(spec, covariates):
    primary = spec["var_donor"]
    if spec["type"] == "categorical":
        primary_term = f"C({primary}, Treatment('{spec['reference']}'))"
    else:
        primary_term = primary
    cov_terms = []
    for c in covariates:
        if c in {"Sex", "Cohort"}:
            cov_terms.append(f"C({c})")
        else:
            cov_terms.append(c)
    return f"prop_snc ~ {primary_term}" + (
        f" + {' + '.join(cov_terms)}" if cov_terms else ""
    )

# ─────────────────────────────────────────────────────────────────────────────
# Helper: extract contrasts from a fitted OLS model for a categorical primary
# Returns one row per non-reference level
# ─────────────────────────────────────────────────────────────────────────────
def _extract_categorical_contrasts(model, spec, n_cells, n_donors):
    primary = spec["var_donor"]
    reference = spec["reference"]
    rows = []
    # Coefficient names look like: C(primary, Treatment('ref'))[T.AD]
    for term in model.params.index:
        if term.startswith(f"C({primary}, Treatment(") and "[T." in term:
            level = term.split("[T.")[1].rstrip("]")
            beta = float(model.params[term])
            se   = float(model.bse[term])
            t_v  = float(model.tvalues[term])
            p_v  = float(model.pvalues[term])
            ci_l, ci_h = (
                float(model.conf_int().loc[term, 0]),
                float(model.conf_int().loc[term, 1]),
            )
            rows.append({
                "Contrast":  level,
                "Reference": reference,
                "N_cells":   int(n_cells),
                "N_donors":  int(n_donors),
                "Beta":      round(beta, 6),
                "SE":        round(se, 6),
                "CI_low":    round(ci_l, 6),
                "CI_high":   round(ci_h, 6),
                "T":         round(t_v, 4),
                "P_value":   round(p_v, 6),
            })
    return rows

# ─────────────────────────────────────────────────────────────────────────────
# §3.1 Iterate PRIMARY_SPECS × CTs
# ─────────────────────────────────────────────────────────────────────────────
df_ols_by_spec = {}

for spec in PRIMARY_SPECS:
    print(f"\n{'━'*64}")
    print(f"  Spec: {spec['label']}  (primary={spec['name']!r}, ref={spec['reference']!r})")
    print(f"{'━'*64}")

    # Resolve covariates against df_donor_ct columns + drop levelless categoricals
    cov_used = []
    for c in spec["covariates_donor"]:
        if c not in df_donor_ct.columns:
            continue
        if c in {"Sex", "Cohort"} and df_donor_ct[c].nunique() < 2:
            print(f"    ⊘ {c} has <2 levels — dropping from formula")
            continue
        cov_used.append(c)

    formula = _format_formula(spec, cov_used)
    print(f"\n  Formula: {formula}")
    print(f"  Covariates used: {cov_used}")
    print(f"  FDR scope: per-CT pool within this spec")

    primary = spec["var_donor"]

    # Per-CT model fit
    spec_rows = []
    for ct in CELLTYPE_ORDER_PLOT:
        df_ct = df_donor_ct[df_donor_ct["Cell_Type"] == ct].copy()

        # Drop NA on essential cols
        essential = ["prop_snc", primary] + cov_used
        df_ct = df_ct.dropna(subset=[c for c in essential if c in df_ct.columns])

        n_donors = df_ct["Donor"].nunique()
        n_cells  = int(df_ct["n_cells"].sum())

        if n_donors < 5:
            print(f"    {ct:<22} ⊘ skipped ({n_donors} donors)")
            continue

        # For categorical primary: ensure ≥2 levels present in subset, including ref
        if spec["type"] == "categorical":
            if df_ct[primary].nunique() < 2:
                print(f"    {ct:<22} ⊘ skipped (only 1 level of {primary})")
                continue
            if spec["reference"] not in df_ct[primary].unique():
                print(f"    {ct:<22} ⊘ skipped (ref {spec['reference']!r} absent)")
                continue

        try:
            model = smf.ols(formula, data=df_ct).fit()
        except Exception as e:
            print(f"    {ct:<22} ✗ fit failed: {str(e)[:60]}")
            continue

        if spec["type"] == "categorical":
            ct_rows = _extract_categorical_contrasts(model, spec, n_cells, n_donors)
        else:
            # Continuous slope
            term = primary
            beta = float(model.params[term])
            se   = float(model.bse[term])
            t_v  = float(model.tvalues[term])
            p_v  = float(model.pvalues[term])
            ci_l, ci_h = (
                float(model.conf_int().loc[term, 0]),
                float(model.conf_int().loc[term, 1]),
            )
            ct_rows = [{
                "Contrast":  primary,
                "Reference": None,
                "N_cells":   n_cells,
                "N_donors":  n_donors,
                "Beta":      round(beta, 6),
                "SE":        round(se, 6),
                "CI_low":    round(ci_l, 6),
                "CI_high":   round(ci_h, 6),
                "T":         round(t_v, 4),
                "P_value":   round(p_v, 6),
            }]

        for r in ct_rows:
            r["Cell_Type"] = ct
            r["Primary"] = spec["name"]
            spec_rows.append(r)

        # Console line per-CT
        for r in ct_rows:
            print(f"    {ct:<22}  {r['Contrast']:<14} "
                  f"β={r['Beta']:>+7.4f}  p={fmt_p(r['P_value'])}  "
                  f"({r['N_donors']} donors)")

    if not spec_rows:
        print(f"    ⊘ No models fit for {spec['name']!r}")
        continue

    df_spec = pd.DataFrame(spec_rows)

    # FDR within spec (across all CT × Contrast)
    df_spec["FDR"] = bh_correction(df_spec["P_value"].values).round(6)
    df_spec["stars"] = df_spec["FDR"].apply(sig_stars)
    df_spec["Significant"] = df_spec["FDR"] < FDR_THRESHOLD

    df_spec = df_spec[[
        "Cell_Type", "Primary", "Contrast", "Reference",
        "N_cells", "N_donors",
        "Beta", "SE", "CI_low", "CI_high", "T", "P_value", "FDR", "stars",
        "Significant",
    ]].sort_values(["Cell_Type", "Contrast"]).reset_index(drop=True)

    df_ols_by_spec[spec["slug"]] = df_spec

    save_table(df_spec, f"donor_ols_{spec['slug']}")

# Concatenated all-spec table
if df_ols_by_spec:
    df_ols = pd.concat(df_ols_by_spec.values(), ignore_index=True)
else:
    df_ols = pd.DataFrame()

# ─────────────────────────────────────────────────────────────────────────────
# §3.2 Plotting helper (reads spec["color_dispatch"])
# ─────────────────────────────────────────────────────────────────────────────
def _plot_donor_per_ct(spec, df_results, df_data, model_label="OLS",
                          slug_prefix="donor_ols"):
    """Per-CT panel grid. Categorical: boxplot + jitter with model annotation.
    Continuous: scatter + fit line.
    Reads spec['color_dispatch'] for level colors — no inline branches."""
    primary = spec["var_donor"]
    palette = spec.get("color_dispatch", {})
    is_cat  = spec["type"] == "categorical"

    cts_with_data = [
        ct for ct in CELLTYPE_ORDER_PLOT
        if ct in df_results["Cell_Type"].values
    ]
    n_cts = len(cts_with_data)
    if n_cts == 0:
        print(f"    ⊘ No CTs to plot for {spec['name']!r}")
        return

    n_cols = 3
    n_rows = int(np.ceil(n_cts / n_cols))
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(min(11.0, 3.0 * n_cols), 2.6 * n_rows),
        constrained_layout=True,
    )
    axes = np.atleast_1d(axes).flatten()

    for idx, ct in enumerate(cts_with_data):
        ax = axes[idx]
        sub = df_data[df_data["Cell_Type"] == ct].dropna(subset=[primary, "prop_snc"])
        ct_color = LINEAGE_COLORS.get(ct, "#666666") if "LINEAGE_COLORS" in dir() else "#444"

        if is_cat:
            # Boxplot + jitter, ordered by level_order (reference first)
            order = [spec["reference"]] + spec["level_order"]
            order = [lv for lv in order if lv in sub[primary].unique()]
            box_data = [sub[sub[primary] == lv]["prop_snc"].values * 100 for lv in order]
            bp = ax.boxplot(
                box_data, positions=range(len(order)), widths=0.55,
                patch_artist=True, showfliers=False,
                medianprops=dict(color="white", linewidth=1.2),
                whiskerprops=dict(color="#333", linewidth=0.6),
                capprops=dict(color="#333", linewidth=0.6),
            )
            for j, (lv, patch) in enumerate(zip(order, bp["boxes"])):
                color = palette.get(lv, "#888888")
                patch.set_facecolor(color)
                patch.set_alpha(0.85)
                patch.set_edgecolor("#333")
                patch.set_linewidth(0.6)
                # Jitter individual donors
                vals = box_data[j]
                jitter = (np.random.RandomState(42 + j).rand(len(vals)) - 0.5) * 0.35
                ax.scatter(j + jitter, vals,
                              s=8, color=color, alpha=0.55,
                              edgecolor="white", linewidth=0.3, zorder=3)

            ax.set_xticks(range(len(order)))
            ax.set_xticklabels(order, fontsize=7.5, rotation=90, ha="center")
            ax.set_xlim(-0.6, len(order) - 0.4)
        else:
            # Continuous: scatter + simple linear fit overlay
            x = sub[primary].values
            y = sub["prop_snc"].values * 100
            ax.scatter(x, y, s=10, color=ct_color, alpha=0.55,
                          edgecolor="white", linewidth=0.3, zorder=3)
            if len(x) >= 3 and np.std(x) > 0:
                fit = np.polyfit(x, y, 1)
                xs = np.linspace(x.min(), x.max(), 60)
                ax.plot(xs, fit[0] * xs + fit[1],
                          color=ct_color, linewidth=1.2, alpha=0.8, zorder=2)
            ax.set_xlabel(primary, fontsize=7.5)

        # Stats annotation (top-left)
        rows = df_results[df_results["Cell_Type"] == ct]
        # Show first contrast (or aggregate if multiple)
        if len(rows) > 0:
            r = rows.iloc[0]
            sig = bool(r.get("Significant", False))
            ann_color = ct_color if sig else "#888"
            ax.text(
                0.04, 0.97,
                f"β = {r['Beta']:+.3f}\n"
                f"p = {fmt_p(r['P_value'])}\n"
                f"FDR = {fmt_p(r['FDR'])}",
                transform=ax.transAxes,
                fontsize=6.5, va="top", ha="left",
                color=ann_color, linespacing=1.4,
            )

        ax.set_title(ct, fontsize=9, fontweight="500", pad=4, color=ct_color)
        ax.set_ylabel("% SnC", fontsize=7.5)
        ax.tick_params(axis="y", labelsize=7)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    for idx in range(n_cts, len(axes)):
        axes[idx].set_visible(False)

    fig.suptitle(
        f"Donor-level {model_label} — {spec['label']}  ·  {DATASET}",
        fontsize=10, fontweight="500", y=1.01,
    )
    save_figure(fig, f"{slug_prefix}_{spec['slug']}")
    plt.show()

# ─────────────────────────────────────────────────────────────────────────────
# §3.3 Plot per spec
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*64}")
print(f"  §3 PLOTS")
print(f"{'─'*64}")

for spec in PRIMARY_SPECS:
    if spec["slug"] not in df_ols_by_spec:
        continue
    df_results = df_ols_by_spec[spec["slug"]]
    print(f"\n  ▸ Plotting {spec['name']!r}")
    _plot_donor_per_ct(spec, df_results, df_donor_ct,
                            model_label="OLS", slug_prefix="donor_ols")

# ─────────────────────────────────────────────────────────────────────────────
# §3.4 Summary
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*64}")
print(f"  §3 SUMMARY")
print(f"{'─'*64}")
for slug, df_s in df_ols_by_spec.items():
    n_models = df_s["Cell_Type"].nunique()
    n_sig = int(df_s["Significant"].sum())
    print(f"  {slug:<22}  {n_models:2d} CTs · {n_sig:2d} sig contrasts (FDR < {FDR_THRESHOLD})")

print(f"\n✓ §3 donor-level OLS complete")
print(f"  Tables  : donor_ols_{{slug}}.csv per spec")
print(f"  Figures : donor_ols_{{slug}}.{{pdf,png,svg}} per spec")
print(f"  Next    : §4 donor-level RLR (Huber M-estimator, robust to outlier donors)")

---
## 24 · Group contrast — donor-level RLR

**Why.** The same design under a Huber M-estimator.

**Watch the small cell types.** The guard admits any type with 5 or more donors, but the formula carries one dummy per non-reference level plus three covariates. A type with 8 donors is fit with roughly as many parameters as observations, and the resulting p-values are not interpretable — compare them against the OLS p-values for the same contrast before reporting anything.

In [ ]:
# =============================================================================
# §4 — DONOR-LEVEL RLR (Huber M-estimator)
# =============================================================================
# Same design as §3 but uses statsmodels.RLM with HuberT — robust to outlier
# donors (down-weights extreme observations rather than throwing them out).
#
# Outputs:
#   df_rlr_by_spec  : dict {slug → DataFrame}
#   df_rlr          : concatenated
#   df_concordance  : OLS↔RLR concordance summary with classification
#   results/donor_rlr_{slug}.csv per spec
#   results/donor_ols_rlr_concordance.csv
#   figures/donor_rlr_{slug}.{pdf,png,svg} per spec
# =============================================================================

print("=" * 80)
print(f"§4 — DONOR-LEVEL RLR  |  {DATASET}")
print("=" * 80)

# ─────────────────────────────────────────────────────────────────────────────
# Defensive harmonization
# ─────────────────────────────────────────────────────────────────────────────
if "CELLTYPE_RENAME" in dir() and CELLTYPE_RENAME:
    n_ren = int(df_donor_ct["Cell_Type"].isin(CELLTYPE_RENAME.keys()).sum())
    if n_ren > 0:
        print(f"  ⚠ Re-applying harmonization to df_donor_ct ({n_ren} rows)")
        df_donor_ct["Cell_Type"] = df_donor_ct["Cell_Type"].replace(CELLTYPE_RENAME)
_refresh_celltype_order()

# ─────────────────────────────────────────────────────────────────────────────
# RLR contrast extractor (mirrors _extract_categorical_contrasts but reads
# from RLM result objects which expose tvalues, pvalues, params, bse)
# ─────────────────────────────────────────────────────────────────────────────
def _extract_categorical_contrasts_rlm(result, spec, n_cells, n_donors):
    primary = spec["var_donor"]
    reference = spec["reference"]
    rows = []
    for term in result.params.index:
        if term.startswith(f"C({primary}, Treatment(") and "[T." in term:
            level = term.split("[T.")[1].rstrip("]")
            beta = float(result.params[term])
            se   = float(result.bse[term])
            t_v  = float(result.tvalues[term])
            p_v  = float(result.pvalues[term])
            # RLM doesn't expose conf_int by default — compute Wald 95% CI
            ci_l, ci_h = beta - 1.96 * se, beta + 1.96 * se
            rows.append({
                "Contrast":  level,
                "Reference": reference,
                "N_cells":   int(n_cells),
                "N_donors":  int(n_donors),
                "Beta":      round(beta, 6),
                "SE":        round(se, 6),
                "CI_low":    round(ci_l, 6),
                "CI_high":   round(ci_h, 6),
                "T":         round(t_v, 4),
                "P_value":   round(p_v, 6),
            })
    return rows

# ─────────────────────────────────────────────────────────────────────────────
# §4.1 Iterate PRIMARY_SPECS × CTs
# ─────────────────────────────────────────────────────────────────────────────
df_rlr_by_spec = {}

for spec in PRIMARY_SPECS:
    print(f"\n{'━'*64}")
    print(f"  Spec: {spec['label']}  (primary={spec['name']!r}, ref={spec['reference']!r})")
    print(f"{'━'*64}")

    cov_used = []
    for c in spec["covariates_donor"]:
        if c not in df_donor_ct.columns:
            continue
        if c in {"Sex", "Cohort"} and df_donor_ct[c].nunique() < 2:
            continue
        cov_used.append(c)

    formula = _format_formula(spec, cov_used)
    print(f"\n  Formula: {formula}")
    print(f"  Estimator: Huber M-estimator (statsmodels.rlm)")
    print(f"  Covariates used: {cov_used}")

    primary = spec["var_donor"]

    spec_rows = []
    for ct in CELLTYPE_ORDER_PLOT:
        df_ct = df_donor_ct[df_donor_ct["Cell_Type"] == ct].copy()
        essential = ["prop_snc", primary] + cov_used
        df_ct = df_ct.dropna(subset=[c for c in essential if c in df_ct.columns])

        n_donors = df_ct["Donor"].nunique()
        n_cells  = int(df_ct["n_cells"].sum())

        if n_donors < 5:
            print(f"    {ct:<22} ⊘ skipped ({n_donors} donors)")
            continue

        if spec["type"] == "categorical":
            if df_ct[primary].nunique() < 2:
                print(f"    {ct:<22} ⊘ skipped (only 1 level of {primary})")
                continue
            if spec["reference"] not in df_ct[primary].unique():
                print(f"    {ct:<22} ⊘ skipped (ref {spec['reference']!r} absent)")
                continue

        try:
            result = smf.rlm(
                formula, data=df_ct, M=sm.robust.norms.HuberT()
            ).fit()
        except Exception as e:
            print(f"    {ct:<22} ✗ fit failed: {str(e)[:60]}")
            continue

        if spec["type"] == "categorical":
            ct_rows = _extract_categorical_contrasts_rlm(
                result, spec, n_cells, n_donors
            )
        else:
            term = primary
            beta = float(result.params[term])
            se   = float(result.bse[term])
            t_v  = float(result.tvalues[term])
            p_v  = float(result.pvalues[term])
            ci_l, ci_h = beta - 1.96 * se, beta + 1.96 * se
            ct_rows = [{
                "Contrast":  primary,
                "Reference": None,
                "N_cells":   n_cells,
                "N_donors":  n_donors,
                "Beta":      round(beta, 6),
                "SE":        round(se, 6),
                "CI_low":    round(ci_l, 6),
                "CI_high":   round(ci_h, 6),
                "T":         round(t_v, 4),
                "P_value":   round(p_v, 6),
            }]

        for r in ct_rows:
            r["Cell_Type"] = ct
            r["Primary"] = spec["name"]
            spec_rows.append(r)

        for r in ct_rows:
            print(f"    {ct:<22}  {r['Contrast']:<14} "
                  f"β={r['Beta']:>+7.4f}  p={fmt_p(r['P_value'])}  "
                  f"({r['N_donors']} donors)")

    if not spec_rows:
        print(f"    ⊘ No models fit for {spec['name']!r}")
        continue

    df_spec = pd.DataFrame(spec_rows)
    df_spec["FDR"] = bh_correction(df_spec["P_value"].values).round(6)
    df_spec["stars"] = df_spec["FDR"].apply(sig_stars)
    df_spec["Significant"] = df_spec["FDR"] < FDR_THRESHOLD

    df_spec = df_spec[[
        "Cell_Type", "Primary", "Contrast", "Reference",
        "N_cells", "N_donors",
        "Beta", "SE", "CI_low", "CI_high", "T", "P_value", "FDR", "stars",
        "Significant",
    ]].sort_values(["Cell_Type", "Contrast"]).reset_index(drop=True)

    df_rlr_by_spec[spec["slug"]] = df_spec
    save_table(df_spec, f"donor_rlr_{spec['slug']}")

if df_rlr_by_spec:
    df_rlr = pd.concat(df_rlr_by_spec.values(), ignore_index=True)
else:
    df_rlr = pd.DataFrame()

# ─────────────────────────────────────────────────────────────────────────────
# §4.2 Plot per spec (re-uses §3's _plot_donor_per_ct helper)
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*64}")
print(f"  §4 PLOTS")
print(f"{'─'*64}")

for spec in PRIMARY_SPECS:
    if spec["slug"] not in df_rlr_by_spec:
        continue
    df_results = df_rlr_by_spec[spec["slug"]]
    print(f"\n  ▸ Plotting {spec['name']!r}")
    _plot_donor_per_ct(spec, df_results, df_donor_ct,
                            model_label="RLR", slug_prefix="donor_rlr")

# ─────────────────────────────────────────────────────────────────────────────
# §4.3 OLS vs RLR concordance check (CLASSIFIED)
#
# Don't just print %diff — that's misleading near β=0 (ratios explode).
# Classify each (CT, Contrast) into a category that tells you what to trust:
#
#   concordant       : both βs same sign, magnitudes within 50% of each other
#                      → trust the effect, OLS and RLR agree
#   amplified        : same sign, RLR magnitude ≥ 1.5× OLS  (|RLR| > 1.5*|OLS|)
#                      → outliers were pulling OLS toward zero; RLR more reliable
#   shrunk           : same sign, RLR magnitude ≤ 0.67× OLS (|RLR| < 0.67*|OLS|)
#                      → outliers were inflating OLS; RLR more conservative
#   direction-flip   : opposite signs, AND both |β| ≥ NEAR_ZERO_β
#                      → outlier-driven; ESTIMATE UNRELIABLE
#   near-zero        : at least one |β| < NEAR_ZERO_β
#                      → effect is too small to interpret direction;
#                        report only as "no detected effect"
#
# Threshold rationale: NEAR_ZERO_β = 0.005 = 0.5 percentage points of %SnC.
# Below that, donor-level signal is in noise territory.
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*64}")
print(f"  OLS vs RLR concordance (per spec, classified)")
print(f"{'─'*64}")

NEAR_ZERO_β = 0.005
AMPLIFY_RATIO = 1.5
SHRINK_RATIO = 0.67

def _classify_concordance(b_ols, b_rlr):
    """Return (category, reason) given two β estimates."""
    abs_o = abs(b_ols)
    abs_r = abs(b_rlr)

    # Both effectively zero
    if abs_o < NEAR_ZERO_β and abs_r < NEAR_ZERO_β:
        return "near-zero", "both |β| < threshold"
    # One is near-zero
    if abs_o < NEAR_ZERO_β or abs_r < NEAR_ZERO_β:
        return "near-zero", "one |β| < threshold"
    # Both have meaningful magnitudes — check sign
    if np.sign(b_ols) != np.sign(b_rlr):
        return "direction-flip", "signs differ"
    # Same sign, both meaningful — check magnitude ratio
    ratio = abs_r / abs_o
    if ratio >= AMPLIFY_RATIO:
        return "amplified", f"RLR {ratio:.1f}× larger"
    if ratio <= SHRINK_RATIO:
        return "shrunk", f"RLR {ratio:.1f}× smaller"
    return "concordant", f"ratio {ratio:.2f}"

# Build classified concordance table
concordance_rows = []

for slug in df_rlr_by_spec:
    if slug not in df_ols_by_spec:
        continue

    df_o = df_ols_by_spec[slug][[
        "Cell_Type", "Contrast", "Beta", "P_value", "FDR", "Significant"
    ]].rename(columns={
        "Beta": "Beta_OLS",
        "P_value": "P_OLS",
        "FDR": "FDR_OLS",
        "Significant": "Sig_OLS",
    })
    df_r = df_rlr_by_spec[slug][[
        "Cell_Type", "Contrast", "Beta", "P_value", "FDR", "Significant"
    ]].rename(columns={
        "Beta": "Beta_RLR",
        "P_value": "P_RLR",
        "FDR": "FDR_RLR",
        "Significant": "Sig_RLR",
    })

    merged = df_o.merge(df_r, on=["Cell_Type", "Contrast"], how="inner")
    if len(merged) == 0:
        continue

    classifications = merged.apply(
        lambda r: _classify_concordance(r["Beta_OLS"], r["Beta_RLR"]),
        axis=1,
    )
    merged["Concordance"] = [c[0] for c in classifications]
    merged["Concordance_reason"] = [c[1] for c in classifications]
    merged["Beta_diff_abs"] = (merged["Beta_RLR"] - merged["Beta_OLS"]).round(6)
    merged["Sig_either"] = merged["Sig_OLS"] | merged["Sig_RLR"]
    merged["Spec"] = slug

    concordance_rows.append(merged)

    # Print per-spec concordance table
    print(f"\n  {slug}:")
    print(f"  {'Cell_Type':<22} {'Contrast':<12} "
          f"{'β OLS':>9} {'β RLR':>9} {'Δβ':>9} {'Sig':>6} "
          f"{'Concordance':<16} {'why':<22}")
    print(f"  {'─'*120}")

    # Sort: most informative first (concordant + amplified at top, near-zero at bottom)
    cat_priority = {
        "concordant":     0,
        "amplified":      1,
        "shrunk":         2,
        "direction-flip": 3,
        "near-zero":      4,
    }
    merged["_order"] = merged["Concordance"].map(cat_priority)
    merged_sorted = merged.sort_values(
        ["_order", "Beta_OLS"], key=lambda c: c.abs() if c.name == "Beta_OLS" else c,
        ascending=[True, False],
    )

    for _, row in merged_sorted.iterrows():
        sig_label = ""
        if row["Sig_OLS"] and row["Sig_RLR"]:
            sig_label = "★★"
        elif row["Sig_OLS"] or row["Sig_RLR"]:
            sig_label = "★"

        # Color hint via simple text marker
        if row["Concordance"] == "concordant":
            cat_marker = "✓"
        elif row["Concordance"] == "amplified":
            cat_marker = "↑"
        elif row["Concordance"] == "shrunk":
            cat_marker = "↓"
        elif row["Concordance"] == "direction-flip":
            cat_marker = "⚠"
        else:  # near-zero
            cat_marker = "·"

        print(f"  {row['Cell_Type']:<22} {row['Contrast']:<12} "
              f"{row['Beta_OLS']:>+9.4f} {row['Beta_RLR']:>+9.4f} "
              f"{row['Beta_diff_abs']:>+9.4f} {sig_label:>6} "
              f"{cat_marker} {row['Concordance']:<14} {row['Concordance_reason']:<22}")

# ─────────────────────────────────────────────────────────────────────────────
# §4.3.2 Aggregate concordance summary across all specs
# ─────────────────────────────────────────────────────────────────────────────
if concordance_rows:
    df_concordance = pd.concat(concordance_rows, ignore_index=True)
    save_table(df_concordance, "donor_ols_rlr_concordance")

    # Cross-spec summary
    print(f"\n{'─'*64}")
    print(f"  CONCORDANCE SUMMARY  (across all PRIMARY_SPECS)")
    print(f"{'─'*64}")
    cat_counts = df_concordance["Concordance"].value_counts()
    total = len(df_concordance)
    for cat in ["concordant", "amplified", "shrunk", "direction-flip", "near-zero"]:
        n = int(cat_counts.get(cat, 0))
        marker = {
            "concordant":     "✓",
            "amplified":      "↑",
            "shrunk":         "↓",
            "direction-flip": "⚠",
            "near-zero":      "·",
        }[cat]
        print(f"    {marker} {cat:<16}  {n:>3} / {total}  ({100*n/total:.1f}%)")

    # Per-CT summary: which CTs have signal both methods agree on?
    print(f"\n  Cell types where OLS and RLR concordance is 'concordant' or 'amplified':")
    trustworthy = df_concordance[
        df_concordance["Concordance"].isin(["concordant", "amplified"])
    ]
    for _, row in trustworthy.iterrows():
        sig_either = row["Sig_OLS"] or row["Sig_RLR"]
        sig_str = "★ FDR<0.05 in at least one" if sig_either else ""
        print(f"    {row['Cell_Type']:<22} {row['Contrast']:<12} "
              f"β_OLS={row['Beta_OLS']:>+7.4f}  "
              f"β_RLR={row['Beta_RLR']:>+7.4f}  {sig_str}")
else:
    df_concordance = pd.DataFrame()

# ─────────────────────────────────────────────────────────────────────────────
# §4.4 Summary
# ─────────────────────────────────────────────────────────────────────────────
print(f"\n{'─'*64}")
print(f"  §4 SUMMARY")
print(f"{'─'*64}")
for slug, df_s in df_rlr_by_spec.items():
    n_models = df_s["Cell_Type"].nunique()
    n_sig = int(df_s["Significant"].sum())
    print(f"  {slug:<22}  {n_models:2d} CTs · {n_sig:2d} sig contrasts (FDR < {FDR_THRESHOLD})")

print(f"\n✓ §4 donor-level RLR complete")
print(f"  Tables  : donor_rlr_{{slug}}.csv per spec")
print(f"           donor_ols_rlr_concordance.csv (classified)")
print(f"  Figures : donor_rlr_{{slug}}.{{pdf,png,svg}} per spec")
print(f"  Next    : §5 cell-level GLMM via lme4")
print(f"\n  How to read concordance:")
print(f"    ✓ concordant     — trust the effect")
print(f"    ↑ amplified      — RLR sees stronger signal; outliers were pulling OLS to 0")
print(f"    ↓ shrunk         — RLR sees weaker signal; outliers were inflating OLS")
print(f"    ⚠ direction-flip — outlier-driven, estimate unreliable")
print(f"    · near-zero      — too small to interpret direction")

---
## 25 · Forest plot helpers

**Why.** `_render_forest` — the table-style forest used by sections 27 and by the per-decade plots. Defined once here so the panels stay identical.

In [ ]:
# =============================================================================
# §5a — FOREST PLOT HELPERS  (R/ggplot table-style, matches user's reference)
# Shared by §5b GLMM / §5c Sloan / §5d cell cycle / §7b/c pathology
# =============================================================================
#
# Layout — 3 panels (4 when extra_col_* provided):
#   LEFT TABLE     (3 sub-columns)  : ROW_LABEL  |  OR/d  |  95% CI
#   FOREST         (axis)           : vline at null + errorbars + diamonds
#   RIGHT TABLE    (2 sub-columns)  : p(adj)     |  Sig
#
# Style notes (from R reference):
#   - Table headers: bold + horizontal underline segment beneath
#   - Tiny fonts (publication-compact)
#   - Filled diamond markers, sized smaller than v1
#   - Errorbar caps on each end
#   - Dashed vertical line at null value (OR=1 / d=0)
#   - pretty x-axis breaks (~4 ticks)
#   - Optional tag letter ("B") in upper-left for multi-panel figures
# =============================================================================

print("=" * 64)
print(f"§5a — FOREST PLOT HELPERS  |  {DATASET}")
print("=" * 64)

import re
from matplotlib.lines import Line2D
from matplotlib.colors import to_rgba
from matplotlib.ticker import MaxNLocator

# Defensive harmonization
if "CELLTYPE_RENAME" in dir() and CELLTYPE_RENAME:
    n_ren = int(df_cells_model["Cell_Type"].isin(CELLTYPE_RENAME.keys()).sum())
    if n_ren > 0:
        print(f"  ⚠ Re-applying harmonization ({n_ren} cells)")
        df_cells_model["Cell_Type"] = df_cells_model["Cell_Type"].replace(CELLTYPE_RENAME)
_refresh_celltype_order()

# ─────────────────────────────────────────────────────────────────────────────
# Utilities
# ─────────────────────────────────────────────────────────────────────────────
def _slugify(s):
    s = re.sub(r"[^A-Za-z0-9]+", "_", str(s)).strip("_").lower()
    return s or "unnamed"

def _truncate_label(s, max_chars=28):
    s = str(s)
    if len(s) <= max_chars:
        return s
    return s[:max_chars - 1] + "…"

def _fmt_p_short(p):
    if pd.isna(p):
        return "—"
    if p < 0.001:
        return f"{p:.1e}"
    if p < 0.01:
        return f"{p:.3f}"
    return f"{p:.3f}"

def _lighten(hex_color, amount=0.55):
    r, g, b, _ = to_rgba(hex_color)
    return (r, g, b, amount)

# Canonical row order (OVERALL pinned to top)
CANONICAL_CT_ORDER_BY_TISSUE = {
    "brain": [
        "Excitatory", "Inhibitory",
        "Astrocyte", "Oligodendrocyte", "Microglia", "OPC",
        "Endothelial", "Pericyte",
    ],
    "pbmc": [
        "cd4t", "cd8t", "unconvT", "nkcell",
        "cd14mono", "cd16mono", "memB", "naiveB",
    ],
    "csf": [
        "cd4t", "cd8t", "nkcell",
        "monocyte", "bcell", "dc",
    ],
}
canonical_order = CANONICAL_CT_ORDER_BY_TISSUE.get(
    TISSUE, list(LINEAGE_CONFIGS.keys()),
)

def _ct_sort_key(ct):
    if ct == "OVERALL":
        return -1
    try:
        return canonical_order.index(ct)
    except ValueError:
        return 999

# ─────────────────────────────────────────────────────────────────────────────
# Effect configs
# ─────────────────────────────────────────────────────────────────────────────
EFFECT_LOG_OR = {
    "scale":      "log",
    "null_value": 1.0,                       # OR=1 = no effect
    "xlabel":     "Odds Ratio",
    "fmt_effect": lambda v: f"{v:.3f}",
    "fmt_ci":     lambda lo, hi: f"[{lo:.3f}, {hi:.3f}]",
    "axis_format":"or",                       # for tick formatting
}

EFFECT_LINEAR_D = {
    "scale":      "linear",
    "null_value": 0.0,
    "xlabel":     "Cohen's d (sen vs non-sen)",
    "fmt_effect": lambda v: f"{v:+.3f}",
    "fmt_ci":     lambda lo, hi: f"[{lo:+.3f}, {hi:+.3f}]",
    "axis_format":"d",
}

EFFECT_LOG_OR_SEN = {
    **EFFECT_LOG_OR,
    "xlabel": "Odds Ratio (sen vs non-sen)",
}

# ─────────────────────────────────────────────────────────────────────────────
# Row color functions
# ─────────────────────────────────────────────────────────────────────────────
def lineage_row_color_fn(row_entity):
    if row_entity == "OVERALL":
        return "#222222"
    return LINEAGE_COLORS.get(row_entity, "#666666")

def constant_row_color_fn(color_hex):
    def _fn(row_entity):
        return color_hex
    return _fn

def phase_row_color_fn(row_entity):
    PHASE_COLORS = {"G1": "#4E79A7", "S": "#F28E2B", "G2M": "#E15759"}
    return PHASE_COLORS.get(row_entity, "#888")

# ─────────────────────────────────────────────────────────────────────────────
# CORE: table-style forest plot (matches R/ggplot reference)
# ─────────────────────────────────────────────────────────────────────────────
def _render_forest(df, *, title, slug,
                       subtitle=None,
                       footnote=None,
                       header_label="Cell Type",
                       extra_col_label=None,
                       extra_col_values=None,
                       extra_col_color="#444",
                       effect_config=EFFECT_LOG_OR,
                       row_color_fn=lineage_row_color_fn,
                       row_entity_col="Cell_Type",
                       row_label_col="row_label",
                       effect_col="OR",
                       ci_lo_col="CI_low",
                       ci_hi_col="CI_high",
                       overall_label="OVERALL",
                       sep_after_overall=True,
                       fig_tag=None,
                       fig_w=8.0,
                       row_h=0.32,
                       max_label_chars=28):
    """
    Table-style forest plot.

    Layout:
        LEFT TABLE (label | effect | CI)  |  FOREST  |  RIGHT TABLE (p_adj | Sig)
        + optional INTERP column between LEFT and FOREST when extra_col_* given.

    df must contain:
        row_label_col, row_entity_col,
        effect_col, ci_lo_col, ci_hi_col,
        "P_value", "FDR" (NaN ok), "Significant" (bool), "stars" (str)
    Optional columns:
        Contrast_type ("vs_reference"|"adjacent")  → adjacent rows indented
        Converged (bool)                            → ↓ alpha if False
    """
    n = len(df)
    if n == 0:
        print(f"  ⊘ {slug}: no rows")
        return
    df = df.reset_index(drop=True).copy()

    has_extra = (extra_col_label is not None
                    and extra_col_values is not None
                    and len(extra_col_values) == n)

    # Compute axis limits in data space (linear OR even for log scale — we set
    # the axis to log scale at render time, but compute the range linearly so
    # `pretty` selects nice OR breaks like 0.5, 1.0, 1.5)
    eff_vals = df[effect_col].dropna().values
    ci_lo_vals = df[ci_lo_col].dropna().values
    ci_hi_vals = df[ci_hi_col].dropna().values
    all_vals = np.concatenate([eff_vals, ci_lo_vals, ci_hi_vals])
    if len(all_vals) == 0:
        print(f"  ⊘ {slug}: no finite effect values")
        return

    is_log = (effect_config["scale"] == "log")
    null_v = effect_config["null_value"]

    if is_log:
        # add log-padding
        log_vals = np.log(np.clip(all_vals, 1e-6, 1e6))
        lo, hi = float(log_vals.min()), float(log_vals.max())
        pad = max((hi - lo) * 0.15, 0.15)
        x_lo, x_hi = float(np.exp(lo - pad)), float(np.exp(hi + pad))
        # Ensure null is visible
        x_lo = min(x_lo, null_v / 1.05)
        x_hi = max(x_hi, null_v * 1.05)
    else:
        lo, hi = float(all_vals.min()), float(all_vals.max())
        pad = max((hi - lo) * 0.15, 0.05)
        x_lo, x_hi = lo - pad, hi + pad
        x_lo = min(x_lo, null_v - 0.05)
        x_hi = max(x_hi, null_v + 0.05)

    # ── Figure geometry ──────────────────────────────────────────────────────
    fig_h = max(1.6, 0.65 + n * row_h + (0.3 if subtitle else 0))

    if has_extra:
        # 4 panels: label-table | extra | forest | p-table
        ratios = [3.0, 1.4, 3.6, 1.6]
        n_cols = 4
    else:
        # 3 panels: label-table | forest | p-table
        ratios = [3.0, 3.6, 1.6]
        n_cols = 3

    fig = plt.figure(figsize=(fig_w, fig_h))
    gs = fig.add_gridspec(
        1, n_cols,
        width_ratios=ratios,
        wspace=0.04,
    )

    if has_extra:
        ax_L  = fig.add_subplot(gs[0, 0])
        ax_M  = fig.add_subplot(gs[0, 1])
        ax_F  = fig.add_subplot(gs[0, 2])
        ax_R  = fig.add_subplot(gs[0, 3])
    else:
        ax_L  = fig.add_subplot(gs[0, 0])
        ax_F  = fig.add_subplot(gs[0, 1])
        ax_R  = fig.add_subplot(gs[0, 2])
        ax_M  = None

    # Y positions, top→bottom
    y_pos = list(range(n - 1, -1, -1))
    df["_y"] = y_pos

    # Header row sits ABOVE all panels (at y = n)
    header_y    = n + 0.5
    underline_y = n + 0.05
    y_lim_lo, y_lim_hi = -0.6, n + 1.1

    # ─────────────────────────────────────────────────────────────────────────
    # LEFT TABLE: label | effect | 95% CI
    # ─────────────────────────────────────────────────────────────────────────
    ax_L.set_xlim(0, 1)
    ax_L.set_ylim(y_lim_lo, y_lim_hi)
    ax_L.set_xticks([]); ax_L.set_yticks([])
    for sp in ax_L.spines.values():
        sp.set_visible(False)

    # Sub-column x positions within LEFT panel
    x_label = 0.02
    x_eff   = 0.62
    x_ci    = 0.78
    eff_h_label = "OR" if is_log else "d"

    # Headers
    ax_L.text(x_label, header_y, header_label,
                 fontsize=7.0, fontweight="bold",
                 ha="left", va="center", color="#222")
    ax_L.text(x_eff, header_y, eff_h_label,
                 fontsize=7.0, fontweight="bold",
                 ha="left", va="center", color="#222")
    ax_L.text(x_ci, header_y, "95% CI",
                 fontsize=6.5, fontweight="bold",
                 ha="left", va="center", color="#222")
    # Header underline
    ax_L.plot([0.0, 1.0], [underline_y, underline_y],
                color="#333", linewidth=0.5,
                clip_on=False, zorder=10)

    # Optional tag (e.g. "A", "B")
    if fig_tag:
        ax_L.text(-0.02, y_lim_hi - 0.05, fig_tag,
                     fontsize=10.0, fontweight="bold",
                     ha="left", va="top", color="#111")

    # Rows
    for _, r in df.iterrows():
        yi    = r["_y"]
        rc    = row_color_fn(r[row_entity_col])
        is_sig = bool(r.get("Significant", False))
        is_overall = (str(r[row_entity_col]) == overall_label)

        lbl = _truncate_label(r[row_label_col], max_chars=max_label_chars)
        if "Contrast_type" in r.index and r["Contrast_type"] == "adjacent":
            lbl = f"  {lbl}"

        ax_L.text(x_label, yi, lbl,
                     fontsize=6.0, fontweight="bold" if (is_sig or is_overall) else "normal",
                     ha="left", va="center", color=rc)

        eff_v = r[effect_col]
        eff_str = "—" if pd.isna(eff_v) else effect_config["fmt_effect"](eff_v)
        ax_L.text(x_eff, yi, eff_str,
                     fontsize=5.8,
                     fontweight="bold" if is_sig else "normal",
                     ha="left", va="center", color="#222",
                     family="DejaVu Sans Mono")

        ci_lo_v = r[ci_lo_col]
        ci_hi_v = r[ci_hi_col]
        if pd.isna(ci_lo_v) or pd.isna(ci_hi_v):
            ci_str = "—"
        else:
            ci_str = effect_config["fmt_ci"](ci_lo_v, ci_hi_v)
        ax_L.text(x_ci, yi, ci_str,
                     fontsize=5.5,
                     ha="left", va="center", color="#666",
                     family="DejaVu Sans Mono")

    # ─────────────────────────────────────────────────────────────────────────
    # OPTIONAL EXTRA COLUMN (e.g., interpretation label)
    # ─────────────────────────────────────────────────────────────────────────
    if has_extra:
        ax_M.set_xlim(0, 1)
        ax_M.set_ylim(y_lim_lo, y_lim_hi)
        ax_M.set_xticks([]); ax_M.set_yticks([])
        for sp in ax_M.spines.values():
            sp.set_visible(False)
        ax_M.text(0.02, header_y, extra_col_label,
                     fontsize=6.5, fontweight="bold",
                     ha="left", va="center", color="#222")
        ax_M.plot([0.0, 1.0], [underline_y, underline_y],
                     color="#333", linewidth=0.5,
                     clip_on=False, zorder=10)
        for i, val in enumerate(extra_col_values):
            yi = df.iloc[i]["_y"]
            ax_M.text(0.02, yi, str(val),
                         fontsize=5.5, ha="left", va="center",
                         color=extra_col_color,
                         family="DejaVu Sans Mono")

    # ─────────────────────────────────────────────────────────────────────────
    # FOREST PANEL
    # ─────────────────────────────────────────────────────────────────────────
    if is_log:
        ax_F.set_xscale("log")
    ax_F.set_xlim(x_lo, x_hi)
    ax_F.set_ylim(y_lim_lo, y_lim_hi)

    # Vertical reference line at null
    ax_F.axvline(null_v, color="#999", linewidth=0.5,
                    linestyle=(0, (3, 2)), zorder=1)

    for _, r in df.iterrows():
        yi = r["_y"]
        is_overall = (str(r[row_entity_col]) == overall_label)
        is_sig     = bool(r.get("Significant", False))
        rc         = row_color_fn(r[row_entity_col])

        alpha_main = 1.0 if (is_sig or is_overall) else 0.65
        if "Converged" in r.index and not bool(r.get("Converged", True)):
            alpha_main = 0.35

        eff_v = r[effect_col]
        ci_lo_v = r[ci_lo_col]
        ci_hi_v = r[ci_hi_col]
        if pd.isna(eff_v):
            continue

        # Errorbar: horizontal line + small vertical caps
        if not pd.isna(ci_lo_v) and not pd.isna(ci_hi_v):
            ax_F.plot([ci_lo_v, ci_hi_v], [yi, yi],
                          color="#4d4d4d",
                          linewidth=0.6, alpha=alpha_main,
                          solid_capstyle="butt", zorder=2)
            # Caps
            cap_h = 0.18
            ax_F.plot([ci_lo_v, ci_lo_v], [yi - cap_h, yi + cap_h],
                          color="#4d4d4d", linewidth=0.6,
                          alpha=alpha_main, zorder=2)
            ax_F.plot([ci_hi_v, ci_hi_v], [yi - cap_h, yi + cap_h],
                          color="#4d4d4d", linewidth=0.6,
                          alpha=alpha_main, zorder=2)

        # Diamond marker (filled, ggplot shape=18)
        marker_size = 38 if is_overall else 28
        ax_F.scatter([eff_v], [yi],
                          marker="D", s=marker_size,
                          facecolor=rc,
                          edgecolor="#222",
                          linewidths=0.4,
                          alpha=alpha_main,
                          zorder=4)

    # X-axis ticks: pretty breaks
    if is_log:
        from matplotlib.ticker import LogLocator, FuncFormatter
        # Pick clean OR ticks within visible range
        candidates = [0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 5.0, 10.0, 20.0]
        ticks = [t for t in candidates if x_lo * 0.95 <= t <= x_hi * 1.05]
        if len(ticks) < 3:
            ticks = candidates
        if len(ticks) > 6:
            # decimate
            ticks = ticks[::2]
        ax_F.set_xticks(ticks)
        def _fmt(t, pos):
            if abs(t - 1.0) < 1e-6:
                return "1"
            if t >= 10:
                return f"{t:.0f}"
            return f"{t:g}"
        ax_F.xaxis.set_major_formatter(FuncFormatter(_fmt))
    else:
        ax_F.xaxis.set_major_locator(MaxNLocator(nbins=4))

    ax_F.set_xlabel(effect_config["xlabel"],
                       fontsize=6.5, labelpad=2)
    ax_F.tick_params(axis="x", labelsize=5.5, length=2, pad=1)
    ax_F.set_yticks([])
    for sp in ["top", "right", "left"]:
        ax_F.spines[sp].set_visible(False)
    ax_F.spines["bottom"].set_color("#444")
    ax_F.spines["bottom"].set_linewidth(0.5)
    ax_F.grid(False)

    # ─────────────────────────────────────────────────────────────────────────
    # RIGHT TABLE: p(adj) | Sig
    # ─────────────────────────────────────────────────────────────────────────
    ax_R.set_xlim(0, 1)
    ax_R.set_ylim(y_lim_lo, y_lim_hi)
    ax_R.set_xticks([]); ax_R.set_yticks([])
    for sp in ax_R.spines.values():
        sp.set_visible(False)

    x_p = 0.20
    x_s = 0.78

    ax_R.text(x_p, header_y, "p(adj)",
                 fontsize=7.0, fontweight="bold",
                 ha="center", va="center", color="#222")
    ax_R.text(x_s, header_y, "Sig",
                 fontsize=7.0, fontweight="bold",
                 ha="center", va="center", color="#222")
    ax_R.plot([0.0, 1.0], [underline_y, underline_y],
                color="#333", linewidth=0.5,
                clip_on=False, zorder=10)

    for _, r in df.iterrows():
        yi = r["_y"]
        is_sig = bool(r.get("Significant", False))

        # p(adj) — prefer FDR if present, else raw P
        fdr_v = r.get("FDR", np.nan)
        if pd.isna(fdr_v):
            p_show = r.get("P_value", np.nan)
            p_color = "#666"
        else:
            p_show = fdr_v
            p_color = "#222" if (not pd.isna(fdr_v) and fdr_v < 0.05) else "#666"
        p_str = _fmt_p_short(p_show)
        p_w = "bold" if (not pd.isna(p_show) and p_show < 0.05) else "normal"
        ax_R.text(x_p, yi, p_str,
                     fontsize=5.8, family="DejaVu Sans Mono",
                     ha="center", va="center",
                     color=p_color, fontweight=p_w)

        # Sig stars
        sig_str = r.get("stars", "")
        if not isinstance(sig_str, str):
            sig_str = ""
        if sig_str and sig_str not in ("ns", "—"):
            ax_R.text(x_s, yi, sig_str,
                         fontsize=7.0, family="DejaVu Sans Mono",
                         ha="center", va="center",
                         color="#222", fontweight="bold")

    # ─────────────────────────────────────────────────────────────────────────
    # OVERALL row separator
    # ─────────────────────────────────────────────────────────────────────────
    if sep_after_overall and (df[row_entity_col].astype(str) == overall_label).any():
        ov_y = df.loc[df[row_entity_col].astype(str) == overall_label, "_y"].iloc[0]
        sep_y = ov_y - 0.5
        for ax in [ax_L, ax_F, ax_R] + ([ax_M] if has_extra else []):
            ax.axhline(sep_y, color="#999", linewidth=0.4,
                          linestyle=(0, (1, 1.5)), alpha=0.6, zorder=1)

    # ─────────────────────────────────────────────────────────────────────────
    # Title / subtitle / footnote
    # ─────────────────────────────────────────────────────────────────────────
    title_y = 0.995
    fig.text(0.5, title_y, title,
                fontsize=9.0, fontweight="bold",
                ha="center", va="top", color="#111")
    if subtitle:
        fig.text(0.5, title_y - 0.030, subtitle,
                    fontsize=6.5, fontfamily="DejaVu Sans Mono",
                    ha="center", va="top", color="#444")
    if footnote:
        fig.text(0.5, 0.005, footnote,
                    fontsize=6.0, fontstyle="italic",
                    ha="center", va="bottom", color="#666")

    save_figure(fig, slug)


print(f"  ✓ _render_forest      defined  (R/ggplot table style)")
print(f"  ✓ Row color fns       lineage / constant / phase")
print(f"  ✓ Effect configs      EFFECT_LOG_OR / EFFECT_LINEAR_D / EFFECT_LOG_OR_SEN")
print(f"  ✓ Utilities           _slugify, _fmt_p_short, _truncate_label, _lighten")
print(f"\n✓ §5a forest plot helpers ready  (R-style table layout)")
print(f"  Used by: §5b GLMM forest, §5c Sloan, §5d cell cycle, §7b/c pathology")

---
## 26 · Group contrast — cell-level GLMM

**Why.** `is_senescent ~ <primary> + <covariates> + (1 | Donor)`, binomial, `glmer` via rpy2. The random intercept is what makes the cell-level n legitimate.

Reports both vs-reference and adjacent contrasts. Guards are stricter here than at donor level: `min_cells = 100`, `min_donors = 15`.

The covariates resolve from `spec['covariates_cell']`, which includes `log10_total_counts_scaled` — so this is the one model in the module that adjusts for depth.

In [ ]:
# =============================================================================
# §5 — CELL-LEVEL LOGISTIC GLMM
# =============================================================================
# Fits a logistic GLMM with random Donor intercept on cell-level data:
#   is_senescent ~ primary + covariates + (1|Donor)
#
# Iterates over PRIMARY_SPECS. For each spec:
#   - OVERALL model: all cell types pooled
#   - PER-CT models: one model per cell type
#
# CONTRAST STRUCTURE:
#   PRIMARY (vs reference) — Wald tests on each non-reference level.
#                            FDR-corrected within spec across all CT × contrast.
#                            Significance flag uses FDR < FDR_THRESHOLD.
#
#   ADJACENT (between non-reference levels) — exact post-hoc linear contrasts
#                            from the fitted model's vcov:
#                              β(B vs A) = β_B − β_A
#                              SE = √(Var_B + Var_A − 2·Cov_AB)
#                            Reported with raw p-values, marked as exploratory,
#                            NOT FDR-corrected.
#
# Outputs:
#   results/s5_glmm_results_{slug}.csv    — one per spec, both contrast types
#   results/s5_glmm_results_combined.csv  — all specs concatenated
#   df_glmm_by_spec  — dict {slug → DataFrame}
# =============================================================================

# ─────────────────────────────────────────────────────────────────────────────
# GLMM_CONFIG — central knobs
# ─────────────────────────────────────────────────────────────────────────────
GLMM_CONFIG = {
    "optimizer":  "bobyqa",
    "maxfun":     100000,
    "nAGQ":       1,
    "min_cells":  100,
    "min_donors": 5,
}

if "FDR_THRESHOLD" not in dir() and "FDR_THRESHOLD" not in globals():
    FDR_THRESHOLD = 0.05

print("=" * 64)
print(f"§5 — CELL-LEVEL LOGISTIC GLMM  |  {DATASET}")
print("=" * 64)
print(f"  GLMM_CONFIG: {GLMM_CONFIG}")
print(f"  FDR_THRESHOLD = {FDR_THRESHOLD}")

# ─────────────────────────────────────────────────────────────────────────────
# Defensive harmonization
# ─────────────────────────────────────────────────────────────────────────────
if "CELLTYPE_RENAME" in dir() and CELLTYPE_RENAME:
    n_ren = int(df_cells_model["Cell_Type"].isin(CELLTYPE_RENAME.keys()).sum())
    if n_ren > 0:
        print(f"  ⚠ Re-applying harmonization to df_cells_model ({n_ren} cells)")
        df_cells_model["Cell_Type"] = df_cells_model["Cell_Type"].replace(CELLTYPE_RENAME)
_refresh_celltype_order()

df_glmm_by_spec = {}

# ─────────────────────────────────────────────────────────────────────────────
# Build LEVEL_ORDER_MAP for adjacent contrasts dynamically
# ─────────────────────────────────────────────────────────────────────────────
def _build_level_order_map():
    out = {}
    if "study_group_order" in dir() or "study_group_order" in globals():
        non_ref_sg = [lv for lv in study_group_order if lv != REFERENCE_GROUP]
        if non_ref_sg:
            out["Study_Group"] = non_ref_sg
    if HAS_PATHOLOGY_GROUP and PATHOLOGY_GROUP_LEVEL_ORDER:
        non_ref_pg = [lv for lv in PATHOLOGY_GROUP_LEVEL_ORDER
                      if lv != PATHOLOGY_GROUP_REFERENCE]
        if non_ref_pg:
            out["pathology_group"] = non_ref_pg
    return out

LEVEL_ORDER_MAP = _build_level_order_map()
print(f"  Adjacent-contrast level order: {LEVEL_ORDER_MAP}")

# ─────────────────────────────────────────────────────────────────────────────
# Helper: fit one GLMM, return both vs-reference and adjacent contrasts
# ─────────────────────────────────────────────────────────────────────────────
def _run_glmm_one(data_df, spec, label="overall"):
    n_cells  = len(data_df)
    n_donors = data_df["Donor"].nunique()
    snc_rate = float(data_df["is_senescent"].mean())

    if n_cells < GLMM_CONFIG["min_cells"]:
        print(f"  {label:24s} ⊘ skipped (n_cells={n_cells:,} < {GLMM_CONFIG['min_cells']})")
        return None
    if n_donors < GLMM_CONFIG["min_donors"]:
        print(f"  {label:24s} ⊘ skipped (n_donors={n_donors} < {GLMM_CONFIG['min_donors']})")
        return None
    if snc_rate == 0 or snc_rate == 1:
        print(f"  {label:24s} ⊘ skipped (no variance: SnC rate = {snc_rate:.3f})")
        return None

    primary = spec["var_cell"]
    cov_terms = []
    for cov in spec["covariates_cell"]:
        if cov not in data_df.columns:
            continue
        if cov in {"Sex", "Cohort"} and data_df[cov].nunique() < 2:
            continue
        cov_terms.append(cov)

    fixed_terms  = [primary] + cov_terms
    formula_full = f"is_senescent ~ {' + '.join(fixed_terms)} + (1|Donor)"

    if spec["type"] == "continuous":
        relevel_cmd = ""
    else:
        relevel_cmd = (
            f'ct_data${primary} <- relevel('
            f'factor(ct_data${primary}), ref = "{spec["reference"]}")'
        )

    try:
        with localconverter(pandas_converter):
            ro.globalenv["ct_data"] = data_df

        r_code = f"""
        suppressMessages(library(lme4))
        {relevel_cmd}

        tryCatch({{
            model <- glmer(
                {formula_full},
                data    = ct_data,
                family  = binomial(link = "logit"),
                control = glmerControl(
                    optimizer = "{GLMM_CONFIG['optimizer']}",
                    optCtrl   = list(maxfun = {GLMM_CONFIG['maxfun']})
                ),
                nAGQ = {GLMM_CONFIG['nAGQ']}
            )

            coef_summary <- summary(model)$coefficients
            vc           <- as.matrix(vcov(model))

            list(
                success    = TRUE,
                term_names = rownames(coef_summary),
                estimates  = as.numeric(coef_summary[, "Estimate"]),
                std_errors = as.numeric(coef_summary[, "Std. Error"]),
                z_values   = as.numeric(coef_summary[, "z value"]),
                p_values   = as.numeric(coef_summary[, "Pr(>|z|)"]),
                vcov_flat  = as.numeric(vc),
                vcov_names = rownames(vc),
                vcov_dim   = as.integer(nrow(vc)),
                re_var     = as.numeric(VarCorr(model)$Donor[1]),
                aic        = AIC(model)
            )
        }}, error = function(e) {{
            list(success = FALSE, error = as.character(e))
        }})
        """
        result = ro.r(r_code)

        if not result.rx2("success")[0]:
            err = str(result.rx2("error")[0])
            print(f"  {label:24s} ✗ R error: {err[:80]}")
            return None

        coef_df = pd.DataFrame({
            "term":     list(result.rx2("term_names")),
            "Estimate": np.array(result.rx2("estimates")),
            "SE":       np.array(result.rx2("std_errors")),
            "Z":        np.array(result.rx2("z_values")),
            "P_value":  np.array(result.rx2("p_values")),
        })
        re_var = float(result.rx2("re_var")[0])
        aic    = float(result.rx2("aic")[0])

        vcov_dim   = int(result.rx2("vcov_dim")[0])
        vcov_flat  = np.array(result.rx2("vcov_flat"))
        vcov_names = list(result.rx2("vcov_names"))
        vcov_mat   = vcov_flat.reshape((vcov_dim, vcov_dim), order="F")
        vcov_df    = pd.DataFrame(vcov_mat, index=vcov_names, columns=vcov_names)

        if spec["type"] == "continuous":
            primary_terms = [t for t in coef_df["term"] if t == primary]
        else:
            primary_terms = [t for t in coef_df["term"]
                                if t.startswith(primary) and t != primary]

        if not primary_terms:
            print(f"  {label:24s} ⚠ target term not found. Available: {coef_df['term'].tolist()}")
            return None

        results = []

        # ── vs-reference contrasts ──────────────────────────────────────────
        for term in primary_terms:
            row  = coef_df[coef_df["term"] == term].iloc[0]
            beta = float(row["Estimate"])
            se   = float(row["SE"])
            if spec["type"] == "continuous":
                contrast = primary
            else:
                contrast = term.replace(primary, "").strip()

            results.append({
                "Cell_Type":     label,
                "Contrast":      contrast,
                "Contrast_full": term,
                "Contrast_type": "vs_reference",
                "Reference":     spec["reference"] if spec["type"] == "categorical" else None,
                "N_cells":       n_cells,
                "N_donors":      n_donors,
                "SnC_rate":      round(snc_rate, 4),
                "Beta":          round(beta, 6),
                "SE":            round(se, 6),
                "OR":            round(float(np.exp(beta)), 4),
                "CI_low":        round(float(np.exp(beta - 1.96 * se)), 4),
                "CI_high":       round(float(np.exp(beta + 1.96 * se)), 4),
                "Z":             round(float(row["Z"]), 4),
                "P_value":       round(float(row["P_value"]), 6),
                "Donor_var":     round(re_var, 4),
                "AIC":           round(aic, 2),
            })

        # ── ADJACENT contrasts ──────────────────────────────────────────────
        if spec["type"] == "categorical" and len(primary_terms) >= 2:
            preferred = LEVEL_ORDER_MAP.get(spec["var_cell"], None)

            term_to_level = {t: t.replace(primary, "").strip() for t in primary_terms}
            level_to_term = {v: k for k, v in term_to_level.items()}
            levels        = list(term_to_level.values())

            if preferred:
                ordered_levels = ([lv for lv in preferred if lv in levels]
                                      + [lv for lv in levels if lv not in preferred])
            else:
                ordered_levels = sorted(levels)

            for i in range(len(ordered_levels) - 1):
                lv_a   = ordered_levels[i]
                lv_b   = ordered_levels[i + 1]
                term_a = level_to_term[lv_a]
                term_b = level_to_term[lv_b]

                if term_a not in vcov_df.index or term_b not in vcov_df.index:
                    continue

                beta_a = float(coef_df.loc[coef_df["term"] == term_a, "Estimate"].iloc[0])
                beta_b = float(coef_df.loc[coef_df["term"] == term_b, "Estimate"].iloc[0])

                var_a  = float(vcov_df.loc[term_a, term_a])
                var_b  = float(vcov_df.loc[term_b, term_b])
                cov_ab = float(vcov_df.loc[term_a, term_b])
                var_diff = var_b + var_a - 2 * cov_ab
                if var_diff <= 0:
                    continue
                se_diff = float(np.sqrt(var_diff))

                beta_diff = beta_b - beta_a
                z_diff    = beta_diff / se_diff
                p_diff    = float(2 * (1 - stats.norm.cdf(abs(z_diff))))

                results.append({
                    "Cell_Type":     label,
                    "Contrast":      f"{lv_b} vs {lv_a}",
                    "Contrast_full": f"({term_b}) - ({term_a})",
                    "Contrast_type": "adjacent",
                    "Reference":     lv_a,
                    "N_cells":       n_cells,
                    "N_donors":      n_donors,
                    "SnC_rate":      round(snc_rate, 4),
                    "Beta":          round(beta_diff, 6),
                    "SE":            round(se_diff, 6),
                    "OR":            round(float(np.exp(beta_diff)), 4),
                    "CI_low":        round(float(np.exp(beta_diff - 1.96 * se_diff)), 4),
                    "CI_high":       round(float(np.exp(beta_diff + 1.96 * se_diff)), 4),
                    "Z":             round(float(z_diff), 4),
                    "P_value":       round(p_diff, 6),
                    "Donor_var":     round(re_var, 4),
                    "AIC":           round(aic, 2),
                })

        first_vsref = next((r for r in results if r["Contrast_type"] == "vs_reference"), None)
        if first_vsref:
            print(f"  {label:24s} ✓ OR({first_vsref['Contrast']})="
                  f"{first_vsref['OR']:.3f}  "
                  f"p={first_vsref['P_value']:.4f}  "
                  f"{sig_stars(first_vsref['P_value'])}")
        else:
            print(f"  {label:24s} ✓ {len(results)} contrast(s)")
        return results

    except Exception as e:
        print(f"  {label:24s} ✗ ERROR: {str(e)[:80]}")
        return None


# ─────────────────────────────────────────────────────────────────────────────
# Iterate over PRIMARY_SPECS
# ─────────────────────────────────────────────────────────────────────────────
for spec in PRIMARY_SPECS:
    print(f"\n{'━'*64}")
    print(f"  Spec: {spec['label']}")
    print(f"{'━'*64}")

    if spec["var_cell"] not in df_cells_model.columns:
        print(f"  ⊘ {spec['var_cell']!r} not in df_cells_model — skipping spec")
        continue

    glmm_cols = ["Donor", "Cell_Type", "is_senescent", spec["var_cell"]]
    for cov in spec["covariates_cell"]:
        if cov in df_cells_model.columns and cov not in glmm_cols:
            glmm_cols.append(cov)

    df_glmm_input = df_cells_model[glmm_cols].copy().dropna()
    df_glmm_input["is_senescent"] = df_glmm_input["is_senescent"].astype(int)
    df_glmm_input["Donor"]        = df_glmm_input["Donor"].astype(str)

    if spec["type"] == "categorical":
        df_glmm_input[spec["var_cell"]] = df_glmm_input[spec["var_cell"]].astype(str)

    for cov in spec["covariates_cell"]:
        if cov in df_glmm_input.columns:
            if cov in {"Sex", "Cohort"}:
                df_glmm_input[cov] = df_glmm_input[cov].astype(str)
            else:
                df_glmm_input[cov] = pd.to_numeric(df_glmm_input[cov], errors="coerce")
    df_glmm_input = df_glmm_input.dropna()

    print(f"  Input    : {len(df_glmm_input):,} cells | "
          f"{df_glmm_input['Donor'].nunique()} donors | "
          f"{df_glmm_input['Cell_Type'].nunique()} cell types")
    print(f"  SnC rate : {df_glmm_input['is_senescent'].mean()*100:.2f}%\n")

    spec_results = []

    print(f"  ▸ OVERALL (pooled across CTs)")
    overall = _run_glmm_one(df_glmm_input, spec, label="OVERALL")
    if overall:
        spec_results.extend(overall)

    print(f"\n  ▸ Per cell type")
    for ct in CELLTYPE_ORDER_PLOT:
        ct_data    = df_glmm_input[df_glmm_input["Cell_Type"] == ct].copy()
        ct_results = _run_glmm_one(ct_data, spec, label=ct)
        if ct_results:
            spec_results.extend(ct_results)

    if not spec_results:
        print(f"\n  ⊘ No GLMM results for spec {spec['name']!r}")
        continue

    df_spec = pd.DataFrame(spec_results)

    df_spec["FDR"]         = np.nan
    df_spec["stars"]       = "—"
    df_spec["Significant"] = False
    df_spec["Exploratory"] = False

    vsref_mask = df_spec["Contrast_type"] == "vs_reference"
    if vsref_mask.any():
        df_spec.loc[vsref_mask, "FDR"] = bh_correction(
            df_spec.loc[vsref_mask, "P_value"].values
        ).round(6)
        df_spec.loc[vsref_mask, "stars"] = (
            df_spec.loc[vsref_mask, "FDR"].apply(sig_stars)
        )
        df_spec.loc[vsref_mask, "Significant"] = (
            df_spec.loc[vsref_mask, "FDR"] < FDR_THRESHOLD
        )

    adj_mask = df_spec["Contrast_type"] == "adjacent"
    if adj_mask.any():
        df_spec.loc[adj_mask, "Exploratory"] = True
        df_spec.loc[adj_mask, "stars"] = (
            df_spec.loc[adj_mask, "P_value"].apply(sig_stars) + " (raw)"
        )

    df_spec["Primary"] = spec["name"]
    df_spec = df_spec.sort_values(["Contrast_type", "P_value"]).reset_index(drop=True)

    # ── vs-reference table ──────────────────────────────────────────────────
    print(f"\n  ─── vs-reference contrasts (FDR-corrected) "
          f"───────────────────────────")
    or_label = "OR/SD" if spec["type"] == "continuous" else "OR"
    print(f"  {'Cell Type':<20} {'Contrast':<24} {'N cells':>9} {'%SnC':>6} "
          f"{or_label:>7} {'CI':>16} {'p':>10} {'FDR':>10} {'':>3}")
    print(f"  {'─'*98}")
    for _, row in df_spec[df_spec["Contrast_type"] == "vs_reference"].iterrows():
        ci_str  = f"[{row['CI_low']:.2f}–{row['CI_high']:.2f}]"
        sig_str = "✓" if row["Significant"] else ""
        prefix  = "► " if row["Cell_Type"] == "OVERALL" else "  "
        print(
            f"  {prefix}{row['Cell_Type']:<18} "
            f"{row['Contrast']:<24} "
            f"{row['N_cells']:>9,} "
            f"{row['SnC_rate']*100:>5.1f}% "
            f"{row['OR']:>7.3f} "
            f"{ci_str:>16} "
            f"{row['P_value']:>10.2e} "
            f"{row['FDR']:>10.2e} "
            f"{sig_str:>3}"
        )

    if adj_mask.any():
        print(f"\n  ─── adjacent contrasts (exploratory, raw p) "
              f"────────────────────────")
        print(f"  {'Cell Type':<20} {'Contrast':<24} {'N cells':>9} {'%SnC':>6} "
              f"{or_label:>7} {'CI':>16} {'p (raw)':>10}")
        print(f"  {'─'*88}")
        for _, row in df_spec[df_spec["Contrast_type"] == "adjacent"].iterrows():
            ci_str = f"[{row['CI_low']:.2f}–{row['CI_high']:.2f}]"
            prefix = "► " if row["Cell_Type"] == "OVERALL" else "  "
            print(
                f"  {prefix}{row['Cell_Type']:<18} "
                f"{row['Contrast']:<24} "
                f"{row['N_cells']:>9,} "
                f"{row['SnC_rate']*100:>5.1f}% "
                f"{row['OR']:>7.3f} "
                f"{ci_str:>16} "
                f"{row['P_value']:>10.2e}"
            )

    save_table(df_spec, f"s5_glmm_results_{spec['slug']}")
    df_glmm_by_spec[spec["slug"]] = df_spec

    n_vsref     = int((df_spec["Contrast_type"] == "vs_reference").sum())
    n_adj       = int((df_spec["Contrast_type"] == "adjacent").sum())
    n_sig_vsref = int(df_spec["Significant"].sum())
    print(f"\n  → {df_spec['Cell_Type'].nunique()} entries  ·  "
          f"{n_vsref} vs-ref  +  {n_adj} adjacent  ·  "
          f"{n_sig_vsref} sig vs-ref (FDR<{FDR_THRESHOLD})")

    if n_sig_vsref > 0:
        print(f"\n  Significant vs-ref hits:")
        for _, row in df_spec[df_spec["Significant"]].iterrows():
            direction = "↑" if row["Beta"] > 0 else "↓"
            print(f"    {direction} {row['Cell_Type']:18s} "
                  f"({row['Contrast']:22s})  "
                  f"OR={row['OR']:.3f}  FDR={row['FDR']:.2e}")

# Combined output
if df_glmm_by_spec:
    df_glmm = pd.concat(df_glmm_by_spec.values(), ignore_index=True)
    save_table(df_glmm, "s5_glmm_results_combined")
else:
    df_glmm = pd.DataFrame()
    print(f"\n⚠ No GLMM results across any spec")

print(f"\n{'─'*64}")
print(f"  §5 SUMMARY")
print(f"{'─'*64}")
for spec in PRIMARY_SPECS:
    slug = spec["slug"]
    if slug not in df_glmm_by_spec:
        print(f"  {spec['name']:18s}  ⊘ no results")
        continue
    df_s        = df_glmm_by_spec[slug]
    n_cts       = df_s["Cell_Type"].nunique()
    n_vsref     = int((df_s["Contrast_type"] == "vs_reference").sum())
    n_adj       = int((df_s["Contrast_type"] == "adjacent").sum())
    n_sig_vsref = int(df_s["Significant"].sum())
    print(f"  {spec['name']:18s}  "
          f"{n_cts:2d} models  ·  "
          f"{n_vsref:2d} vs-ref  +  {n_adj:2d} adj  ·  "
          f"{n_sig_vsref:2d} sig vs-ref (FDR<{FDR_THRESHOLD})")

print(f"\n✓ §5 Cell-level GLMM complete")
print(f"  Tables : s5_glmm_results_{{slug}}.csv  + combined")
print(f"  Next   : §5a forest plot helpers → §5b GLMM forest plots")

---
## 27 · GLMM forests

**Why.** Per-cell-type forests on the odds-ratio scale, null at OR = 1.

In [ ]:
# =============================================================================
# §5b — GLMM FOREST PLOTS  (uses v1 _render_forest from §5a)
# =============================================================================
# Two passes:
#   PASS A — Per-spec full forest  (one figure per spec × non-ref contrast)
#   PASS B — OVERALL across specs (combined)
# =============================================================================

print("=" * 64)
print(f"§5b — GLMM FOREST PLOTS  |  {DATASET}")
print("=" * 64)

# Defensive harmonization
if "CELLTYPE_RENAME" in dir() and CELLTYPE_RENAME:
    n_ren = int(df_cells_model["Cell_Type"].isin(CELLTYPE_RENAME.keys()).sum())
    if n_ren > 0:
        print(f"  ⚠ Re-applying harmonization ({n_ren} cells)")
        df_cells_model["Cell_Type"] = df_cells_model["Cell_Type"].replace(CELLTYPE_RENAME)
_refresh_celltype_order()

if not df_glmm_by_spec:
    print(f"  ⊘ df_glmm_by_spec empty — run §5 first")
else:
    # ─────────────────────────────────────────────────────────────────────────
    # PASS A — Per-spec, per-contrast forest
    # ─────────────────────────────────────────────────────────────────────────
    print(f"\n{'─'*64}")
    print(f"  PASS A — Per-spec forest (full, one fig per non-ref contrast)")
    print(f"{'─'*64}")

    for spec in PRIMARY_SPECS:
        slug = spec["slug"]
        if slug not in df_glmm_by_spec:
            print(f"  ⊘ {slug}: no GLMM results")
            continue

        df_s = df_glmm_by_spec[slug]
        # vs-reference rows only — adjacent contrasts duplicate CT rows
        df_for = df_s[df_s["Contrast_type"] == "vs_reference"].copy()
        if len(df_for) == 0:
            print(f"  ⊘ {slug}: no vs-reference rows")
            continue

        contrasts_in_spec = sorted(df_for["Contrast"].unique())

        for contrast in contrasts_in_spec:
            df_contrast = df_for[df_for["Contrast"] == contrast].copy()

            # Sort by canonical CT order (OVERALL pinned to top via _ct_sort_key)
            df_contrast["_sort"] = df_contrast["Cell_Type"].map(_ct_sort_key)
            df_contrast = df_contrast.sort_values("_sort").reset_index(drop=True)

            # row_label is what the LEFT panel shows
            df_contrast["row_label"] = df_contrast["Cell_Type"]

            n_rows = len(df_contrast)
            n_sig  = int(df_contrast["Significant"].sum())
            print(f"  ▸ {slug} | {contrast}: {n_rows} CTs, {n_sig} sig")

            title = (f"Cell-level GLMM — {spec['label']}: {contrast}"
                       f"  ·  {DATASET}")
            subtitle = f"is_senescent ~ {spec['var_cell']} + covariates + (1|Donor)"

            slug_out = f"s5b_glmm_forest_{slug}"
            if len(contrasts_in_spec) > 1:
                slug_out = f"s5b_glmm_forest_{slug}_{_slugify(contrast)}"

            _render_forest(
                df              = df_contrast,
                title           = title,
                slug            = slug_out,
                subtitle        = subtitle,
                header_label    = "Cell Type",
                effect_config   = EFFECT_LOG_OR,
                row_color_fn    = lineage_row_color_fn,
                row_entity_col  = "Cell_Type",
                row_label_col   = "row_label",
                effect_col      = "OR",
                ci_lo_col       = "CI_low",
                ci_hi_col       = "CI_high",
                overall_label   = "OVERALL",
                sep_after_overall = True,
            )

    # ─────────────────────────────────────────────────────────────────────────
    # PASS B — OVERALL across specs (combined)
    # ─────────────────────────────────────────────────────────────────────────
    print(f"\n{'─'*64}")
    print(f"  PASS B — OVERALL across specs")
    print(f"{'─'*64}")

    overall_rows = []
    for spec in PRIMARY_SPECS:
        slug = spec["slug"]
        if slug not in df_glmm_by_spec:
            continue
        df_s = df_glmm_by_spec[slug]
        df_overall = df_s[
            (df_s["Cell_Type"] == "OVERALL")
            & (df_s["Contrast_type"] == "vs_reference")
        ].copy()
        if len(df_overall) == 0:
            continue
        df_overall["row_label"] = (
            df_overall["Primary"] + " · " + df_overall["Contrast"]
        )
        overall_rows.append(df_overall)

    if overall_rows:
        df_all = pd.concat(overall_rows, ignore_index=True)
        # All rows are OVERALL — color by Primary instead
        spec_color_map = {}
        palette_fallback = ["#222222", "#1F77B4", "#2CA02C", "#D62728",
                              "#9467BD", "#8C564B", "#E377C2"]
        for i, p in enumerate(df_all["Primary"].unique()):
            spec_color_map[p] = palette_fallback[i % len(palette_fallback)]

        def _spec_color_fn(row_entity):
            # row_entity will be the row_label, e.g. "Study_Group · AD"
            for p, c in spec_color_map.items():
                if str(row_entity).startswith(p):
                    return c
            return "#666"

        # Make all rows look like OVERALL diamonds — set Cell_Type to OVERALL
        # so _render_forest pins them with the special diamond style
        df_all["Cell_Type"] = "OVERALL"

        print(f"  ▸ {len(df_all)} OVERALL rows across "
              f"{df_all['Primary'].nunique()} specs")

        _render_forest(
            df              = df_all,
            title           = f"Cell-level GLMM — OVERALL across specs  ·  {DATASET}",
            slug            = "s5b_glmm_forest_overall_combined",
            subtitle        = None,
            header_label    = "Spec · Contrast",
            effect_config   = EFFECT_LOG_OR,
            row_color_fn    = _spec_color_fn,
            row_entity_col  = "row_label",   # color by spec label
            row_label_col   = "row_label",
            effect_col      = "OR",
            ci_lo_col       = "CI_low",
            ci_hi_col       = "CI_high",
            overall_label   = "__never_match__",   # disable overall-special-treatment
            sep_after_overall = False,
        )
    else:
        print(f"  ⊘ no OVERALL rows across any spec")

    print(f"\n✓ §5b GLMM forest plots complete")
    print(f"  Per-spec : s5b_glmm_forest_{{slug}}[_contrast].{{pdf,png,svg}}")
    print(f"  Combined : s5b_glmm_forest_overall_combined.{{pdf,png,svg}}")
    print(f"  Next     : §5c Sloan validation, §5d cell cycle")

---
## 28 · Cross-model summary

**Why.** Puts the donor-level and cell-level estimates side by side per cell type and flags where they disagree.

**This section no-ops if the GLMM produced nothing** — it prints `⊘ No GLMM results — skipping` and moves on. If you see that line, the comparison did not happen, and the donor-level and cell-level results have not been checked against each other.

In [ ]:
# =============================================================================
# §6 — CROSS-MODEL SUMMARY
# =============================================================================
# Builds one unified convergence table:
#   one row per Cell_Type × Contrast (joining OLS / RLR / GLMM by these keys)
#   plus per-CT validation columns from Sloan (and Cell Cycle if available)
#
# Columns produced:
#   Identification  : Cell_Type, Contrast, Contrast_type, Primary, Reference
#   OLS  (§3)       : OLS_beta,    OLS_p,    OLS_FDR,    OLS_sig
#   RLR  (§4)       : RLR_beta,    RLR_p,    RLR_FDR,    RLR_sig
#   GLMM (§5)       : GLMM_OR,     GLMM_CI,  GLMM_p,     GLMM_FDR, GLMM_sig
#   Sloan (§5c)     : Sloan_d_max, Sloan_module_max, Sloan_FDR_min, Sloan_sig
#   CellCycle (§5d) : CC_G1_OR, CC_G1_FDR, CC_G1_sig, ... (optional)
#
# No scoring — reader judges convergence by reading across columns.
# Adjacent contrasts from §5 are included as their own rows.
#
# Outputs:
#   results/s6_cross_model_summary.csv
# =============================================================================

print("=" * 64)
print(f"§6 — CROSS-MODEL SUMMARY  |  {DATASET}")
print("=" * 64)

# Defensive harmonization
if "CELLTYPE_RENAME" in dir() and CELLTYPE_RENAME:
    n_ren = int(df_cells_model["Cell_Type"].isin(CELLTYPE_RENAME.keys()).sum())
    if n_ren > 0:
        print(f"  ⚠ Re-applying harmonization ({n_ren} cells)")
        df_cells_model["Cell_Type"] = df_cells_model["Cell_Type"].replace(CELLTYPE_RENAME)
_refresh_celltype_order()

# ─────────────────────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────────────────────
def _flag_sig(fdr_eff_pairs, threshold=None):
    """Return arrow flag: ↑ for sig+positive, ↓ for sig+negative, blank for ns."""
    if threshold is None:
        threshold = FDR_THRESHOLD
    out = []
    for fdr, eff in fdr_eff_pairs:
        if pd.isna(fdr) or pd.isna(eff):
            out.append("")
        elif fdr < threshold:
            out.append("↑" if eff > 0 else "↓")
        else:
            out.append("")
    return out

def _safe_get(df, mask, col, default=np.nan):
    sub = df.loc[mask, col]
    if len(sub) == 0:
        return default
    return sub.iloc[0]

def _fmt_signed(v, decimals=3):
    if pd.isna(v):
        return "—"
    return f"{v:+.{decimals}f}"

def _fmt_or(v):
    if pd.isna(v):
        return "—"
    return f"{v:.2f}"

# ─────────────────────────────────────────────────────────────────────────────
# Step 1: Skeleton from GLMM
# ─────────────────────────────────────────────────────────────────────────────
if not df_glmm_by_spec:
    print(f"\n⊘ No GLMM results — skipping §6")
else:
    skeleton_rows = []
    for slug, df_s in df_glmm_by_spec.items():
        spec_obj = next((s for s in PRIMARY_SPECS if s["slug"] == slug), None)
        primary  = spec_obj["name"] if spec_obj else slug
        for _, r in df_s.iterrows():
            skeleton_rows.append({
                "Cell_Type":     r["Cell_Type"],
                "Contrast":      r["Contrast"],
                "Contrast_type": r["Contrast_type"],
                "Primary":       primary,
                "Reference":     r.get("Reference", None),
            })
    df_summary = pd.DataFrame(skeleton_rows)
    print(f"\n  Skeleton: {len(df_summary)} rows "
          f"({df_summary['Cell_Type'].nunique()} CTs × "
          f"{df_summary['Contrast'].nunique()} contrasts)")

    # ─── Step 2: Join OLS ────────────────────────────────────────────────────
    print(f"\n  Joining OLS (§3)...")
    if "df_ols_by_spec" in dir() and df_ols_by_spec:
        ols_frames = []
        for slug, df_o in df_ols_by_spec.items():
            spec_obj = next((s for s in PRIMARY_SPECS if s["slug"] == slug), None)
            primary  = spec_obj["name"] if spec_obj else slug
            df_oc    = df_o.copy()
            df_oc["Primary"] = primary
            ols_frames.append(df_oc)
        df_ols_all = pd.concat(ols_frames, ignore_index=True)

        ols_cols = ["Cell_Type", "Contrast", "Primary",
                     "Beta", "P_value", "FDR"]
        ols_cols = [c for c in ols_cols if c in df_ols_all.columns]
        df_ols_merge = (df_ols_all[ols_cols]
                            .rename(columns={
                                "Beta":    "OLS_beta",
                                "P_value": "OLS_p",
                                "FDR":     "OLS_FDR",
                            }))
        df_summary = df_summary.merge(
            df_ols_merge, on=["Cell_Type", "Contrast", "Primary"], how="left",
        )
        n_ols = df_summary["OLS_beta"].notna().sum()
        print(f"    {n_ols}/{len(df_summary)} rows matched OLS")
    else:
        print(f"    ⊘ df_ols_by_spec not available")
        df_summary["OLS_beta"] = np.nan
        df_summary["OLS_p"]    = np.nan
        df_summary["OLS_FDR"]  = np.nan

    # ─── Step 3: Join RLR ────────────────────────────────────────────────────
    print(f"\n  Joining RLR (§4)...")
    if "df_rlr_by_spec" in dir() and df_rlr_by_spec:
        rlr_frames = []
        for slug, df_r in df_rlr_by_spec.items():
            spec_obj = next((s for s in PRIMARY_SPECS if s["slug"] == slug), None)
            primary  = spec_obj["name"] if spec_obj else slug
            df_rc    = df_r.copy()
            df_rc["Primary"] = primary
            rlr_frames.append(df_rc)
        df_rlr_all = pd.concat(rlr_frames, ignore_index=True)

        rlr_cols = ["Cell_Type", "Contrast", "Primary",
                     "Beta", "P_value", "FDR"]
        rlr_cols = [c for c in rlr_cols if c in df_rlr_all.columns]
        df_rlr_merge = (df_rlr_all[rlr_cols]
                            .rename(columns={
                                "Beta":    "RLR_beta",
                                "P_value": "RLR_p",
                                "FDR":     "RLR_FDR",
                            }))
        df_summary = df_summary.merge(
            df_rlr_merge, on=["Cell_Type", "Contrast", "Primary"], how="left",
        )
        n_rlr = df_summary["RLR_beta"].notna().sum()
        print(f"    {n_rlr}/{len(df_summary)} rows matched RLR")
    else:
        print(f"    ⊘ df_rlr_by_spec not available")
        df_summary["RLR_beta"] = np.nan
        df_summary["RLR_p"]    = np.nan
        df_summary["RLR_FDR"]  = np.nan

    # ─── Step 4: Join GLMM (defines skeleton) ────────────────────────────────
    print(f"\n  Joining GLMM (§5)...")
    glmm_frames = []
    for slug, df_s in df_glmm_by_spec.items():
        spec_obj = next((s for s in PRIMARY_SPECS if s["slug"] == slug), None)
        primary  = spec_obj["name"] if spec_obj else slug
        df_gc    = df_s.copy()
        df_gc["Primary"] = primary
        glmm_frames.append(df_gc)
    df_glmm_all = pd.concat(glmm_frames, ignore_index=True)

    glmm_cols = ["Cell_Type", "Contrast", "Contrast_type", "Primary",
                  "OR", "CI_low", "CI_high", "P_value", "FDR"]
    df_glmm_merge = (df_glmm_all[glmm_cols]
                        .rename(columns={
                            "OR":      "GLMM_OR",
                            "CI_low":  "GLMM_CI_low",
                            "CI_high": "GLMM_CI_high",
                            "P_value": "GLMM_p",
                            "FDR":     "GLMM_FDR",
                        }))
    df_summary = df_summary.merge(
        df_glmm_merge,
        on=["Cell_Type", "Contrast", "Contrast_type", "Primary"],
        how="left",
    )
    df_summary["GLMM_CI"] = df_summary.apply(
        lambda r: (f"[{r['GLMM_CI_low']:.2f}–{r['GLMM_CI_high']:.2f}]"
                       if pd.notna(r["GLMM_CI_low"]) else "—"),
        axis=1,
    )

    # ─── Step 5: Join Sloan (§5c) — collapse to per-CT (best |Cohen's d|) ────
    print(f"\n  Joining Sloan validation (§5c)...")
    if "df_sloan" in dir() and len(df_sloan) > 0:
        df_sloan_per_ct = []
        for ct, grp in df_sloan.groupby("Cell_Type"):
            idx = grp["Cohens_d"].abs().idxmax()
            best = grp.loc[idx]
            df_sloan_per_ct.append({
                "Cell_Type":        ct,
                "Sloan_d_max":      round(float(best["Cohens_d"]), 3),
                "Sloan_module_max": str(best["Module"]),
                "Sloan_FDR_min":    round(float(grp["FDR"].min()), 6),
                "Sloan_n_sig":      int(grp["Significant"].sum()),
            })
        df_sloan_per_ct = pd.DataFrame(df_sloan_per_ct)
        df_summary = df_summary.merge(df_sloan_per_ct,
                                              on="Cell_Type", how="left")
        n_sloan = df_summary["Sloan_d_max"].notna().sum()
        print(f"    {n_sloan}/{len(df_summary)} rows have Sloan data "
              f"({df_sloan_per_ct['Cell_Type'].nunique()} CTs validated)")
    else:
        print(f"    ⊘ df_sloan not available — skipping Sloan")
        df_summary["Sloan_d_max"]      = np.nan
        df_summary["Sloan_module_max"] = None
        df_summary["Sloan_FDR_min"]    = np.nan
        df_summary["Sloan_n_sig"]      = np.nan

    # ─── Step 6: Join Cell Cycle (§5d) — pivot phase × CT (optional) ─────────
    print(f"\n  Joining Cell Cycle validation (§5d)...")
    if "df_cc_lmm" in dir() and len(df_cc_lmm) > 0:
        cc_per_ct = []
        for ct, grp in df_cc_lmm.groupby("Cell_Type"):
            row = {"Cell_Type": ct}
            for phase in ["G1", "S", "G2M"]:
                sub = grp[grp["phase"] == phase]
                if len(sub) > 0:
                    s = sub.iloc[0]
                    row[f"CC_{phase}_OR"]  = round(float(s["OR"]), 3)
                    row[f"CC_{phase}_FDR"] = round(float(s["FDR"]), 6)
                else:
                    row[f"CC_{phase}_OR"]  = np.nan
                    row[f"CC_{phase}_FDR"] = np.nan
            cc_per_ct.append(row)
        df_cc_per_ct = pd.DataFrame(cc_per_ct)
        df_summary = df_summary.merge(df_cc_per_ct,
                                              on="Cell_Type", how="left")
        n_cc = df_summary["CC_G1_OR"].notna().sum()
        print(f"    {n_cc}/{len(df_summary)} rows have Cell Cycle data "
              f"({df_cc_per_ct['Cell_Type'].nunique()} CTs validated)")
    else:
        print(f"    ⊘ df_cc_lmm not available — skipping Cell Cycle")
        for phase in ["G1", "S", "G2M"]:
            df_summary[f"CC_{phase}_OR"]  = np.nan
            df_summary[f"CC_{phase}_FDR"] = np.nan

    # ─── Step 7: Significance flags ──────────────────────────────────────────
    print(f"\n  Computing significance flags...")
    df_summary["OLS_sig"] = _flag_sig(zip(df_summary["OLS_FDR"],
                                                  df_summary["OLS_beta"]))
    df_summary["RLR_sig"] = _flag_sig(zip(df_summary["RLR_FDR"],
                                                  df_summary["RLR_beta"]))
    glmm_eff = df_summary["GLMM_OR"].apply(
        lambda v: np.nan if pd.isna(v) else (v - 1.0)
    )
    glmm_fdr = df_summary.apply(
        lambda r: r["GLMM_FDR"] if r["Contrast_type"] == "vs_reference"
                  else (r["GLMM_p"] if r["GLMM_p"] < 0.05 else np.nan),
        axis=1,
    )
    df_summary["GLMM_sig"] = _flag_sig(zip(glmm_fdr, glmm_eff))

    df_summary["Sloan_sig"] = _flag_sig(
        zip(df_summary["Sloan_FDR_min"], df_summary["Sloan_d_max"])
    )

    for phase in ["G1", "S", "G2M"]:
        cc_eff = df_summary[f"CC_{phase}_OR"].apply(
            lambda v: np.nan if pd.isna(v) else (v - 1.0)
        )
        df_summary[f"CC_{phase}_sig"] = _flag_sig(
            zip(df_summary[f"CC_{phase}_FDR"], cc_eff)
        )

    # ─── Step 8: Column order ────────────────────────────────────────────────
    output_cols = [
        "Cell_Type", "Contrast", "Contrast_type", "Primary", "Reference",
        "OLS_beta", "OLS_p", "OLS_FDR", "OLS_sig",
        "RLR_beta", "RLR_p", "RLR_FDR", "RLR_sig",
        "GLMM_OR", "GLMM_CI", "GLMM_p", "GLMM_FDR", "GLMM_sig",
        "Sloan_d_max", "Sloan_module_max",
        "Sloan_FDR_min", "Sloan_n_sig", "Sloan_sig",
        "CC_G1_OR",  "CC_G1_FDR",  "CC_G1_sig",
        "CC_S_OR",   "CC_S_FDR",   "CC_S_sig",
        "CC_G2M_OR", "CC_G2M_FDR", "CC_G2M_sig",
    ]
    output_cols = [c for c in output_cols if c in df_summary.columns]
    df_summary  = df_summary[output_cols].copy()

    df_summary["_ct_rank"]   = df_summary["Cell_Type"].apply(_ct_sort_key)
    df_summary["_type_rank"] = (df_summary["Contrast_type"]
                                      == "adjacent").astype(int)
    df_summary = (df_summary
                       .sort_values(["_ct_rank", "Primary",
                                       "_type_rank", "Contrast"])
                       .drop(columns=["_ct_rank", "_type_rank"])
                       .reset_index(drop=True))

    save_table(df_summary, "s6_cross_model_summary")

    # ─── Console preview — vs-reference rows ─────────────────────────────────
    print(f"\n{'─'*64}")
    print(f"  §6 PREVIEW — vs-reference contrasts only")
    print(f"{'─'*64}")
    df_vsref = df_summary[df_summary["Contrast_type"] == "vs_reference"]
    print(f"  {'Cell_Type':<18} {'Contrast':<22} "
          f"{'OLS':>4} {'RLR':>4} {'GLMM_OR':>8} {'GLMM':>5} "
          f"{'Sloan_d':>8} {'Slo':>4} "
          f"{'G1':>4} {'S':>4} {'G2M':>4}")
    print(f"  {'─'*100}")
    for _, r in df_vsref.iterrows():
        print(
            f"  {r['Cell_Type']:<18} "
            f"{r['Contrast']:<22} "
            f"{r['OLS_sig']:>4} "
            f"{r['RLR_sig']:>4} "
            f"{_fmt_or(r['GLMM_OR']):>8} "
            f"{r['GLMM_sig']:>5} "
            f"{_fmt_signed(r['Sloan_d_max'], 2):>8} "
            f"{r['Sloan_sig']:>4} "
            f"{r['CC_G1_sig']:>4} "
            f"{r['CC_S_sig']:>4} "
            f"{r['CC_G2M_sig']:>4}"
        )

    # ─── Adjacent rows preview ───────────────────────────────────────────────
    df_adj = df_summary[df_summary["Contrast_type"] == "adjacent"]
    if len(df_adj) > 0:
        print(f"\n{'─'*64}")
        print(f"  §6 PREVIEW — adjacent contrasts (exploratory, raw p)")
        print(f"{'─'*64}")
        print(f"  {'Cell_Type':<18} {'Contrast':<28} "
              f"{'GLMM_OR':>8} {'p (raw)':>10} {'sig*':>5}")
        print(f"  {'─'*72}")
        for _, r in df_adj.iterrows():
            print(
                f"  {r['Cell_Type']:<18} "
                f"{r['Contrast']:<28} "
                f"{_fmt_or(r['GLMM_OR']):>8} "
                f"{r['GLMM_p']:>10.2e} "
                f"{r['GLMM_sig']:>5}"
            )
        print(f"\n  *sig flag for adjacent uses raw p<0.05 (not FDR)")

    # ─── Convergence counts ──────────────────────────────────────────────────
    print(f"\n{'─'*64}")
    print(f"  CONVERGENCE COUNTS  (vs-reference contrasts only)")
    print(f"{'─'*64}")
    df_v = df_vsref.copy()
    df_v["n_methods_sig"] = (
        (df_v["OLS_sig"]    != "").astype(int)
        + (df_v["RLR_sig"]   != "").astype(int)
        + (df_v["GLMM_sig"]  != "").astype(int)
        + (df_v["Sloan_sig"] != "").astype(int)
        + (df_v["CC_G1_sig"] != "").astype(int)
    )
    print(f"  Methods agreeing (out of 5: OLS, RLR, GLMM, Sloan, CC-G1)")
    print(f"  {'≥3/5 methods sig':<25} : "
          f"{int((df_v['n_methods_sig'] >= 3).sum())} rows")
    print(f"  {'≥4/5 methods sig':<25} : "
          f"{int((df_v['n_methods_sig'] >= 4).sum())} rows")
    print(f"  {'5/5 methods sig':<25} : "
          f"{int((df_v['n_methods_sig'] == 5).sum())} rows")

    if (df_v["n_methods_sig"] >= 3).any():
        print(f"\n  Top convergent rows (≥3/5 methods, sorted by n_methods desc):")
        top = df_v[df_v["n_methods_sig"] >= 3] \
                  .sort_values("n_methods_sig", ascending=False)
        for _, r in top.iterrows():
            print(
                f"    [{r['n_methods_sig']}/5]  "
                f"{r['Cell_Type']:<18} {r['Contrast']:<22} "
                f"OLS={r['OLS_sig'] or '-'} "
                f"RLR={r['RLR_sig'] or '-'} "
                f"GLMM={r['GLMM_sig'] or '-'} "
                f"Sloan={r['Sloan_sig'] or '-'} "
                f"G1={r['CC_G1_sig'] or '-'}"
            )

    print(f"\n✓ §6 Cross-model summary complete")
    print(f"  Table: s6_cross_model_summary.csv  ({len(df_summary)} rows)")

---
## 29 · Burden frame — age bins

**Why.** Rebuilds `df_burden` directly from `adata.obs` for the per-decade models below. Re-declares its own paths and constants, as in the source; the module config in section 01 remains authoritative for everything else.

In [ ]:
# =============================================================================
# §2 — BUILD df_burden  |  psychad_aging  (Age bins: 20-29 … 80-100)
# =============================================================================
import numpy as np, pandas as pd
from pathlib import Path
print("="*64); print("§2 — BUILD df_burden (aging, age bins)"); print("="*64)

# ── paths (aging scheme) ──────────────────────────────────────────────────────
SCRATCH=_env("SENESCENCE_DATA"); TISSUE="brain"
STUDY_TYPE="aging"; DATASET="psychad_aging"
CONDITION_SUBPATH=STUDY_TYPE; CONDITION_TAG=STUDY_TYPE     # = "aging"
BASE=f"{SCRATCH}/{TISSUE}/module_03_burden_modeling/{CONDITION_SUBPATH}/{DATASET}"
PATHS={"data":f"{BASE}/data","figures":f"{BASE}/figures","results":f"{BASE}/results"}
for p in PATHS.values(): Path(p).mkdir(parents=True, exist_ok=True)
def save_table(df, slug, index=False):
    fp=f"{PATHS['results']}/{slug}.csv"; df.to_csv(fp,index=index); print(f"  ✓ saved → {Path(fp).name} ({len(df)})")

GROUP_VAR        = "Study_Group"
GROUP_ORDER      = ["Age_20_29","Age_30_39","Age_40_49","Age_50_59","Age_60_69","Age_70_79","Age_80_100"]
DROP_CELL_TYPES  = ["PVM","VLMC","VSMC","Adaptive","Endothelial","Pericyte"]
BURDEN_MIN_CELLS = 10
SEN_COL          = "is_senescent"
COVARIATES       = ["Age","Sex","Cohort"]

keep=["Donor","Cell_Type",SEN_COL,GROUP_VAR]+COVARIATES
obs=adata.obs[keep].copy()
obs["Donor"]=obs["Donor"].astype(str); obs["Cell_Type"]=obs["Cell_Type"].astype(str); obs[SEN_COL]=obs[SEN_COL].astype(int)
obs=obs[obs[GROUP_VAR].astype(str).isin(GROUP_ORDER)].copy()
present_groups=[g for g in GROUP_ORDER if g in set(obs[GROUP_VAR].astype(str).unique())]
print(f"  cells: {len(obs)} | donors: {obs['Donor'].nunique()}")
print("  group counts:\n" + obs[GROUP_VAR].value_counts().reindex(present_groups).to_string())

donor_tot=obs.groupby("Donor").size().rename("donor_total")
active_cts=[c for c in ["Excitatory","Inhibitory","Astrocyte","Oligodendrocyte","Microglia","OPC"]
            if c in obs["Cell_Type"].unique() and c not in DROP_CELL_TYPES]
g=(obs[obs["Cell_Type"].isin(active_cts)].groupby(["Donor","Cell_Type"])
   .agg(n_cells=(SEN_COL,"size"), n_snc=(SEN_COL,"sum")).reset_index())
full_idx=pd.MultiIndex.from_product([donor_tot.index, active_cts], names=["Donor","Cell_Type"])
df_burden=g.set_index(["Donor","Cell_Type"]).reindex(full_idx, fill_value=0).reset_index().merge(donor_tot, on="Donor")
df_burden["CellProp"]=df_burden["n_cells"]/df_burden["donor_total"]
df_burden["SenBurden"]=df_burden["n_snc"]/df_burden["donor_total"]
df_burden["SenFrac"]=np.where(df_burden["n_cells"]>=BURDEN_MIN_CELLS,
                              df_burden["n_snc"]/df_burden["n_cells"].replace(0,np.nan), np.nan)
dmeta=obs[["Donor",GROUP_VAR]+COVARIATES].drop_duplicates("Donor")
df_burden=df_burden.merge(dmeta, on="Donor", how="left")
df_burden[GROUP_VAR]=pd.Categorical(df_burden[GROUP_VAR].astype(str), categories=present_groups, ordered=True)
df_burden["Cell_Type"]=pd.Categorical(df_burden["Cell_Type"], categories=active_cts, ordered=True)
print(f"\n  shape: {df_burden.shape} | donors × CT = {df_burden['Donor'].nunique()} × {len(active_cts)}")
print("  overall senescent fraction by age bin:")
print(df_burden.groupby(GROUP_VAR, observed=True).apply(
      lambda d: d.groupby("Donor")["SenBurden"].sum().mean()*100).round(2).to_string())
print("\n  mean SenBurden (%) by bin × cell type:")
print((df_burden.groupby([GROUP_VAR,"Cell_Type"],observed=True)["SenBurden"].mean()
       .mul(100).round(3).unstack()).to_string())
save_table(df_burden, f"burden_donor_celltype_{CONDITION_TAG}")
print("\n✓ §2 complete")

---
## 30 · Susceptibility GLMM — per decade

**Why.** `is_senescent ~ age_dec + Sex + Cohort + (1 | Donor)`, fit within each cell type. Effect is the odds ratio per decade of age.

No depth term and no abundance term — this is the per-cell propensity, asked directly.

In [ ]:
# =============================================================================
# §5 — SUSCEPTIBILITY GLMM (aging) · is_senescent ~ Age_dec + Sex + Cohort + (1|Donor)
#   Predictor = Age (per decade) — the EXPOSURE. OR = per-decade senescence-odds.
# =============================================================================
import os, numpy as np, pandas as pd
from pathlib import Path
_r_home = os.environ.get("SENESCENCE_R_HOME")
if _r_home:
    os.environ.setdefault("R_HOME",      _r_home)
    os.environ.setdefault("R_LIBS",      _r_home + "/library")
    os.environ.setdefault("R_LIBS_USER", _r_home + "/library")
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, numpy2ri
from rpy2.robjects.conversion import localconverter
from statsmodels.stats.multitest import multipletests
pandas_converter = ro.default_converter + pandas2ri.converter
print("="*64); print("§5 — SUSCEPTIBILITY GLMM (aging, per decade)"); print("="*64)

SCRATCH=_env("SENESCENCE_DATA"); TISSUE="brain"
STUDY_TYPE="aging"; DATASET="psychad_aging"; CONDITION_TAG=STUDY_TYPE
BASE=f"{SCRATCH}/{TISSUE}/module_03_burden_modeling/{STUDY_TYPE}/{DATASET}"
PATHS={"results":f"{BASE}/results","figures":f"{BASE}/figures"}
for p in PATHS.values(): Path(p).mkdir(parents=True, exist_ok=True)
def save_table(df, slug, index=False):
    fp=f"{PATHS['results']}/{slug}.csv"; df.to_csv(fp,index=index); print(f"  ✓ saved → {Path(fp).name} ({len(df)})")

SEN_COL="is_senescent"
PRED="age_dec"                        # continuous per-decade predictor
COVARIATES=["Sex","Cohort"]           # Age is the exposure → NOT a covariate
active_cts=["Excitatory","Inhibitory","Astrocyte","Oligodendrocyte","Microglia","OPC"]
GLMM_CONFIG={"optimizer":"bobyqa","maxfun":100000,"nAGQ":1,"min_cells":100,"min_donors":5}

print("\n1. DATA — cell-level from adata.obs")
cell_df=adata.obs[["Donor","Cell_Type",SEN_COL,"Age"]+COVARIATES].copy()
cell_df["Donor"]=cell_df["Donor"].astype(str); cell_df["Cell_Type"]=cell_df["Cell_Type"].astype(str)
cell_df[SEN_COL]=cell_df[SEN_COL].astype(int)
cell_df=cell_df[cell_df["Cell_Type"].isin(active_cts)].copy()
cell_df["age_dec"]=cell_df["Age"].astype(float)/10.0
ccovs=[c for c in [PRED]+COVARIATES if c in cell_df.columns]
print(f"   cells: {len(cell_df)} | donors: {cell_df['Donor'].nunique()} | age range: "
      f"{cell_df['age_dec'].min():.1f}–{cell_df['age_dec'].max():.1f} decades")

print("\n2-3. FORMULA")
FORMULA=f"{SEN_COL} ~ {' + '.join(ccovs)} + (1|Donor)"
print(f"   {FORMULA}  | binomial(logit), glmer | effect = OR per DECADE")

print("\n4. RESULTS")
def fit_glmer(ct):
    sub=cell_df[cell_df.Cell_Type==ct].dropna(subset=[SEN_COL]+ccovs).copy()
    if len(sub)<GLMM_CONFIG["min_cells"] or sub.Donor.nunique()<GLMM_CONFIG["min_donors"]:
        print(f"   {ct:18s} ✗ below min"); return None
    if int(sub[SEN_COL].sum())<5: print(f"   {ct:18s} ✗ <5 senescent"); return None
    cv=[c for c in ccovs if not (c in {"Sex","Cohort"} and sub[c].astype(str).nunique()<2)]
    f=f"{SEN_COL} ~ {' + '.join(cv)} + (1|Donor)"
    with localconverter(pandas_converter): ro.globalenv["ct_data"]=sub
    rcode=f'''suppressMessages(library(lme4))
    tryCatch({{
        m <- glmer({f}, data=ct_data, family=binomial(link="logit"),
                   control=glmerControl(optimizer="{GLMM_CONFIG['optimizer']}",
                                        optCtrl=list(maxfun={GLMM_CONFIG['maxfun']})), nAGQ={GLMM_CONFIG['nAGQ']})
        cs <- summary(m)$coefficients
        list(ok=TRUE, est=cs["{PRED}","Estimate"], se=cs["{PRED}","Std. Error"],
             p=cs["{PRED}","Pr(>|z|)"], singular=isSingular(m),
             warned=(length(m@optinfo$conv$lme4$messages)>0))
    }}, error=function(e) list(ok=FALSE, err=as.character(e)))'''
    r=ro.r(rcode)
    if not r.rx2("ok")[0]: print(f"   {ct:18s} ✗ {str(r.rx2('err')[0])[:50]}"); return None
    b=float(r.rx2("est")[0]); se=float(r.rx2("se")[0])
    sing=bool(r.rx2("singular")[0]); warned=bool(r.rx2("warned")[0])
    flag=" ⚠singular" if sing else (" ⚠warn" if warned else "")
    print(f"   {ct:18s} ✓ OR={np.exp(b):.3f}/decade{flag}")
    return dict(Cell_Type=ct, n_cells=len(sub), n_donors=sub.Donor.nunique(), n_snc=int(sub[SEN_COL].sum()),
                beta=b, se=se, lo=b-1.96*se, hi=b+1.96*se, p=float(r.rx2("p")[0]), singular=sing, warned=warned)

rows=[r for r in (fit_glmer(ct) for ct in active_cts) if r]
glmm=pd.DataFrame(rows); glmm["padj"]=multipletests(glmm["p"], method="fdr_bh")[1]
glmm["OR"]=np.exp(glmm.beta); glmm["OR_lo"]=np.exp(glmm.lo); glmm["OR_hi"]=np.exp(glmm.hi)
glmm=glmm.sort_values("beta", ascending=False).reset_index(drop=True)
def fmt_p(x): return "<0.001" if x<0.001 else f"{x:.3f}"
print()
print(glmm.assign(OR=lambda d:d.OR.round(3), p=lambda d:d.p.map(fmt_p), padj=lambda d:d.padj.map(fmt_p),
                  flag=lambda d:np.where(d.singular,"singular",np.where(d.warned,"warn","")))
      [["Cell_Type","n_cells","n_donors","n_snc","OR","p","padj","flag"]].to_string(index=False))
print("   (OR per decade; >1 = senescence odds rise with age)")
save_table(glmm, f"susceptibility_glmm_{CONDITION_TAG}")
print("\n✓ §5 complete")

---
## 31 · Susceptibility forest

**Why.** Table-style forest: OR per decade, 95% CI, p, FDR.

In [ ]:
# =============================================================================
# §5b — PLOT: susceptibility GLMM table-forest (OR/decade · 95% CI · p · FDR)
# =============================================================================
import numpy as np, matplotlib as mpl, matplotlib.pyplot as plt
mpl.rcParams.update({"pdf.fonttype":42,"ps.fonttype":42,"svg.fonttype":"none",
    "font.family":"sans-serif","font.sans-serif":["Arial","Helvetica","DejaVu Sans"],
    "axes.linewidth":0.6,"font.size":7})
print("="*64); print("§5b — susceptibility GLMM forest (aging)"); print("="*64)
COL={"Excitatory":"#0072B2","Inhibitory":"#E69F00","Astrocyte":"#009E73",
     "Oligodendrocyte":"#56B4E9","Microglia":"#D55E00","OPC":"#CC79A7"}
NEURO={"Excitatory","Inhibitory"}
def fmt_p(x): return "<0.001" if x<0.001 else f"{x:.3f}"
d=glmm.sort_values("OR", ascending=False).reset_index(drop=True); n=len(d); y=np.arange(n)[::-1]
fig=plt.figure(figsize=(7.0,2.7),dpi=300); ax=fig.add_subplot(111)
ax.set_xlim(0,1); ax.set_ylim(-1.0,n+0.3); ax.axis("off")
xc={"ct":0.005,"fL":0.28,"fR":0.52,"or":0.625,"p":0.875,"fdr":0.99}
logv=np.concatenate([np.log(d.OR_lo.values),np.log(d.OR_hi.values)]); xmin,xmax=logv.min()-0.10,logv.max()+0.10
def tox(v): v=np.clip(v,xmin,xmax); return xc["fL"]+(v-xmin)/(xmax-xmin)*(xc["fR"]-xc["fL"])
y_hdr=n-0.35; y_line=n-0.55
ax.text(xc["ct"],y_hdr,"Cell type",fontsize=7.3,fontweight="bold",ha="left",va="bottom")
for lab,k,ha in [("OR (95% CI)","or","center"),("p","p","right"),("FDR","fdr","right")]:
    ax.text(xc[k],y_hdr,lab,fontsize=7.3,fontweight="bold",ha=ha,va="bottom")
ax.text((xc["fL"]+xc["fR"])/2,y_hdr,"per decade",fontsize=6.8,fontweight="bold",ha="center",va="bottom")
ax.plot([0,0.99],[y_line,y_line],"k-",lw=0.9,clip_on=False)
or_ticks=[0.8,1.0,1.25,1.5,2.0]
for t in or_ticks:
    lv=np.log(t)
    if xmin<=lv<=xmax:
        gx=tox(lv); ax.plot([gx,gx],[-0.55,y_line],color="#eee",lw=0.5,zorder=0)
        ax.text(gx,-0.72,f"{t:g}",fontsize=5.8,ha="center",va="top",color="#666")
ax.plot([tox(0)]*2,[-0.55,y_line],color="#888",ls="--",lw=0.8,alpha=0.7,zorder=1)
ax.text((xc["fL"]+xc["fR"])/2,-1.0,"Odds ratio per decade (log scale)",fontsize=6.2,ha="center",va="top",color="#444")
for i,r in d.iterrows():
    yy=y[i]; col=COL.get(r.Cell_Type,"#888"); sig=r.padj<0.05
    if i%2==0: ax.axhspan(yy-0.5,yy+0.5,xmin=0.003,xmax=0.997,color="#f7f7f7",zorder=0)
    ax.plot([tox(np.log(r.OR_lo)),tox(np.log(r.OR_hi))],[yy,yy],color=col,lw=1.4,alpha=1 if sig else .5,solid_capstyle="round",zorder=2)
    for xb in (np.log(r.OR_lo),np.log(r.OR_hi)): ax.plot([tox(xb)]*2,[yy-0.09,yy+0.09],color=col,lw=1.0,alpha=1 if sig else .5,zorder=2)
    ax.plot(tox(np.log(r.OR)),yy,marker="o",markersize=5.2,zorder=4,markerfacecolor=col if sig else "white",markeredgecolor=col,markeredgewidth=1.1)
    pre="\u2605 " if sig else ""; lab=f"{pre}{r.Cell_Type}\u2009\u2020" if r.Cell_Type in NEURO else f"{pre}{r.Cell_Type}"
    ax.text(xc["ct"],yy,lab,fontsize=7,ha="left",va="center",fontweight="bold" if sig else "normal",color=col if sig else "#333")
    ax.text(xc["or"],yy,f"{r.OR:.2f} ({r.OR_lo:.2f}\u2013{r.OR_hi:.2f})",fontsize=6.3,ha="center",va="center",fontweight="bold" if sig else "normal")
    ax.text(xc["p"],yy,fmt_p(r.p),fontsize=6.3,ha="right",va="center",fontweight="bold" if r.p<0.05 else "normal")
    ax.text(xc["fdr"],yy,fmt_p(r.padj),fontsize=6.3,ha="right",va="center",fontweight="bold" if sig else "normal",color=col if sig else "#333")
ax.set_title("Per-cell senescence rises with age (covariate-adjusted GLMM)",fontsize=7.6,fontweight="bold",pad=6,loc="left")
fig.text(0.06,-0.04,"\u2605 FDR<0.05   \u2020 neuronal   model: is_senescent ~ Age + Sex + Cohort + (1|Donor), per decade",fontsize=5.2,color="0.45",ha="left")
fig.tight_layout(pad=0.5)
save_figure(fig, f"susceptibility_glmm_forest_{CONDITION_TAG}")
plt.show(); print("\n✓ §5b complete")

---
## 32 · Burden GLMM — per decade

**Why.** `cbind(n_snc, n_other) ~ age_dec + Sex + Cohort + (1 | Donor)` where `n_other = donor_total − n_snc`. The aggregated binomial form of burden, one row per donor per cell type.

**`(1 | Donor)` here is an observation-level random effect,** not a donor clustering term — the frame has exactly one row per donor within each cell type. It absorbs overdispersion, which is what the cell's own comment says it is for.

In [ ]:
# =============================================================================
# §6 — BURDEN GLMM (aging) · cbind(n_snc, total−n_snc) ~ age_dec + Sex + Cohort + (1|Donor)
#   Predictor = Age per decade (exposure). OR = per-decade burden-odds.
# =============================================================================
import os, numpy as np, pandas as pd
from pathlib import Path
_r_home = os.environ.get("SENESCENCE_R_HOME")
if _r_home:
    os.environ.setdefault("R_HOME",      _r_home)
    os.environ.setdefault("R_LIBS",      _r_home + "/library")
    os.environ.setdefault("R_LIBS_USER", _r_home + "/library")
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, numpy2ri
from rpy2.robjects.conversion import localconverter
from statsmodels.stats.multitest import multipletests
pandas_converter = ro.default_converter + pandas2ri.converter
print("="*64); print("§6 — BURDEN GLMM (aging, per decade)"); print("="*64)

SCRATCH=_env("SENESCENCE_DATA"); TISSUE="brain"
STUDY_TYPE="aging"; DATASET="psychad_aging"; CONDITION_TAG=STUDY_TYPE
BASE=f"{SCRATCH}/{TISSUE}/module_03_burden_modeling/{STUDY_TYPE}/{DATASET}"
PATHS={"results":f"{BASE}/results","figures":f"{BASE}/figures"}
for p in PATHS.values(): Path(p).mkdir(parents=True, exist_ok=True)
def save_table(df, slug, index=False):
    fp=f"{PATHS['results']}/{slug}.csv"; df.to_csv(fp,index=index); print(f"  ✓ saved → {Path(fp).name} ({len(df)})")

PRED="age_dec"; COVARIATES=["Sex","Cohort"]    # Age is exposure → not a covariate
active_cts=["Excitatory","Inhibitory","Astrocyte","Oligodendrocyte","Microglia","OPC"]
GLMM_CONFIG={"optimizer":"bobyqa","maxfun":100000,"nAGQ":1,"min_donors":5}

print("\n1. DATA — df_burden (donor × CT)")
if "df_burden" not in dir():
    df_burden=pd.read_csv(f"{PATHS['results']}/burden_donor_celltype_{CONDITION_TAG}.csv"); print("   (loaded CSV)")
bdf=df_burden[df_burden["Cell_Type"].astype(str).isin(active_cts)].copy()
bdf["Donor"]=bdf["Donor"].astype(str); bdf["Cell_Type"]=bdf["Cell_Type"].astype(str)
bdf["n_snc"]=bdf["n_snc"].astype(int); bdf["donor_total"]=bdf["donor_total"].astype(int)
bdf["n_other"]=(bdf["donor_total"]-bdf["n_snc"]).astype(int)
bdf["age_dec"]=bdf["Age"].astype(float)/10.0
ccovs=[c for c in [PRED]+COVARIATES if c in bdf.columns]
print(f"   rows: {len(bdf)} | donors: {bdf['Donor'].nunique()} | age: "
      f"{bdf['age_dec'].min():.1f}–{bdf['age_dec'].max():.1f} decades")

print("\n2-3. FORMULA")
FORMULA=f"cbind(n_snc, n_other) ~ {' + '.join(ccovs)} + (1|Donor)"
print(f"   {FORMULA}  | binomial(logit), glmer | effect = OR per DECADE on BURDEN")
print("   (1|Donor) = observation-level RE for over-dispersion")

print("\n4. RESULTS")
def fit_glmer(ct):
    sub=bdf[bdf.Cell_Type==ct].dropna(subset=["n_snc","n_other"]+ccovs).copy()
    if sub.Donor.nunique()<GLMM_CONFIG["min_donors"] or sub["n_snc"].sum()<5:
        print(f"   {ct:18s} ✗ insufficient"); return None
    cv=[c for c in ccovs if not (c in {"Sex","Cohort"} and sub[c].astype(str).nunique()<2)]
    f=f"cbind(n_snc, n_other) ~ {' + '.join(cv)} + (1|Donor)"
    with localconverter(pandas_converter): ro.globalenv["ct_data"]=sub
    rcode=f'''suppressMessages(library(lme4))
    tryCatch({{
        m <- glmer({f}, data=ct_data, family=binomial(link="logit"),
                   control=glmerControl(optimizer="{GLMM_CONFIG['optimizer']}",
                                        optCtrl=list(maxfun={GLMM_CONFIG['maxfun']})), nAGQ={GLMM_CONFIG['nAGQ']})
        cs <- summary(m)$coefficients
        list(ok=TRUE, est=cs["{PRED}","Estimate"], se=cs["{PRED}","Std. Error"],
             p=cs["{PRED}","Pr(>|z|)"], singular=isSingular(m),
             warned=(length(m@optinfo$conv$lme4$messages)>0))
    }}, error=function(e) list(ok=FALSE, err=as.character(e)))'''
    r=ro.r(rcode)
    if not r.rx2("ok")[0]: print(f"   {ct:18s} ✗ {str(r.rx2('err')[0])[:50]}"); return None
    b=float(r.rx2("est")[0]); se=float(r.rx2("se")[0])
    sing=bool(r.rx2("singular")[0]); warned=bool(r.rx2("warned")[0])
    flag=" ⚠singular" if sing else (" ⚠warn" if warned else "")
    print(f"   {ct:18s} ✓ OR={np.exp(b):.3f}/decade{flag}")
    return dict(Cell_Type=ct, n_donors=sub.Donor.nunique(), n_snc=int(sub["n_snc"].sum()),
                beta=b, se=se, lo=b-1.96*se, hi=b+1.96*se, p=float(r.rx2("p")[0]), singular=sing, warned=warned)

rows=[r for r in (fit_glmer(ct) for ct in active_cts) if r]
glmm_b=pd.DataFrame(rows); glmm_b["padj"]=multipletests(glmm_b["p"], method="fdr_bh")[1]
glmm_b["OR"]=np.exp(glmm_b.beta); glmm_b["OR_lo"]=np.exp(glmm_b.lo); glmm_b["OR_hi"]=np.exp(glmm_b.hi)
glmm_b=glmm_b.sort_values("beta", ascending=False).reset_index(drop=True)
def fmt_p(x): return "<0.001" if x<0.001 else f"{x:.3f}"
print()
print(glmm_b.assign(OR=lambda d:d.OR.round(3), p=lambda d:d.p.map(fmt_p), padj=lambda d:d.padj.map(fmt_p),
                    flag=lambda d:np.where(d.singular,"singular",np.where(d.warned,"warn","")))
      [["Cell_Type","n_donors","n_snc","OR","p","padj","flag"]].to_string(index=False))
print("   (OR per decade on burden; >1 = senescent-cell load rises with age)")
save_table(glmm_b, f"burden_glmm_{CONDITION_TAG}")
print("\n✓ §6 complete")

---
## 33 · Burden forest

**Why.** Table-style forest: OR per decade on burden, 95% CI, p, FDR.

Read this against section 31. A type that moves on burden but not on susceptibility is changing because it is becoming more common; a type that moves on susceptibility but not burden is becoming more senescent without becoming more numerous.

In [ ]:
# =============================================================================
# §6b — PLOT: burden GLMM table-forest (OR/decade · 95% CI · p · FDR)
# =============================================================================
import numpy as np, matplotlib as mpl, matplotlib.pyplot as plt
mpl.rcParams.update({"pdf.fonttype":42,"ps.fonttype":42,"svg.fonttype":"none",
    "font.family":"sans-serif","font.sans-serif":["Arial","Helvetica","DejaVu Sans"],"axes.linewidth":0.6,"font.size":7})
print("="*64); print("§6b — burden GLMM forest (aging)"); print("="*64)
COL={"Excitatory":"#0072B2","Inhibitory":"#E69F00","Astrocyte":"#009E73","Oligodendrocyte":"#56B4E9","Microglia":"#D55E00","OPC":"#CC79A7"}
NEURO={"Excitatory","Inhibitory"}
def fmt_p(x): return "<0.001" if x<0.001 else f"{x:.3f}"
d=glmm_b.sort_values("OR", ascending=False).reset_index(drop=True); n=len(d); y=np.arange(n)[::-1]
fig=plt.figure(figsize=(7.0,2.7),dpi=300); ax=fig.add_subplot(111); ax.set_xlim(0,1); ax.set_ylim(-1.0,n+0.3); ax.axis("off")
xc={"ct":0.005,"fL":0.28,"fR":0.52,"or":0.625,"p":0.875,"fdr":0.99}
logv=np.concatenate([np.log(d.OR_lo.values),np.log(d.OR_hi.values)]); xmin,xmax=logv.min()-0.10,logv.max()+0.10
def tox(v): v=np.clip(v,xmin,xmax); return xc["fL"]+(v-xmin)/(xmax-xmin)*(xc["fR"]-xc["fL"])
y_hdr=n-0.35; y_line=n-0.55
ax.text(xc["ct"],y_hdr,"Cell type",fontsize=7.3,fontweight="bold",ha="left",va="bottom")
for lab,k,ha in [("OR (95% CI)","or","center"),("p","p","right"),("FDR","fdr","right")]:
    ax.text(xc[k],y_hdr,lab,fontsize=7.3,fontweight="bold",ha=ha,va="bottom")
ax.text((xc["fL"]+xc["fR"])/2,y_hdr,"per decade",fontsize=6.8,fontweight="bold",ha="center",va="bottom")
ax.plot([0,0.99],[y_line,y_line],"k-",lw=0.9,clip_on=False)
or_ticks=[0.8,1.0,1.25,1.5,2.0]
for t in or_ticks:
    lv=np.log(t)
    if xmin<=lv<=xmax:
        gx=tox(lv); ax.plot([gx,gx],[-0.55,y_line],color="#eee",lw=0.5,zorder=0); ax.text(gx,-0.72,f"{t:g}",fontsize=5.8,ha="center",va="top",color="#666")
ax.plot([tox(0)]*2,[-0.55,y_line],color="#888",ls="--",lw=0.8,alpha=0.7,zorder=1)
ax.text((xc["fL"]+xc["fR"])/2,-1.0,"Odds ratio per decade (log scale)",fontsize=6.2,ha="center",va="top",color="#444")
for i,r in d.iterrows():
    yy=y[i]; col=COL.get(r.Cell_Type,"#888"); sig=r.padj<0.05
    if i%2==0: ax.axhspan(yy-0.5,yy+0.5,xmin=0.003,xmax=0.997,color="#f7f7f7",zorder=0)
    ax.plot([tox(np.log(r.OR_lo)),tox(np.log(r.OR_hi))],[yy,yy],color=col,lw=1.4,alpha=1 if sig else .5,solid_capstyle="round",zorder=2)
    for xb in (np.log(r.OR_lo),np.log(r.OR_hi)): ax.plot([tox(xb)]*2,[yy-0.09,yy+0.09],color=col,lw=1.0,alpha=1 if sig else .5,zorder=2)
    ax.plot(tox(np.log(r.OR)),yy,marker="o",markersize=5.2,zorder=4,markerfacecolor=col if sig else "white",markeredgecolor=col,markeredgewidth=1.1)
    pre="\u2605 " if sig else ""; lab=f"{pre}{r.Cell_Type}\u2009\u2020" if r.Cell_Type in NEURO else f"{pre}{r.Cell_Type}"
    ax.text(xc["ct"],yy,lab,fontsize=7,ha="left",va="center",fontweight="bold" if sig else "normal",color=col if sig else "#333")
    ax.text(xc["or"],yy,f"{r.OR:.2f} ({r.OR_lo:.2f}\u2013{r.OR_hi:.2f})",fontsize=6.3,ha="center",va="center",fontweight="bold" if sig else "normal")
    ax.text(xc["p"],yy,fmt_p(r.p),fontsize=6.3,ha="right",va="center",fontweight="bold" if r.p<0.05 else "normal")
    ax.text(xc["fdr"],yy,fmt_p(r.padj),fontsize=6.3,ha="right",va="center",fontweight="bold" if sig else "normal",color=col if sig else "#333")
ax.set_title("Senescent-cell burden rises with age (covariate-adjusted GLMM)",fontsize=7.6,fontweight="bold",pad=6,loc="left")
fig.text(0.06,-0.04,"\u2605 FDR<0.05   \u2020 neuronal   model: cbind(n_snc, total\u2212n_snc) ~ Age + Sex + Cohort + (1|Donor), per decade",fontsize=5.2,color="0.45",ha="left")
fig.tight_layout(pad=0.5)
save_figure(fig, f"burden_glmm_forest_{CONDITION_TAG}")
plt.show(); print("\n✓ §6b complete")

---
## 34 · Garg differential proportion

**Why.** Cube-root transform on the cell-type proportion, then a slope on age (Garg et al. 2025, Eq. 5). The compositional counterpart to the burden models — it asks whether the *abundance* of each type is shifting, which is the confound the burden/susceptibility split exists to separate.

In [ ]:
# =============================================================================
# §7 — DIFFERENTIAL PROPORTION (Garg Eq. 5) · aging · age_dec slope
#   (Proportion)^(1/3) ~ age_dec + Sex + Cohort   [Gaussian glm]   β = per decade
# =============================================================================
import numpy as np, pandas as pd
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from pathlib import Path
print("="*64); print("§7 — DIFF PROPORTION (Garg, aging per-decade slope)"); print("="*64)

SCRATCH=_env("SENESCENCE_DATA"); TISSUE="brain"
STUDY_TYPE="aging"; DATASET="psychad_aging"; CONDITION_TAG=STUDY_TYPE
BASE=f"{SCRATCH}/{TISSUE}/module_03_burden_modeling/{STUDY_TYPE}/{DATASET}"
PATHS={"results":f"{BASE}/results","figures":f"{BASE}/figures"}
for p in PATHS.values(): Path(p).mkdir(parents=True, exist_ok=True)
def save_table(df, slug, index=False):
    fp=f"{PATHS['results']}/{slug}.csv"; df.to_csv(fp,index=index); print(f"  ✓ saved → {Path(fp).name} ({len(df)})")

if "df_burden" not in dir():
    df_burden=pd.read_csv(f"{PATHS['results']}/burden_donor_celltype_{CONDITION_TAG}.csv")

CT_ORDER=["Excitatory","Inhibitory","Astrocyte","Oligodendrocyte","Microglia","OPC"]
active_cts=[c for c in CT_ORDER if c in set(df_burden["Cell_Type"].astype(str).unique())]
COVARIATES=["Sex","Cohort"]               # Age is exposure → predictor, not covariate
df_burden=df_burden.copy(); df_burden["age_dec"]=df_burden["Age"].astype(float)/10.0

print("\nFORMULA (Garg Eq. 5, aging slope):")
print("    (Proportion)^(1/3) ~ age_dec + Sex + Cohort      [Gaussian glm]")
print("    β = change in cube-root proportion per DECADE; FDR-BH within metric")

METRICS={"CellProp":"Proportion (abundance)","SenBurden":"Senescent-cell burden"}

def garg_glm(metric):
    rows=[]; covs=["age_dec"]+[c for c in COVARIATES if c in df_burden.columns]
    for ct in active_cts:
        sub=df_burden[df_burden["Cell_Type"].astype(str)==ct].dropna(subset=[metric]+covs).copy()
        if len(sub)<10: continue
        sub["y"]=np.cbrt(sub[metric].astype(float))
        cv=[c for c in covs if not (c in {"Sex","Cohort"} and sub[c].astype(str).nunique()<2)]
        m=smf.glm(f"y ~ {' + '.join(cv)}", data=sub).fit()
        if "age_dec" not in m.params.index: continue
        ci=m.conf_int().loc["age_dec"]
        rows.append(dict(Cell_Type=ct, beta=m.params["age_dec"], se=m.bse["age_dec"],
                         lo=ci[0], hi=ci[1], p=m.pvalues["age_dec"]))
    res=pd.DataFrame(rows)
    if len(res): res["padj"]=multipletests(res["p"], method="fdr_bh")[1]
    return res

def fmt_p(x): return "<0.001" if x<0.001 else f"{x:.3f}"
results={}
for metric,label in METRICS.items():
    print("\n"+"─"*64); print(f"▸ {metric} ({label}) — per-decade slope")
    res=garg_glm(metric); results[metric]=res
    print(res.assign(beta=lambda d:d.beta.round(4), se=lambda d:d.se.round(4),
                     CI=lambda d:"["+d.lo.round(3).astype(str)+", "+d.hi.round(3).astype(str)+"]",
                     p=lambda d:d.p.map(fmt_p), padj=lambda d:d.padj.map(fmt_p)+np.where(d.padj<0.05," *",""))
          [["Cell_Type","beta","se","CI","p","padj"]].to_string(index=False))

combined=pd.concat([r.assign(metric=m) for m,r in results.items()], ignore_index=True)
save_table(combined, "garg_proportion_glm_aging")
garg_aging={m:r.set_index("Cell_Type") for m,r in results.items()}
print("\n✓ §7 complete")

---
## Not in this module

Present in the source notebook, deliberately left out:

- **§5c Sloan module validation** — belongs to module 05, where the rest of the
  hallmark panels live.
- **§7a-c pathology index, senescence ~ pathology, interaction and mediation** —
  a different question from burden. Its own module.
- **§8.1-§8.4 and §9.1-§9.4 sensitivity analyses** — already rebuilt as
  module 04.
- **Repeat aging panels** (source cells 85 / 107 / 109, and 86 / 87 / 94) —
  successive redraws of the same figure. Sections 31 and 33 carry the version
  that the others converge on.
- **51 empty cells.**